In [91]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Data Load

### EPT reductions

In [92]:
# import pandas as pd
# import numpy as np
# pd.set_option('display.max_columns', None)
# file_path_external = '/content/drive/MyDrive/lower ept y awt/impact_measure.xlsx'

In [93]:
import gspread
from google.colab import auth
import google.auth

auth.authenticate_user()

credentials, _ = google.auth.default(scopes=[
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
])

gc = gspread.authorize(credentials)

sheet_id = "17lef3Nf4BOwEbtzMyvePV1wVs_1Klb1wm8Xb1wDtRfk"
sheet_name = "ept_adjustment_history"

worksheet = gc.open_by_key(sheet_id).worksheet(sheet_name)
data = worksheet.get_all_values()
reduction_raw = pd.DataFrame(data[1:], columns=data[0])

reduction_raw.head()

KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display


# ============================================================
# ANÁLISIS DE VENDORS POR WEEK Y CRITERIO
#
# Este bloque se ejecuta justo después de leer el DataFrame
# `reduction` desde ept_adjustment_history.
#
# Salidas:
#   1. Barras apiladas: todos los vendors por Week y criterio.
#   2. Barras apiladas: vendors reincidentes por Week y criterio.
#   3. Tabla: historia completa de criterios por vendor reincidente.
#
# Reglas:
#   - Incluye solamente waves cuyo nombre contiene "week".
#   - Excluye waves cuyo nombre contiene "mini".
#   - NO conserva solamente la última wave del vendor.
#   - Un reincidente es un vendor que aparece en una Week posterior
#     después de haber aparecido en al menos una Week anterior.
# ============================================================


# ============================================================
# CONFIGURACIÓN
# ============================================================

WAVE_COL = "wave"
FLAG_COL = "flag"
VENDOR_COL = "vendor_code"

DATE_CANDIDATES = [
    "executed_at",
    "start_date",
    "created_at",
]


# ============================================================
# ESTILO PARA PRESENTACIÓN
# ============================================================

sns.set_theme(
    style="whitegrid",
    context="talk",
)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "font.family": "sans-serif",
    "font.size": 14,
    "axes.titlesize": 22,
    "axes.titleweight": "bold",
    "axes.labelsize": 16,
    "axes.labelweight": "semibold",
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 13,
    "legend.title_fontsize": 14,
})


# ============================================================
# VALIDAR INPUT
# ============================================================

required_columns = {
    WAVE_COL,
    FLAG_COL,
    VENDOR_COL,
}

missing_columns = required_columns - set(reduction_raw.columns)

if missing_columns:
    raise KeyError(
        "Faltan columnas obligatorias en reduction: "
        f"{sorted(missing_columns)}"
    )


# ============================================================
# FILTRAR WAVES
#
# Equivalente a:
#   LOWER(wave) LIKE '%week%'
#   AND LOWER(wave) NOT LIKE '%mini%'
# ============================================================

reductions_week = reduction_raw.copy()

reductions_week[WAVE_COL] = (
    reductions_week[WAVE_COL]
    .astype("string")
    .str.strip()
)

wave_lower = reductions_week[WAVE_COL].str.lower()

reductions_week = reductions_week.loc[
    wave_lower.str.contains("week", na=False)
    & ~wave_lower.str.contains("mini", na=False)
].copy()

if reductions_week.empty:
    raise ValueError(
        "No quedaron registros después de filtrar waves Week "
        "y excluir waves Mini."
    )


# ============================================================
# LIMPIAR VENDOR Y CRITERIO
# ============================================================

reductions_week[VENDOR_COL] = (
    reductions_week[VENDOR_COL]
    .astype("string")
    .str.strip()
)

reductions_week[FLAG_COL] = (
    reductions_week[FLAG_COL]
    .astype("string")
    .str.strip()
    .replace({
        "": "SIN_FLAG",
        "<NA>": "SIN_FLAG",
        "nan": "SIN_FLAG",
        "None": "SIN_FLAG",
        "null": "SIN_FLAG",
    })
    .fillna("SIN_FLAG")
)

reductions_week = reductions_week.loc[
    reductions_week[VENDOR_COL].notna()
    & reductions_week[VENDOR_COL].ne("")
].copy()


# ============================================================
# ORDEN CRONOLÓGICO DE LAS WAVES
# ============================================================

date_col = next(
    (
        column
        for column in DATE_CANDIDATES
        if column in reductions_week.columns
    ),
    None,
)

wave_order_table = pd.DataFrame({
    WAVE_COL: reductions_week[WAVE_COL].drop_duplicates()
})

wave_order_table["_week_number"] = pd.to_numeric(
    wave_order_table[WAVE_COL].str.extract(
        r"(?i)week\D*(\d+)",
        expand=False,
    ),
    errors="coerce",
)

if date_col is not None:
    reductions_week["_event_date"] = pd.to_datetime(
        reductions_week[date_col],
        errors="coerce",
    )

    wave_dates = (
        reductions_week
        .groupby(WAVE_COL, as_index=False)
        .agg(_wave_date=("_event_date", "min"))
    )

    wave_order_table = wave_order_table.merge(
        wave_dates,
        on=WAVE_COL,
        how="left",
    )

    wave_order_table = wave_order_table.sort_values(
        ["_wave_date", "_week_number", WAVE_COL],
        na_position="last",
    )

else:
    wave_order_table = wave_order_table.sort_values(
        ["_week_number", WAVE_COL],
        na_position="last",
    )

wave_order = wave_order_table[WAVE_COL].tolist()

wave_position = {
    wave: position
    for position, wave in enumerate(wave_order)
}

reductions_week["_wave_position"] = (
    reductions_week[WAVE_COL]
    .map(wave_position)
)


# ============================================================
# UNA FILA POR VENDOR + WAVE
#
# Normalmente un vendor tiene un único flag dentro de una wave.
# Si existen varios, se conservan todos unidos por " | ", de modo
# que el vendor siga contando una sola vez en esa wave.
# ============================================================

def combine_unique_flags(values):
    flags = sorted(
        set(values.dropna().astype(str))
    )
    return " | ".join(flags) if flags else "SIN_FLAG"


vendor_wave_history = (
    reductions_week
    .groupby(
        [VENDOR_COL, WAVE_COL, "_wave_position"],
        as_index=False,
    )
    .agg(
        criterio=(FLAG_COL, combine_unique_flags),
    )
    .sort_values(
        [VENDOR_COL, "_wave_position"]
    )
    .reset_index(drop=True)
)

# Número de aparición del vendor dentro de las waves filtradas.
# La primera aparición no es reincidencia; desde la segunda, sí.
vendor_wave_history["numero_aparicion"] = (
    vendor_wave_history
    .groupby(VENDOR_COL)
    .cumcount()
    .add(1)
)

vendor_wave_history["es_reincidente_en_wave"] = (
    vendor_wave_history["numero_aparicion"] > 1
)


# ============================================================
# DATOS DEL PRIMER GRÁFICO: TODOS LOS VENDORS
# ============================================================

vendors_por_wave_criterio = (
    vendor_wave_history
    .groupby(
        [WAVE_COL, "_wave_position", "criterio"],
        as_index=False,
    )
    .agg(
        vendors=(VENDOR_COL, "nunique"),
    )
    .sort_values(
        ["_wave_position", "criterio"]
    )
)


# ============================================================
# DATOS DEL SEGUNDO GRÁFICO: SOLO REINCIDENTES
#
# El criterio mostrado corresponde al criterio adoptado en la
# wave donde el vendor está reincidiendo.
# ============================================================

reincidencias_wave = vendor_wave_history.loc[
    vendor_wave_history["es_reincidente_en_wave"]
].copy()

reincidentes_por_wave_criterio = (
    reincidencias_wave
    .groupby(
        [WAVE_COL, "_wave_position", "criterio"],
        as_index=False,
    )
    .agg(
        vendors=(VENDOR_COL, "nunique"),
    )
    .sort_values(
        ["_wave_position", "criterio"]
    )
)


# ============================================================
# COLORES CONSISTENTES ENTRE AMBOS GRÁFICOS
# ============================================================

criteria_order = (
    vendors_por_wave_criterio
    .groupby("criterio")["vendors"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

palette = sns.color_palette(
    "colorblind",
    n_colors=max(len(criteria_order), 3),
)

criterion_colors = {
    criterion: palette[position]
    for position, criterion in enumerate(criteria_order)
}


# ============================================================
# FUNCIÓN DE GRÁFICO APILADO PARA PRESENTACIÓN
# ============================================================

def plot_stacked_vendors(
    summary_df,
    title,
    subtitle,
    output_name=None,
):
    fig, ax = plt.subplots(figsize=(15, 8.5))

    if summary_df.empty:
        ax.text(
            0.5,
            0.5,
            "No hay vendors para mostrar",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=20,
            fontweight="semibold",
            color="#475467",
        )
        ax.set_axis_off()
        fig.suptitle(
            title,
            fontsize=22,
            fontweight="bold",
            color="#101828",
            y=0.98,
        )
        plt.show()
        return

    plot_table = (
        summary_df
        .pivot_table(
            index=WAVE_COL,
            columns="criterio",
            values="vendors",
            aggfunc="sum",
            fill_value=0,
        )
        .reindex(index=wave_order, fill_value=0)
        .reindex(columns=criteria_order, fill_value=0)
    )

    plot_table.plot(
        kind="bar",
        stacked=True,
        width=0.72,
        color=[
            criterion_colors[column]
            for column in plot_table.columns
        ],
        edgecolor="white",
        linewidth=0.8,
        ax=ax,
    )

    # Cifras dentro de cada segmento.
    for container in ax.containers:
        labels = [
            f"{int(bar.get_height()):,}"
            if bar.get_height() > 0
            else ""
            for bar in container
        ]

        ax.bar_label(
            container,
            labels=labels,
            label_type="center",
            fontsize=12,
            fontweight="bold",
            color="white",
        )

    # Total de cada barra.
    totals = plot_table.sum(axis=1)
    max_total = max(float(totals.max()), 1.0)

    for x_position, total in enumerate(totals):
        if total > 0:
            ax.text(
                x_position,
                total + max_total * 0.025,
                f"{int(total):,}",
                ha="center",
                va="bottom",
                fontsize=14,
                fontweight="bold",
                color="#101828",
            )

    ax.set_ylim(0, max_total * 1.16)

    ax.set_title(
        title,
        fontsize=22,
        fontweight="bold",
        color="#101828",
        pad=30,
    )

    ax.text(
        0,
        1.015,
        subtitle,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=14,
        color="#667085",
    )

    ax.set_xlabel("")
    ax.set_ylabel(
        "Vendors únicos",
        fontsize=16,
        fontweight="semibold",
        color="#344054",
    )

    rotation = 35 if len(plot_table.index) > 8 else 0

    ax.tick_params(
        axis="x",
        labelrotation=rotation,
        labelsize=14,
        colors="#344054",
    )

    ax.tick_params(
        axis="y",
        labelsize=13,
        colors="#475467",
    )

    ax.grid(
        axis="y",
        color="#D0D5DD",
        linewidth=0.8,
        alpha=0.65,
    )
    ax.grid(axis="x", visible=False)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_color("#D0D5DD")

    legend = ax.legend(
        title="Criterio",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
        labelspacing=0.9,
        handlelength=1.4,
    )

    legend.get_title().set_fontweight("bold")

    plt.tight_layout(rect=[0, 0, 0.82, 1])

    # Para guardar una imagen lista para Slides/PowerPoint,
    # descomenta estas dos líneas y asigna output_name.
    # if output_name:
    #     fig.savefig(output_name, bbox_inches="tight", facecolor="white")

    plt.show()


# ============================================================
# 1. BARRAS APILADAS: TODOS LOS VENDORS
# ============================================================

plot_stacked_vendors(
    summary_df=vendors_por_wave_criterio,
    title="Vendors ajustados por wave y criterio",
    subtitle=(
        "Cada vendor cuenta una vez dentro de cada wave. "
        "Se incluyen Week y se excluyen Mini."
    ),
    output_name="vendors_por_wave_criterio.png",
)


# ============================================================
# 2. BARRAS APILADAS: SOLO VENDORS REINCIDENTES
# ============================================================

plot_stacked_vendors(
    summary_df=reincidentes_por_wave_criterio,
    title="Vendors reincidentes por wave y criterio",
    subtitle=(
        "Cuenta vendors que ya habían aparecido en una Week anterior; "
        "el color corresponde al criterio de la wave actual."
    ),
    output_name="reincidentes_por_wave_criterio.png",
)


# ============================================================
# 3. TABLA: HISTORIA DE CRITERIOS DE LOS REINCIDENTES
#
# Se incluyen todas las apariciones del vendor, incluida su primera
# wave, para poder leer su trayectoria completa de izquierda a derecha.
# Las columnas Week se crean automáticamente.
# ============================================================

waves_por_vendor = (
    vendor_wave_history
    .groupby(VENDOR_COL)[WAVE_COL]
    .nunique()
)

recurrent_vendor_ids = waves_por_vendor.loc[
    waves_por_vendor > 1
].index

historia_reincidentes_long = (
    vendor_wave_history
    .loc[
        vendor_wave_history[VENDOR_COL]
        .isin(recurrent_vendor_ids),
        [VENDOR_COL, WAVE_COL, "_wave_position", "criterio"],
    ]
    .sort_values(
        [VENDOR_COL, "_wave_position"]
    )
    .reset_index(drop=True)
)

resumen_reincidente = (
    historia_reincidentes_long
    .groupby(VENDOR_COL)
    .agg(
        waves_con_ajuste=(WAVE_COL, "nunique"),
        criterios_distintos=("criterio", "nunique"),
    )
)

resumen_reincidente["cambio_criterio"] = np.where(
    resumen_reincidente["criterios_distintos"] > 1,
    "Sí",
    "No",
)

tabla_historia_reincidentes = (
    historia_reincidentes_long
    .pivot(
        index=VENDOR_COL,
        columns=WAVE_COL,
        values="criterio",
    )
    .reindex(columns=wave_order)
    .join(resumen_reincidente)
    .reset_index()
)

tabla_historia_reincidentes = tabla_historia_reincidentes[
    [
        VENDOR_COL,
        "waves_con_ajuste",
        "criterios_distintos",
        "cambio_criterio",
        *wave_order,
    ]
]

tabla_historia_reincidentes[wave_order] = (
    tabla_historia_reincidentes[wave_order]
    .fillna("—")
)

tabla_historia_reincidentes = (
    tabla_historia_reincidentes
    .sort_values(
        [
            "waves_con_ajuste",
            "criterios_distintos",
            VENDOR_COL,
        ],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)


# ============================================================
# MOSTRAR TABLA CON FORMATO MÁS GRANDE
# ============================================================

print(
    "\nHistoria de vendors reincidentes\n"
    f"Vendors reincidentes: {len(tabla_historia_reincidentes):,}\n"
    "Cada columna Week muestra el criterio aplicado en esa wave.\n"
)

tabla_historia_reincidentes_styled = (
    tabla_historia_reincidentes
    .style
    .hide(axis="index")
    .set_caption(
        "Historia de criterios de vendors reincidentes"
    )
    .set_properties(**{
        "font-size": "14px",
        "padding": "10px 13px",
        "text-align": "center",
        "border-bottom": "1px solid #EAECF0",
        "white-space": "normal",
    })
    .set_properties(
        subset=[VENDOR_COL],
        **{
            "font-weight": "bold",
            "color": "#101828",
        },
    )
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "20px"),
                ("font-weight", "bold"),
                ("color", "#101828"),
                ("text-align", "left"),
                ("padding", "10px 0 16px 0"),
            ],
        },
        {
            "selector": "th",
            "props": [
                ("font-size", "14px"),
                ("font-weight", "bold"),
                ("background-color", "#F2F4F7"),
                ("color", "#344054"),
                ("text-align", "center"),
                ("padding", "11px 13px"),
                ("border-bottom", "2px solid #D0D5DD"),
            ],
        },
        {
            "selector": "tbody tr:nth-child(even)",
            "props": [
                ("background-color", "#F9FAFB"),
            ],
        },
    ])
)

display(tabla_historia_reincidentes_styled)


# ============================================================
# OBJETOS FINALES DISPONIBLES
# ============================================================
#
# vendors_por_wave_criterio
#     Conteo para el gráfico de todos los vendors.
#
# reincidentes_por_wave_criterio
#     Conteo para el gráfico de reincidentes por criterio actual.
#
# historia_reincidentes_long
#     Historia en formato largo: vendor + wave + criterio.
#
# tabla_historia_reincidentes
#     Historia en formato ancho: una columna por Week.
# ============================================================


### AWT0 / AWT null — última semana cerrada

Este bloque consulta **directamente BigQuery** para la última semana completa (lunes–domingo) y usa `reduction` para identificar la **última wave tipo `WeekXX`**.

Para **AWT0** y **AWT null** se construyen buckets según el share del vendor (`[0–10%)`, `[10–20%)`, …, `[90–100%]`) y se muestran dos lecturas:

- **Vendors:** cantidad de vendors CL en cada bucket, apilando `última wave` + `resto CL`.
- **Orders:** suma del **% de orders CL** de los vendors ubicados en cada bucket, apilando `última wave` + `(CL − última wave)`. No se bucketean órdenes individualmente.

El universo replica las reglas principales de `comparison`: Chile, delivery primaria, partner online, sin preorders y excluyendo `courier_business`.


In [ ]:
# ============================================================
# AWT0 / AWT NULL — ÚLTIMA SEMANA CERRADA
# 1) Identificar última Week wave desde reduction
# 2) Consultar métricas vendor-level directamente en BigQuery
# ============================================================

import re
import numpy as np
import pandas as pd
from IPython.display import display

# Base de reductions usada por esta sección.
# Se reconstruye desde el raw para que el resultado no dependa
# del orden en que se hayan ejecutado otras celdas del notebook.
reduction = reduction_raw.copy()

# ------------------------------------------------------------
# 1. ÚLTIMA WAVE TIPO WeekXX
# ------------------------------------------------------------

reduction_awt = reduction.copy()

required_reduction_cols = {"wave", "vendor_code"}
missing = required_reduction_cols - set(reduction_awt.columns)

if missing:
    raise KeyError(
        "Faltan columnas en reduction para identificar la última wave: "
        f"{sorted(missing)}"
    )

reduction_awt["wave"] = (
    reduction_awt["wave"]
    .astype("string")
    .str.strip()
)

reduction_awt["vendor_code"] = (
    reduction_awt["vendor_code"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# Mantener la misma exclusión del flujo principal.
if "flag" in reduction_awt.columns:
    reduction_awt = reduction_awt.loc[
        reduction_awt["flag"]
        .astype("string")
        .str.strip()
        .ne("Manual adjustment")
    ].copy()

wave_mask = (
    reduction_awt["wave"].str.contains(r"(?i)week", na=False)
    & ~reduction_awt["wave"].str.contains(r"(?i)mini", na=False)
)

reduction_week_awt = reduction_awt.loc[wave_mask].copy()

if reduction_week_awt.empty:
    raise ValueError(
        "No se encontraron waves tipo WeekXX en reduction."
    )

# Fecha efectiva de la wave: preferimos executed_at y luego start_date.
date_candidates = [
    col
    for col in ["executed_at", "start_date", "created_at"]
    if col in reduction_week_awt.columns
]

if date_candidates:
    reduction_week_awt["_wave_date"] = pd.NaT

    for col in date_candidates:
        parsed = pd.to_datetime(
            reduction_week_awt[col],
            errors="coerce",
        )

        reduction_week_awt["_wave_date"] = (
            reduction_week_awt["_wave_date"]
            .fillna(parsed)
        )
else:
    reduction_week_awt["_wave_date"] = pd.NaT

reduction_week_awt["_week_number"] = pd.to_numeric(
    reduction_week_awt["wave"].str.extract(
        r"(?i)week\D*(\d+)",
        expand=False,
    ),
    errors="coerce",
)

wave_order_awt = (
    reduction_week_awt
    .groupby("wave", as_index=False)
    .agg(
        wave_date=("_wave_date", "max"),
        week_number=("_week_number", "max"),
    )
    .sort_values(
        ["wave_date", "week_number", "wave"],
        na_position="first",
    )
)

latest_week_wave = wave_order_awt.iloc[-1]["wave"]

latest_wave_vendors = set(
    reduction_week_awt.loc[
        reduction_week_awt["wave"].eq(latest_week_wave),
        "vendor_code",
    ]
    .dropna()
    .loc[lambda s: s.ne("")]
    .tolist()
)

if not latest_wave_vendors:
    raise ValueError(
        f"{latest_week_wave} no tiene vendor_code válidos en reduction."
    )

print(
    f"Última reduction wave: {latest_week_wave}"
    f"\nVendors configurados en la wave: {len(latest_wave_vendors):,}"
)


# ------------------------------------------------------------
# 2. QUERY VENDOR-LEVEL — ÚLTIMA SEMANA COMPLETA
# ------------------------------------------------------------

awt_last_closed_week_query = """
WITH params AS (
  SELECT
    DATE_SUB(
      DATE_TRUNC(
        CURRENT_DATE('America/Santiago'),
        WEEK(MONDAY)
      ),
      INTERVAL 1 WEEK
    ) AS week_start,

    DATE_SUB(
      DATE_TRUNC(
        CURRENT_DATE('America/Santiago'),
        WEEK(MONDAY)
      ),
      INTERVAL 1 DAY
    ) AS week_end
),

base AS (
  SELECT DISTINCT
    l.peya_order_id,
    CAST(l.vendor.vendor_code AS STRING) AS vendor_code,
    SAFE_DIVIDE(
      l.timings.avoidable_wait_time,
      60.0
    ) AS awt_min

  FROM `peya-bi-tools-pro.il_logistics.fact_logistic_orders` l

  LEFT JOIN UNNEST(l.deliveries) d

  LEFT JOIN `peya-bi-tools-pro.il_core.dim_partner` p
    ON l.vendor.vendor_code = CAST(p.partner_id AS STRING)

  CROSS JOIN params

  WHERE l.country_code = 'cl'
    AND l.peya_order_id IS NOT NULL
    AND d.is_primary = TRUE
    AND p.is_online = TRUE
    AND l.is_preorder = FALSE
    AND COALESCE(l.vendor.vertical_type, '') != 'courier_business'
    AND DATE(l.created_date_local)
        BETWEEN params.week_start AND params.week_end
),

vendor_metrics AS (
  SELECT
    vendor_code,

    COUNT(DISTINCT peya_order_id) AS total_orders,

    COUNT(DISTINCT IF(
      awt_min = 0,
      peya_order_id,
      NULL
    )) AS awt0_orders,

    COUNT(DISTINCT IF(
      awt_min IS NULL,
      peya_order_id,
      NULL
    )) AS awt_null_orders

  FROM base

  GROUP BY
    vendor_code
),

cl_total AS (
  SELECT
    COUNT(DISTINCT peya_order_id) AS total_orders_cl
  FROM base
)

SELECT
  p.week_start,
  p.week_end,
  v.vendor_code,
  v.total_orders,
  v.awt0_orders,
  v.awt_null_orders,
  c.total_orders_cl,

  SAFE_DIVIDE(
    v.total_orders,
    c.total_orders_cl
  ) AS orders_share_cl,

  SAFE_DIVIDE(
    v.awt0_orders,
    v.total_orders
  ) AS awt0_share,

  SAFE_DIVIDE(
    v.awt_null_orders,
    v.total_orders
  ) AS awtnull_share

FROM vendor_metrics v
CROSS JOIN cl_total c
CROSS JOIN params p

WHERE v.total_orders > 0

ORDER BY
  v.total_orders DESC
"""

project_id = "peya-chile"

vendor_awt_last_week = pd.read_gbq(
    awt_last_closed_week_query,
    project_id=project_id,
)

vendor_awt_last_week["vendor_code"] = (
    vendor_awt_last_week["vendor_code"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

numeric_cols = [
    "total_orders",
    "awt0_orders",
    "awt_null_orders",
    "total_orders_cl",
    "orders_share_cl",
    "awt0_share",
    "awtnull_share",
]

for col in numeric_cols:
    vendor_awt_last_week[col] = pd.to_numeric(
        vendor_awt_last_week[col],
        errors="coerce",
    )

vendor_awt_last_week["is_latest_week_wave"] = (
    vendor_awt_last_week["vendor_code"]
    .isin(latest_wave_vendors)
)

if vendor_awt_last_week.empty:
    raise ValueError(
        "BigQuery no devolvió vendors para la última semana cerrada."
    )

week_start_awt = pd.to_datetime(
    vendor_awt_last_week["week_start"].iloc[0]
).date()

week_end_awt = pd.to_datetime(
    vendor_awt_last_week["week_end"].iloc[0]
).date()

total_orders_cl_awt = float(
    vendor_awt_last_week["total_orders_cl"].max()
)

active_latest_wave_vendors = int(
    vendor_awt_last_week.loc[
        vendor_awt_last_week["is_latest_week_wave"],
        "vendor_code",
    ].nunique()
)

latest_wave_orders = float(
    vendor_awt_last_week.loc[
        vendor_awt_last_week["is_latest_week_wave"],
        "total_orders",
    ].sum()
)

print(
    f"\nSemana analizada: {week_start_awt:%d/%m/%Y}"
    f"–{week_end_awt:%d/%m/%Y}"
    f"\nOrders CL: {total_orders_cl_awt:,.0f}"
    f"\nVendors CL con orders: "
    f"{vendor_awt_last_week['vendor_code'].nunique():,}"
    f"\nVendors de {latest_week_wave} con orders: "
    f"{active_latest_wave_vendors:,} / {len(latest_wave_vendors):,}"
    f"\nOrders de {latest_week_wave}: "
    f"{latest_wave_orders:,.0f} "
    f"({100 * latest_wave_orders / total_orders_cl_awt:.2f}% CL)"
)

display(
    vendor_awt_last_week[
        [
            "vendor_code",
            "total_orders",
            "orders_share_cl",
            "awt0_share",
            "awtnull_share",
            "is_latest_week_wave",
        ]
    ]
    .assign(
        orders_share_cl_pct=lambda x: 100 * x["orders_share_cl"],
        awt0_share_pct=lambda x: 100 * x["awt0_share"],
        awtnull_share_pct=lambda x: 100 * x["awtnull_share"],
    )
    [
        [
            "vendor_code",
            "total_orders",
            "orders_share_cl_pct",
            "awt0_share_pct",
            "awtnull_share_pct",
            "is_latest_week_wave",
        ]
    ]
    .head(10)
    .round(2)
)


In [ ]:
# ============================================================
# BUCKETS + TABLAS + GRÁFICOS APILADOS
#
# Cada vendor cae en un bucket según SU share AWT0 / AWT null.
#
# Vendors:
#   latest wave + (CL - latest wave)
#
# Orders:
#   suma del % orders CL de los vendors de cada bucket
#   latest wave + (CL - latest wave)
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

BUCKET_EDGES = [
    0, 0.10, 0.20, 0.30, 0.40, 0.50,
    0.60, 0.70, 0.80, 0.90, 1.0000001,
]

BUCKET_LABELS = [
    "[0–10%)",
    "[10–20%)",
    "[20–30%)",
    "[30–40%)",
    "[40–50%)",
    "[50–60%)",
    "[60–70%)",
    "[70–80%)",
    "[80–90%)",
    "[90–100%]",
]

SEGMENT_LATEST = f"{latest_week_wave}"
SEGMENT_REST = "Resto CL"
SEGMENT_ORDER = [
    SEGMENT_LATEST,
    SEGMENT_REST,
]


def build_awt_bucket_summary(
    vendor_df,
    share_col,
    metric_label,
):
    df = vendor_df.copy()

    df = df.loc[
        df[share_col].between(
            0,
            1,
            inclusive="both",
        )
    ].copy()

    df["bucket"] = pd.cut(
        df[share_col],
        bins=BUCKET_EDGES,
        labels=BUCKET_LABELS,
        right=False,
        include_lowest=True,
    )

    df["segment"] = np.where(
        df["is_latest_week_wave"],
        SEGMENT_LATEST,
        SEGMENT_REST,
    )

    grouped = (
        df
        .groupby(
            ["bucket", "segment"],
            observed=True,
            as_index=False,
        )
        .agg(
            vendors=("vendor_code", "nunique"),
            orders=("total_orders", "sum"),
            orders_share_cl=("orders_share_cl", "sum"),
        )
    )

    vendors = (
        grouped
        .pivot_table(
            index="bucket",
            columns="segment",
            values="vendors",
            aggfunc="sum",
            fill_value=0,
            observed=True,
        )
        .reindex(
            index=BUCKET_LABELS,
            columns=SEGMENT_ORDER,
            fill_value=0,
        )
    )

    orders = (
        grouped
        .pivot_table(
            index="bucket",
            columns="segment",
            values="orders",
            aggfunc="sum",
            fill_value=0,
            observed=True,
        )
        .reindex(
            index=BUCKET_LABELS,
            columns=SEGMENT_ORDER,
            fill_value=0,
        )
    )

    orders_share = (
        grouped
        .pivot_table(
            index="bucket",
            columns="segment",
            values="orders_share_cl",
            aggfunc="sum",
            fill_value=0,
            observed=True,
        )
        .reindex(
            index=BUCKET_LABELS,
            columns=SEGMENT_ORDER,
            fill_value=0,
        )
        .mul(100)
    )

    summary = pd.DataFrame(
        index=BUCKET_LABELS
    )

    summary.index.name = "bucket"

    summary["latest_wave_vendors"] = (
        vendors[SEGMENT_LATEST]
        .astype(int)
    )

    summary["rest_cl_vendors"] = (
        vendors[SEGMENT_REST]
        .astype(int)
    )

    summary["total_vendors"] = (
        summary["latest_wave_vendors"]
        + summary["rest_cl_vendors"]
    )

    summary["latest_wave_orders"] = (
        orders[SEGMENT_LATEST]
    )

    summary["rest_cl_orders"] = (
        orders[SEGMENT_REST]
    )

    summary["total_orders"] = (
        summary["latest_wave_orders"]
        + summary["rest_cl_orders"]
    )

    summary["latest_wave_orders_share_cl_pct"] = (
        orders_share[SEGMENT_LATEST]
    )

    summary["rest_cl_orders_share_cl_pct"] = (
        orders_share[SEGMENT_REST]
    )

    summary["total_orders_share_cl_pct"] = (
        summary["latest_wave_orders_share_cl_pct"]
        + summary["rest_cl_orders_share_cl_pct"]
    )

    summary.attrs["metric_label"] = metric_label

    return summary.reset_index()


def plot_vendor_bucket_stack(
    summary,
    metric_label,
):
    x = np.arange(len(summary))

    latest = summary["latest_wave_vendors"].to_numpy()
    rest = summary["rest_cl_vendors"].to_numpy()
    total = summary["total_vendors"].to_numpy()

    fig, ax = plt.subplots(figsize=(15, 7.5))

    bars_latest = ax.bar(
        x,
        latest,
        label=SEGMENT_LATEST,
    )

    bars_rest = ax.bar(
        x,
        rest,
        bottom=latest,
        label=SEGMENT_REST,
    )

    max_total = max(float(total.max()), 1)

    for i, value in enumerate(total):
        if value > 0:
            ax.text(
                i,
                value + max_total * 0.02,
                f"{int(value):,}",
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold",
            )

    for bars in [bars_latest, bars_rest]:
        for bar in bars:
            value = bar.get_height()

            if value > 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_y() + value / 2,
                    f"{int(round(value)):,}",
                    ha="center",
                    va="center",
                    fontsize=9,
                )

    ax.set_xticks(x)
    ax.set_xticklabels(
        summary["bucket"],
        rotation=35,
        ha="right",
    )

    ax.set_ylabel("Vendors")
    ax.set_xlabel(f"Bucket de share {metric_label}")

    ax.set_title(
        f"{metric_label}: vendors CL por bucket\n"
        f"{week_start_awt:%d/%m/%Y}–{week_end_awt:%d/%m/%Y} | "
        f"{SEGMENT_LATEST} vs resto CL"
    )

    ax.legend()

    ax.set_ylim(
        0,
        max_total * 1.16,
    )

    plt.tight_layout()
    plt.show()


def plot_orders_bucket_stack(
    summary,
    metric_label,
):
    x = np.arange(len(summary))

    latest = (
        summary["latest_wave_orders_share_cl_pct"]
        .to_numpy()
    )

    rest = (
        summary["rest_cl_orders_share_cl_pct"]
        .to_numpy()
    )

    total = (
        summary["total_orders_share_cl_pct"]
        .to_numpy()
    )

    fig, ax = plt.subplots(figsize=(15, 7.5))

    bars_latest = ax.bar(
        x,
        latest,
        label=SEGMENT_LATEST,
    )

    bars_rest = ax.bar(
        x,
        rest,
        bottom=latest,
        label=SEGMENT_REST,
    )

    max_total = max(float(total.max()), 1)

    for i, value in enumerate(total):
        if value > 0:
            ax.text(
                i,
                value + max_total * 0.02,
                f"{value:.2f}%",
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold",
            )

    for bars in [bars_latest, bars_rest]:
        for bar in bars:
            value = bar.get_height()

            if value >= 0.10:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_y() + value / 2,
                    f"{value:.2f}%",
                    ha="center",
                    va="center",
                    fontsize=9,
                )

    ax.set_xticks(x)
    ax.set_xticklabels(
        summary["bucket"],
        rotation=35,
        ha="right",
    )

    ax.set_ylabel("% de orders CL")
    ax.set_xlabel(f"Bucket de share {metric_label}")

    ax.set_title(
        f"{metric_label}: distribución del volumen CL por bucket\n"
        f"Suma del % orders CL de los vendors | "
        f"{SEGMENT_LATEST} + (CL − {SEGMENT_LATEST})"
    )

    ax.yaxis.set_major_formatter(
        FuncFormatter(
            lambda value, _:
            f"{value:.0f}%"
        )
    )

    ax.legend()

    ax.set_ylim(
        0,
        max_total * 1.16,
    )

    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# AWT0
# ------------------------------------------------------------

awt0_bucket_analysis = build_awt_bucket_summary(
    vendor_awt_last_week,
    share_col="awt0_share",
    metric_label="AWT0",
)

print("\nAWT0 — resumen por bucket")
display(
    awt0_bucket_analysis.round({
        "latest_wave_orders": 0,
        "rest_cl_orders": 0,
        "total_orders": 0,
        "latest_wave_orders_share_cl_pct": 2,
        "rest_cl_orders_share_cl_pct": 2,
        "total_orders_share_cl_pct": 2,
    })
)

plot_vendor_bucket_stack(
    awt0_bucket_analysis,
    "AWT0",
)

plot_orders_bucket_stack(
    awt0_bucket_analysis,
    "AWT0",
)


# ------------------------------------------------------------
# AWT NULL
# ------------------------------------------------------------

awtnull_bucket_analysis = build_awt_bucket_summary(
    vendor_awt_last_week,
    share_col="awtnull_share",
    metric_label="AWT null",
)

print("\nAWT null — resumen por bucket")
display(
    awtnull_bucket_analysis.round({
        "latest_wave_orders": 0,
        "rest_cl_orders": 0,
        "total_orders": 0,
        "latest_wave_orders_share_cl_pct": 2,
        "rest_cl_orders_share_cl_pct": 2,
        "total_orders_share_cl_pct": 2,
    })
)

plot_vendor_bucket_stack(
    awtnull_bucket_analysis,
    "AWT null",
)

plot_orders_bucket_stack(
    awtnull_bucket_analysis,
    "AWT null",
)


# ------------------------------------------------------------
# CONTROLES
# ------------------------------------------------------------

print(
    "\nControles:"
    f"\n• Suma buckets AWT0: "
    f"{awt0_bucket_analysis['total_orders_share_cl_pct'].sum():.2f}% CL"
    f"\n• Suma buckets AWT null: "
    f"{awtnull_bucket_analysis['total_orders_share_cl_pct'].sum():.2f}% CL"
    f"\n• {SEGMENT_LATEST}: "
    f"{active_latest_wave_vendors:,} vendors con orders en la semana"
)


#### () queries dashboard

In [ ]:
reduction = reduction_raw.copy()

In [ ]:
reduction['start_date'] = reduction['executed_at']

In [ ]:
# Convert 'wave' and 'flag' to string and strip whitespace
reduction['wave'] = reduction['wave'].astype(str).str.strip()
reduction['flag'] = reduction['flag'].astype(str).str.strip()

# Filter out rows where 'wave' is empty or 'nan' after string conversion
reduction = reduction[
    (reduction['wave'] != '') & (reduction['wave'].str.lower() != 'nan')
].copy()

# Filter out rows where 'flag' is 'Manual adjustment'
reduction = reduction[reduction['flag'] != 'Manual adjustment'].copy()

In [ ]:
import pandas as pd

reduction = reduction.copy()

## Convertir start_date a fecha
reduction["start_date"] = pd.to_datetime(
    reduction["start_date"],
    errors="coerce"
)

## Eliminar fechas inválidas
reduction = reduction.dropna(subset=["start_date"])

## # Quedarse solo con los últimos 30 días
# cutoff_date = pd.Timestamp.today().normalize() - pd.Timedelta(days=30)

# reduction_30d = reduction[
#     reduction["start_date"] >= cutoff_date
# ].copy()
reduction_latest_wave = reduction.copy()
# Última wave de cada vendor
reduction_latest_wave = (
    reduction_latest_wave
    .sort_values(
        ["vendor_code", "start_date"],
        ascending=[True, False]
    )
    .drop_duplicates(
        subset=["vendor_code"],
        keep="first"
    )
    .copy()
)

# Crear identificador W6_2026-12-30
reduction_latest_wave["wave_date"] = (
    reduction_latest_wave["start_date"].dt.strftime("%Y-%m-%d")
    + "_"
    + reduction_latest_wave["wave"].astype(str)
)

# Generar CASE WHEN
case_lines = ["CASE"]

for wave_date, df_wave in reduction_latest_wave.groupby(
    "wave_date",
    sort=False
):
    vendors = (
        df_wave["vendor_code"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.replace("'", "''", regex=False)
        .unique()
    )

    vendors_string = ", ".join(f"'{vendor}'" for vendor in vendors)

    case_lines.append(
        f"  WHEN CAST(vendor_code AS STRING) IN ({vendors_string}) "
        f"THEN '{wave_date}'"
    )

case_lines.extend([
    "  ELSE NULL",
    "END"
])

case_query = "\n".join(case_lines)

print(case_query)

In [ ]:
case_lines = ["CASE"]

for flag, df_flag in reduction_latest_wave.groupby(
    "flag",
    sort=False,
    dropna=False
):
    # Ignorar flags nulos o vacíos
    if pd.isna(flag) or str(flag).strip() == "":
        continue

    vendors = (
        df_flag["vendor_code"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.replace("'", "''", regex=False)
        .unique()
    )

    vendors_string = ", ".join(
        f"'{vendor}'"
        for vendor in vendors
    )

    flag_clean = str(flag).strip().replace("'", "''")

    case_lines.append(
        f"  WHEN CAST(vendor_code AS STRING) IN ({vendors_string}) "
        f"THEN '{flag_clean}'"
    )

case_lines.extend([
    "  ELSE NULL",
    "END"
])

case_flag_query = "\n".join(case_lines)

print(case_flag_query)

In [ ]:
import pandas as pd
from datetime import date, timedelta

# Convert 'start_date' to datetime.date objects
reduction['start_date'] = pd.to_datetime(reduction['start_date']).dt.date

# Calculate the date 30 days ago from today
today_date = date.today()
thirty_days_ago_date = today_date - timedelta(days=30)

# Filter for reductions from the last 30 days
reduction = reduction[reduction['start_date'] >= thirty_days_ago_date].copy()

reduction.head()

In [ ]:
# quitar waves de hoy
reduction['start_date'] = pd.to_datetime(reduction['start_date']).dt.date
today_date = date.today()
thirty_days_ago_date = today_date - timedelta(days=0)
reduction = reduction[reduction['start_date'] < thirty_days_ago_date].copy()

reduction.head()

### query maker

## Universo y reglas de esta versión

### `comparison` versus `timings_awt`

| Regla del universo | `comparison` | `timings_awt` |
|---|---:|---:|
| Chile | Sí | Sí |
| `peya_order_id IS NOT NULL` | Sí | Sí |
| `d.is_primary = TRUE` | Sí | Sí |
| `p.is_online = TRUE` | Sí | No |
| Excluir preorders | Sí | No |
| Excluir `courier_business` | Sí | No |

### Regla de waves

- Si un vendor aparece en más de una wave, se conserva únicamente su wave más
  reciente (`latest`).
- Post comienza en `start_date` y usa como máximo 14 días, incluyendo el día de
  inicio.
- Mientras no hayan pasado los 14 días, se usan solo los días disponibles.
- Después del día 14, la wave queda cerrada y sus fechas no siguen creciendo.
- Solo los vendors con órdenes en Pre y Post entran a las Holy Tables; los casos
  perdidos quedan en la auditoría de cohorte.

### Nombres de AWT

- `awt10` y `awt10_ratio`: métrica causal histórica.
- `awt_null`, `awt_0min`, `awt_1_3min`, `awt_4plus`: cuatro buckets no superpuestos usados en tablas y gráficos.

Las tablas por wave son independientes. Si se suman waves con fechas solapadas,
el total `General` puede repetir órdenes entre waves.
### Lectura country level

- `all_chile`: todo el universo CL elegible, incluidas otras waves.
- `resto_de_chile`: excluye vendors de todas las waves seleccionadas.
- Ambos agregados se conservan separados por `vertical_type`; la tabla CL general vuelve a sumarlos y la tabla restaurants filtra antes de calcular.
- `CL limpio`: treatment de la wave actual + `resto_de_chile`.
- El aporte estimado pondera el DID treatment-versus-resto por el peso
  Post de la wave en el denominador nacional de cada métrica.


In [ ]:
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo
from IPython.display import display
import json
import pandas as pd


def build_vendor_pre_post_query(
    reduction_df,
    today=None,
    timezone="America/Santiago",
    max_days=14
):
    """
    Preserva el nombre original de cada wave y devuelve:
    - treatment: vendors incluidos en la wave.
    - control: vendors no intervenidos de las mismas franchises (puede no existir).
    - resto_de_chile: vendors fuera de todas las waves seleccionadas,
      agregados por vertical_type.
    - all_chile: universo CL elegible agregado por vertical_type.

    Si un vendor aparece en varias waves, conserva solo la más reciente.
    Cada wave usa como máximo 14 días Post desde su fecha de inicio.

    Long tail puede pertenecer a treatment aunque no tenga franchise_name.
    courier_business se excluye desde la base.
    """

    required_cols = {
        "wave", "start_date", "franchise_name", "vendor_code"
    }
    missing_cols = required_cols.difference(reduction_df.columns)

    if missing_cols:
        raise ValueError(
            f"Faltan columnas en reduction_df: {sorted(missing_cols)}"
        )

    def parse_date(value):
        if isinstance(value, datetime):
            return value.date()
        if isinstance(value, date):
            return value
        return pd.to_datetime(value, errors="raise").date()

    def bq_string(value):
        # Mismo formato de comparison (26): strings SQL con comillas dobles.
        # Ejemplo: Wendy's -> "Wendy's".
        return json.dumps(str(value), ensure_ascii=False)

    def bq_array(values, indent=0):
        # Lista normal de BigQuery: ["a", "b"].
        # No agrega un tipo explícito delante de la lista.
        clean_values = sorted({
            str(value).strip()
            for value in values
            if pd.notna(value) and str(value).strip()
        })
        return "[" + ", ".join(
            bq_string(value) for value in clean_values
        ) + "]"

    today = (
        datetime.now(ZoneInfo(timezone)).date()
        if today is None
        else parse_date(today)
    )

    reduction_clean = reduction_df.copy()
    reduction_clean["start_date"] = pd.to_datetime(
        reduction_clean["start_date"],
        errors="coerce"
    ).dt.date

    for col in ["wave", "franchise_name", "vendor_code"]:
        reduction_clean[col] = (
            reduction_clean[col]
            .astype("string")
            .str.strip()
            .fillna("")
            .replace({"nan": "", "none": ""})
        )

    reduction_clean["vendor_code"] = (
        reduction_clean["vendor_code"]
        .str.replace(r"\.0$", "", regex=True)
    )

    valid = (
        reduction_clean["start_date"].notna()
        & reduction_clean["wave"].ne("")
        & reduction_clean["wave"].str.lower().ne("nan")
        & reduction_clean["vendor_code"].ne("")
        & reduction_clean["vendor_code"].str.lower().ne("nan")
    )
    reduction_clean = reduction_clean[valid].copy()

    if reduction_clean.empty:
        raise ValueError("No quedaron filas válidas en reduction_df.")

    reduction_latest = (
        reduction_clean
        .sort_values(
            ["vendor_code", "start_date"],
            ascending=[True, False]
        )
        .drop_duplicates("vendor_code", keep="first")
        .copy()
    )

    wave_date_counts = (
        reduction_latest.groupby("wave")["start_date"].nunique()
    )
    invalid_waves = wave_date_counts[wave_date_counts > 1].index.tolist()

    if invalid_waves:
        raise ValueError(
            "Cada wave debe tener una sola start_date. "
            f"Revisar: {invalid_waves}"
        )

    wave_config = (
        reduction_latest[["wave", "start_date"]]
        .drop_duplicates()
        .sort_values(["start_date", "wave"])
    )

    yesterday = today - timedelta(days=1)
    period_structs = []

    for row in wave_config.itertuples(index=False):
        wave = row.wave
        inicio_post = row.start_date

        if inicio_post > yesterday:
            raise ValueError(
                f"{wave}: start_date={inicio_post} es posterior a ayer={yesterday}."
            )

        n_days = min((yesterday - inicio_post).days + 1, max_days)
        if n_days <= 0:
            raise ValueError(f"{wave}: no hay días post disponibles.")

        fin_post = inicio_post + timedelta(days=n_days - 1)
        inicio_pre = inicio_post - timedelta(days=7)
        fin_pre = inicio_pre + timedelta(days=n_days - 1)

        while fin_pre >= inicio_post:
            inicio_pre -= timedelta(days=7)
            fin_pre -= timedelta(days=7)

        wave_rows = reduction_latest[
            reduction_latest["wave"].eq(wave)
        ]

        period_structs.append(f"""    STRUCT(
      {bq_string(wave)} AS wave,
      DATE '{inicio_pre}' AS inicio_pre,
      DATE '{fin_pre}' AS fin_pre,
      DATE '{inicio_post}' AS inicio_post,
      DATE '{fin_post}' AS fin_post,
      {bq_array(wave_rows["franchise_name"], indent=6)} AS franchises,
      {bq_array(wave_rows["vendor_code"], indent=6)} AS treatment_vendors
    )""")

    period_config_sql = ",\n".join(period_structs)
    selected_wave_vendors_sql = bq_array(
        reduction_latest["vendor_code"],
        indent=4
    )
    # Definición exacta de timings_awt:
    # awt_0min = 0; awt_1min = (0, 1]; awt_2min = (1, 2]; ...;
    # awt_20min = (19, 20); awt_21plus = [20, +inf).
    awt_detail_specs = [
        (bucket, f"awt_{bucket}min")
        for bucket in range(2, 21)
    ]

    awt_detail_count_sql = ",\n".join(
        f"""    COUNT(DISTINCT IF(
      awt_bucket = {bucket},
      peya_order_id,
      NULL
    )) AS awt_{bucket}_orders"""
        for bucket, _ in awt_detail_specs
    )

    awt_detail_count_sql += """,
    COUNT(DISTINCT IF(
      awt_bucket = 21,
      peya_order_id,
      NULL
    )) AS awt_21plus_orders"""

    awt_detail_count_select_sql = ",\n".join(
        f"  v.awt_{bucket}_orders"
        for bucket, _ in awt_detail_specs
    )
    awt_detail_count_select_sql += ",\n  v.awt_21plus_orders"

    awt_detail_ratio_select_sql = ",\n".join(
        f"""  ROUND(SAFE_DIVIDE(
    v.awt_{bucket}_orders, v.total_orders
  ), 4) AS {share_col}"""
        for bucket, share_col in awt_detail_specs
    )
    awt_detail_ratio_select_sql += """,
  ROUND(SAFE_DIVIDE(
    v.awt_21plus_orders, v.total_orders
  ), 4) AS awt_21plus"""

    query = f"""
WITH period_config AS (
  SELECT *
  FROM UNNEST([
{period_config_sql}
  ])
),

global_wave_config AS (
  SELECT
    {selected_wave_vendors_sql} AS vendors_in_selected_waves
),

date_bounds AS (
  SELECT
    MIN(inicio_pre) AS min_date,
    MAX(fin_post) AS max_date
  FROM period_config
),

base_raw AS (
  SELECT
    l.peya_order_id,
    CAST(l.vendor.vendor_code AS STRING) AS vendor_code,
    COALESCE(p.partner_name, l.vendor.name) AS store_name,
    l.city.city_name AS city_name,
    CAST(p.franchise.franchise_id AS STRING) AS franchise_id,
    p.franchise.franchise_name AS franchise_name,
    l.vendor.vertical_type AS vertical_type,
    l.food_is_ready_at,
    DATE(l.created_date_local) AS order_date,
    SAFE_DIVIDE(l.estimated_prep_time, 60) AS ept_min,
    SAFE_DIVIDE(l.timings.avoidable_wait_time, 60) AS awt_min,
    SAFE_DIVIDE(
      DATETIME_DIFF(
        d.rider_picked_up_at_local,
        l.sent_to_vendor_at_local,
        SECOND
      ),
      60.0
    ) AS pickup_time_min,
    o.is_slow_order AS slow,
    o.non_seamless_order AS non_seamless

  FROM `peya-bi-tools-pro.il_logistics.fact_logistic_orders` l
  LEFT JOIN UNNEST(l.deliveries) d
  LEFT JOIN `peya-bi-tools-pro.il_core.dim_partner` p
    ON l.vendor.vendor_code = CAST(p.partner_id AS STRING)
  LEFT JOIN `peya-datamarts-pro.dm_fulfillment.non_seamless_delivery_order_level` o
    ON o.platform_order_code = l.peya_order_id
  CROSS JOIN date_bounds db

  WHERE l.country_code = 'cl'
    AND l.peya_order_id IS NOT NULL
    AND d.is_primary = TRUE
    AND p.is_online = TRUE
    AND l.is_preorder = FALSE
    AND COALESCE(l.vendor.vertical_type, '') != 'courier_business'
    AND DATE(l.created_date_local) BETWEEN db.min_date AND db.max_date
),

base AS (
  SELECT
    br.*,
    CASE
      WHEN br.awt_min IS NULL THEN NULL
      WHEN br.awt_min = 0 THEN 0
      WHEN br.awt_min >= 20 THEN 21
      ELSE CAST(CEIL(br.awt_min) AS INT64)
    END AS awt_bucket
  FROM base_raw br
),

base_labeled_raw AS (
  SELECT
    b.*,
    pc.wave,
    pc.inicio_pre,
    pc.fin_pre,
    pc.inicio_post,
    pc.fin_post,

    CASE
      WHEN b.order_date BETWEEN pc.inicio_pre AND pc.fin_pre THEN 'pre'
      WHEN b.order_date BETWEEN pc.inicio_post AND pc.fin_post THEN 'post'
    END AS period,

    CASE
      WHEN b.vendor_code IN UNNEST(pc.treatment_vendors)
        THEN 'treatment'
      WHEN b.franchise_name IN UNNEST(pc.franchises)
       AND b.vendor_code NOT IN UNNEST(g.vendors_in_selected_waves)
        THEN 'control'
      -- resto_de_chile se construye aparte para que también incluya
      -- los vendors de control. Aquí no se usa como categoría excluyente.
      ELSE 'exclude'
    END AS analysis_group

  FROM base b
  CROSS JOIN period_config pc
  CROSS JOIN global_wave_config g

  WHERE
    b.order_date BETWEEN pc.inicio_pre AND pc.fin_pre
    OR b.order_date BETWEEN pc.inicio_post AND pc.fin_post
),

-- Referencia nacional independiente: todos los vendors que no están en
-- ninguna wave. Un vendor de control también pertenece a esta referencia.
resto_labeled_raw AS (
  SELECT
    b.*,
    pc.wave,
    pc.inicio_pre,
    pc.fin_pre,
    pc.inicio_post,
    pc.fin_post,
    CASE
      WHEN b.order_date BETWEEN pc.inicio_pre AND pc.fin_pre THEN 'pre'
      WHEN b.order_date BETWEEN pc.inicio_post AND pc.fin_post THEN 'post'
    END AS period,
    'resto_de_chile' AS analysis_group
  FROM base b
  CROSS JOIN period_config pc
  CROSS JOIN global_wave_config g
  WHERE b.vendor_code NOT IN UNNEST(g.vendors_in_selected_waves)
    AND (
      b.order_date BETWEEN pc.inicio_pre AND pc.fin_pre
      OR b.order_date BETWEEN pc.inicio_post AND pc.fin_post
    )
),

-- Total nacional independiente. Incluye treatment, otras waves,
-- control y resto; sirve para medir el cambio CL realmente observado.
all_chile_labeled_raw AS (
  SELECT
    b.*,
    pc.wave,
    pc.inicio_pre,
    pc.fin_pre,
    pc.inicio_post,
    pc.fin_post,
    CASE
      WHEN b.order_date BETWEEN pc.inicio_pre AND pc.fin_pre THEN 'pre'
      WHEN b.order_date BETWEEN pc.inicio_post AND pc.fin_post THEN 'post'
    END AS period,
    'all_chile' AS analysis_group
  FROM base b
  CROSS JOIN period_config pc
  WHERE
    b.order_date BETWEEN pc.inicio_pre AND pc.fin_pre
    OR b.order_date BETWEEN pc.inicio_post AND pc.fin_post
),

base_labeled_union AS (
  SELECT * FROM base_labeled_raw
  UNION ALL
  SELECT * FROM resto_labeled_raw
  UNION ALL
  SELECT * FROM all_chile_labeled_raw
),

chile_period_orders AS (
  SELECT
    wave,
    period,
    COUNT(DISTINCT peya_order_id) AS cl_total_orders
  FROM base_labeled_raw
  WHERE period IS NOT NULL
  GROUP BY
    wave,
    period
),

base_labeled AS (
  SELECT
    peya_order_id,
    wave,
    inicio_pre,
    fin_pre,
    inicio_post,
    fin_post,
    period,
    analysis_group,

    CASE
      WHEN analysis_group = 'resto_de_chile' THEN '__RESTO_DE_CHILE__'
      WHEN analysis_group = 'all_chile' THEN '__ALL_CHILE__'
      ELSE franchise_id
    END AS franchise_id,
    CASE
      WHEN analysis_group = 'resto_de_chile' THEN 'resto_de_chile'
      WHEN analysis_group = 'all_chile' THEN 'all_chile'
      ELSE franchise_name
    END AS franchise_name,
    CASE
      WHEN analysis_group = 'resto_de_chile' THEN '__RESTO_DE_CHILE__'
      WHEN analysis_group = 'all_chile' THEN '__ALL_CHILE__'
      ELSE vendor_code
    END AS vendor_code,
    CASE
      WHEN analysis_group = 'resto_de_chile' THEN 'resto_de_chile'
      WHEN analysis_group = 'all_chile' THEN 'all_chile'
      ELSE store_name
    END AS store_name,
    CASE
      WHEN analysis_group IN ('resto_de_chile', 'all_chile') THEN 'Chile'
      ELSE city_name
    END AS city_name,
    -- Mantener el vertical real permite reconstruir tanto CL total
    -- como el impacto nacional específico de restaurants.
    vertical_type AS vertical_type,

    food_is_ready_at,
    ept_min,
    awt_min,
    awt_bucket,
    pickup_time_min,
    slow,
    non_seamless

  FROM base_labeled_union
  WHERE analysis_group != 'exclude'
),

base_order_keys AS (
  SELECT DISTINCT
    CAST(peya_order_id AS STRING) AS order_id,
    wave,
    inicio_pre,
    fin_pre,
    inicio_post,
    fin_post,
    period,
    analysis_group,
    franchise_id,
    franchise_name,
    vendor_code,
    store_name,
    city_name,
    vertical_type
  FROM base_labeled
),

vendor_metrics AS (
  SELECT
    wave,
    analysis_group,
    franchise_id,
    franchise_name,
    vendor_code,
    store_name,
    city_name,
    vertical_type,
    period,
    inicio_pre,
    fin_pre,
    inicio_post,
    fin_post,

    COUNT(DISTINCT peya_order_id) AS total_orders,
    COUNT(DISTINCT IF(
      food_is_ready_at IS NOT NULL, peya_order_id, NULL
    )) AS orders_with_fir,
    SAFE_DIVIDE(
      COUNT(DISTINCT IF(
        food_is_ready_at IS NOT NULL, peya_order_id, NULL
      )),
      COUNT(DISTINCT peya_order_id)
    ) AS fir,
    COUNT(DISTINCT IF(slow = 1, peya_order_id, NULL)) AS slow_orders,
    COUNT(DISTINCT IF(
      non_seamless = TRUE, peya_order_id, NULL
    )) AS non_seamless_orders,
    COUNT(DISTINCT IF(
      awt_min IS NULL, peya_order_id, NULL
    )) AS awt_null_orders,
    COUNT(DISTINCT IF(
      awt_bucket = 0, peya_order_id, NULL
    )) AS awt_0_orders,
    COUNT(DISTINCT IF(
      awt_bucket = 1, peya_order_id, NULL
    )) AS awt_1_orders,
{awt_detail_count_sql},
    -- Se conserva para compatibilidad: buckets 2 o superiores.
    COUNT(DISTINCT IF(
      awt_bucket >= 2, peya_order_id, NULL
    )) AS awt_2plus_orders,

    SUM(ept_min) AS ept_sum,
    COUNT(DISTINCT IF(
      ept_min IS NOT NULL, peya_order_id, NULL
    )) AS ept_orders,
    SUM(awt_min) AS awt_sum,
    COUNT(DISTINCT IF(
      awt_min IS NOT NULL, peya_order_id, NULL
    )) AS awt_orders,
    SUM(pickup_time_min) AS pickup_time_sum,
    COUNT(DISTINCT IF(
      pickup_time_min IS NOT NULL, peya_order_id, NULL
    )) AS pickup_time_orders,

    ROUND(SAFE_DIVIDE(
      SUM(ept_min),
      COUNT(DISTINCT IF(ept_min IS NOT NULL, peya_order_id, NULL))
    ), 2) AS ept,
    ROUND(SAFE_DIVIDE(
      SUM(awt_min),
      COUNT(DISTINCT IF(awt_min IS NOT NULL, peya_order_id, NULL))
    ), 2) AS awt,
    ROUND(
      SAFE_DIVIDE(
        SUM(ept_min),
        COUNT(DISTINCT IF(ept_min IS NOT NULL, peya_order_id, NULL))
      )
      + SAFE_DIVIDE(
        SUM(awt_min),
        COUNT(DISTINCT IF(awt_min IS NOT NULL, peya_order_id, NULL))
      ),
      2
    ) AS tt,
    ROUND(SAFE_DIVIDE(
      SUM(pickup_time_min),
      COUNT(DISTINCT IF(
        pickup_time_min IS NOT NULL, peya_order_id, NULL
      ))
    ), 2) AS ctp

  FROM base_labeled
  GROUP BY
    wave,
    analysis_group,
    franchise_id,
    franchise_name,
    vendor_code,
    store_name,
    city_name,
    vertical_type,
    period,
    inicio_pre,
    fin_pre,
    inicio_post,
    fin_post
),

ns_metrics AS (
  SELECT
    bok.wave,
    bok.analysis_group,
    bok.franchise_id,
    bok.franchise_name,
    bok.vendor_code,
    bok.store_name,
    bok.city_name,
    bok.vertical_type,
    bok.period,
    bok.inicio_pre,
    bok.fin_pre,
    bok.inicio_post,
    bok.fin_post,

    COUNT(DISTINCT n.order_id) AS ns_distinct_orders,
    COUNT(DISTINCT IF(
      n.SEAMLESS = 0, n.order_id, NULL
    )) AS ns_non_seamless_orders,
    COUNT(DISTINCT IF(
      n.SEAMLESS = 1, n.order_id, NULL
    )) AS ns_seamless_orders,
    SUM(COALESCE(n.PARTNER_PERFORMANCE, 0)) AS partner_perfo,

    SUM(
      COALESCE(n.TR_HIGH_PREP_TIME, 0)
        + COALESCE(n.TR_HIGH_PREP_TIME_AND_DISTANCE, 0) / 2
        + COALESCE(
            n.TR_HIGH_PREP_TIME_AND_DISTANCE_COMBINATION, 0
          ) / 2
    ) AS high_preptimes,

    SUM(
      IF(
        IFNULL(n.undispatch_reason, '') != 'Late Order Preparation',
        COALESCE(n.TR_STAFFING_AND_PARTNER_PERFO, 0) / 2
          + COALESCE(
              n.TR_STAFFING_RIDER_AND_PARTNER_PERFO, 0
            ) / 3
          + COALESCE(n.TR_RIDER_AND_PARTNER_PERFO, 0) / 2
          + COALESCE(n.TR_PARTNER_PERFO, 0),
        0
      )
    ) AS awt10,

    SUM(COALESCE(n.PREDICTION_MODEL, 0)) AS prediction_model

  FROM `peya-delivery-and-support.automated_tables_reports.non_seamless_reasons` n
  CROSS JOIN date_bounds db
  INNER JOIN base_order_keys bok
    ON CAST(n.order_id AS STRING) = bok.order_id

  WHERE n.country_name = 'Chile'
    AND n.created_date_local BETWEEN db.min_date AND db.max_date

  GROUP BY
    bok.wave,
    bok.analysis_group,
    bok.franchise_id,
    bok.franchise_name,
    bok.vendor_code,
    bok.store_name,
    bok.city_name,
    bok.vertical_type,
    bok.period,
    bok.inicio_pre,
    bok.fin_pre,
    bok.inicio_post,
    bok.fin_post
)

SELECT
  v.wave,
  v.analysis_group,
  v.franchise_id,
  v.franchise_name,
  v.vendor_code,
  v.store_name,
  v.city_name,
  v.vertical_type,
  v.period,
  v.inicio_pre,
  v.fin_pre,
  v.inicio_post,
  v.fin_post,
  c.cl_total_orders,

  v.total_orders,
  v.orders_with_fir,
  ROUND(v.fir, 4) AS fir,
  v.slow_orders,
  ROUND(SAFE_DIVIDE(
    v.slow_orders, v.total_orders
  ), 4) AS slow_orders_ratio,
  v.non_seamless_orders,
  ROUND(SAFE_DIVIDE(
    v.non_seamless_orders, v.total_orders
  ), 4) AS non_seamless_orders_ratio,
  v.awt_null_orders,
  v.awt_0_orders,
  v.awt_1_orders,
{awt_detail_count_select_sql},
  v.awt_2plus_orders,
  ROUND(SAFE_DIVIDE(
    v.awt_null_orders, v.total_orders
  ), 4) AS awt_null,
  ROUND(SAFE_DIVIDE(
    v.awt_0_orders, v.total_orders
  ), 4) AS awt_0min,
  ROUND(SAFE_DIVIDE(
    v.awt_1_orders, v.total_orders
  ), 4) AS awt_1min,
{awt_detail_ratio_select_sql},
  ROUND(SAFE_DIVIDE(
    v.awt_2plus_orders, v.total_orders
  ), 4) AS awt_2plus,
  v.ept_sum,
  v.ept_orders,
  v.awt_sum,
  v.awt_orders,
  v.pickup_time_sum,
  v.pickup_time_orders,
  v.ept,
  v.awt,
  v.tt,
  v.ctp,

  COALESCE(ns.ns_distinct_orders, 0) AS ns_distinct_orders,
  COALESCE(ns.ns_non_seamless_orders, 0) AS ns_non_seamless_orders,
  COALESCE(ns.ns_seamless_orders, 0) AS ns_seamless_orders,

  ROUND(COALESCE(ns.partner_perfo, 0), 4) AS partner_perfo,
  ROUND(SAFE_DIVIDE(
    COALESCE(ns.partner_perfo, 0), v.total_orders
  ), 4) AS partner_perfo_ratio,

  ROUND(COALESCE(ns.high_preptimes, 0), 4) AS high_preptimes,
  ROUND(SAFE_DIVIDE(
    COALESCE(ns.high_preptimes, 0), v.total_orders
  ), 4) AS high_preptimes_ratio,

  ROUND(COALESCE(ns.awt10, 0), 4) AS awt10,
  ROUND(SAFE_DIVIDE(
    COALESCE(ns.awt10, 0), v.total_orders
  ), 4) AS awt10_ratio,

  ROUND(COALESCE(ns.prediction_model, 0), 4) AS prediction_model,
  ROUND(SAFE_DIVIDE(
    COALESCE(ns.prediction_model, 0), v.total_orders
  ), 4) AS prediction_model_ratio

FROM vendor_metrics v
LEFT JOIN chile_period_orders c
  ON v.wave = c.wave
 AND v.period = c.period
LEFT JOIN ns_metrics ns
  ON v.wave = ns.wave
 AND v.analysis_group = ns.analysis_group
 AND COALESCE(v.franchise_id, '__NULL_FRANCHISE_ID__')
     = COALESCE(ns.franchise_id, '__NULL_FRANCHISE_ID__')
 AND COALESCE(v.franchise_name, '__NULL_FRANCHISE__')
     = COALESCE(ns.franchise_name, '__NULL_FRANCHISE__')
 AND v.vendor_code = ns.vendor_code
 AND COALESCE(v.city_name, '__NULL_CITY__')
     = COALESCE(ns.city_name, '__NULL_CITY__')
 AND COALESCE(v.vertical_type, 'NULL_VERTICAL')
     = COALESCE(ns.vertical_type, 'NULL_VERTICAL')
 AND v.period = ns.period
 AND v.inicio_pre = ns.inicio_pre
 AND v.fin_pre = ns.fin_pre
 AND v.inicio_post = ns.inicio_post
 AND v.fin_post = ns.fin_post

ORDER BY
  v.inicio_post,
  CASE v.analysis_group
    WHEN 'treatment' THEN 1
    WHEN 'control' THEN 2
    WHEN 'resto_de_chile' THEN 3
    WHEN 'all_chile' THEN 4
  END,
  v.franchise_name,
  v.store_name,
  CASE v.period
    WHEN 'pre' THEN 1
    WHEN 'post' THEN 2
  END
"""

    return query.strip()

In [ ]:
REDUCT_DATE_COL = "start_date"
REDUCT_FRANCHISE_COL = "franchise_name"
REDUCT_VENDOR_COL = "vendor_code"
REDUCT_WAVE_COL = "wave"
REDUCT_FLAG_COL = "flag"

reduct_base = reduction.copy()

required_cols = [
    REDUCT_WAVE_COL,
    REDUCT_DATE_COL,
    REDUCT_FRANCHISE_COL,
    REDUCT_VENDOR_COL,
    REDUCT_FLAG_COL
]
missing_cols = [
    col for col in required_cols
    if col not in reduct_base.columns
]

if missing_cols:
    raise ValueError(
        f"Faltan columnas en reduction: {missing_cols}"
    )

reduct_base[REDUCT_DATE_COL] = pd.to_datetime(
    reduct_base[REDUCT_DATE_COL],
    errors="coerce"
).dt.date

for col in [
    REDUCT_WAVE_COL,
    REDUCT_FRANCHISE_COL,
    REDUCT_VENDOR_COL,
    REDUCT_FLAG_COL
]:
    reduct_base[col] = (
        reduct_base[col].astype(str).str.strip()
    )

# W9, Week32, etc. se conservan exactamente como vienen en reduction.
wave_config_preview = (
    reduct_base[required_cols]
    .drop_duplicates()
    .sort_values(
        [REDUCT_DATE_COL, REDUCT_WAVE_COL, REDUCT_FRANCHISE_COL]
    )
)
# display(wave_config_preview)

query = build_vendor_pre_post_query(
    reduct_base,
    max_days=14,
)
# print(query)


In [ ]:
# input_confirmation = input("¿Ya tomaste los resultados de la query y actualizaste los EPT pre y post en la hoja de cálculo? \n https://docs.google.com/spreadsheets/d/1RwbpetVhmSIIqf78Jx2hm_wffgXOxsxDWqvMQDcVrEw \n Escribe 'sí' para continuar: ")

# if input_confirmation.lower() != 'si':
#     raise ValueError("Por favor, actualiza los EPTs en la hoja de cálculo y vuelve a ejecutar esta celda.")

# print("Confirmación recibida. Continuando con el procesamiento.")

In [ ]:
!gcloud config get-value project

### pre/post df

In [ ]:
# from datetime import datetime
# from zoneinfo import ZoneInfo

# file_name = datetime.now(
#     ZoneInfo("America/Santiago")
# ).strftime("%Y-%m-%d.csv")

# print(file_name)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# file_name = '2026-09-02.csv'

# file_path_main = '/content/drive/MyDrive/lower ept y awt/deprecated/'
# file_path_main = '/content/drive/MyDrive/lower ept y awt/comparison_data/'
# file_path = file_path_main + file_name
# df_pre_post = pd.read_csv(file_path)
# df_pre_post.head(3)

In [ ]:
import pandas as pd
project_id = 'peya-chile'

# Ejecuta la consulta en BigQuery y descarga los datos a Colab
df_pre_post = pd.read_gbq(query, project_id=project_id)

# Muestra las primeras 5 filas del resultado
df_pre_post.head()

In [ ]:
# # desde google sheets
# import gspread
# from google.colab import auth
# import google.auth

# auth.authenticate_user()

# credentials, _ = google.auth.default(scopes=[
#     "https://www.googleapis.com/auth/spreadsheets",
#     "https://www.googleapis.com/auth/drive"
# ])

# gc = gspread.authorize(credentials)
# sheet_id =  "1RwbpetVhmSIIqf78Jx2hm_wffgXOxsxDWqvMQDcVrEw"
# sheet_name = "Sheet1"

# worksheet = gc.open_by_key(sheet_id).worksheet(sheet_name)
# data = worksheet.get_all_values()
# df_pre_post = pd.DataFrame(data[1:], columns=data[0])

# df_pre_post.head()

In [ ]:
# # EPT actuales
# file_name = 'W10_WW32_ANALYSE.csv'
# file_name = 'last_waves_martes_.csv'
# file_name = 'last_waves_martes_final.csv'

# from google.colab import drive
# drive.mount('/content/drive')
# file_path_main = '/content/drive/MyDrive/lower ept y awt/ept reductions input data/'
# file_path = file_path_main + file_name
# df_pre_post = pd.read_csv(file_path)
# df_pre_post.head(3)

In [ ]:
print("Filas originales:", len(df_pre_post))

# Seguridad adicional para CSVs generados con una versión anterior.
if "vertical_type" in df_pre_post.columns:
    df_pre_post = df_pre_post[
        df_pre_post["vertical_type"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .ne("courier_business")
    ].copy()

required_query_columns = {
    "ept_sum", "ept_orders",
    "awt_sum", "awt_orders",
    "pickup_time_sum", "pickup_time_orders",
    "awt10", "awt10_ratio",
}
missing_query_columns = required_query_columns.difference(
    df_pre_post.columns
)

if missing_query_columns:
    raise ValueError(
        "El CSV no viene de la query corregida. Vuelve a ejecutar la query "
        f"y carga el resultado nuevo. Faltan: {sorted(missing_query_columns)}"
    )

print("Filas sin courier_business:", len(df_pre_post))
required_analysis_groups = {
    "treatment",
    "resto_de_chile",
    "all_chile",
}
present_analysis_groups = set(
    df_pre_post["analysis_group"].dropna().astype(str).str.strip()
)
missing_analysis_groups = (
    required_analysis_groups - present_analysis_groups
)

if missing_analysis_groups:
    raise ValueError(
        "El CSV debe volver a generarse con la query nueva. "
        "Faltan grupos: "
        f"{sorted(missing_analysis_groups)}"
    )


In [ ]:
# df_pre_post = pd.read_excel(file_path_external, sheet_name='all_pre_post')
df_pre = df_pre_post[df_pre_post['period']=='pre']
df_post = df_pre_post[df_pre_post['period']=='post']

In [ ]:
df_pre.head()

In [ ]:
df_post.head()

In [ ]:
df_post['wave'].unique()

In [ ]:
reduction

# Data Processing

### functions

build_franchise_summary()

In [ ]:
def weighted_average(df, value_col, weight_col):
    if value_col not in df.columns or weight_col not in df.columns:
        return np.nan

    temp_df = df[[value_col, weight_col]].dropna()
    if temp_df.empty:
        return np.nan

    total_weight = temp_df[weight_col].sum()
    if total_weight == 0:
        return np.nan

    return (temp_df[value_col] * temp_df[weight_col]).sum() / total_weight


def safe_ratio(num, den):
    if pd.isna(den) or den == 0:
        return np.nan
    return num / den


def build_franchise_summary(
    df,
    metric_config,
    group_cols=("wave", "franchise_id", "franchise_name", "analysis_group"),
    post_weight_col="total_orders",
    pre_weight_col="total_orders_0",
    pre_suffix="_0",
    ratio_scale=100,
    strict_group_cols=True,
):
    """
    Agrega primero numeradores y denominadores y luego divide.

    aggregate_mean se usa para tiempos: por ejemplo,
    SUM(awt_min) / COUNT(orders con AWT no nulo).
    """
    df = df.copy()
    df = df.loc[:, ~df.columns.duplicated()]

    missing_group_cols = [col for col in group_cols if col not in df.columns]
    if strict_group_cols and missing_group_cols:
        raise ValueError(f"Faltan columnas de agrupación: {missing_group_cols}")

    group_cols = [col for col in group_cols if col in df.columns]
    if not group_cols:
        raise ValueError("No quedó ninguna columna válida para groupby.")

    def metric_type(spec):
        return spec if isinstance(spec, str) else spec.get("type")

    def group_sum(g, col):
        return g[col].sum(min_count=1) if col in g.columns else np.nan

    def summarize_group(g):
        out = {}

        for metric, spec in metric_config.items():
            kind = metric_type(spec)

            if kind in ["weighted_avg", "weighted_average", "weighted"]:
                out[f"avg_{metric}_post"] = weighted_average(
                    g, metric, post_weight_col
                )
                out[f"avg_{metric}_pre"] = weighted_average(
                    g, f"{metric}{pre_suffix}", pre_weight_col
                )

            elif kind == "aggregate_mean":
                num_col = spec["num"]
                den_col = spec["den"]
                out[f"avg_{metric}_post"] = safe_ratio(
                    group_sum(g, num_col), group_sum(g, den_col)
                )
                out[f"avg_{metric}_pre"] = safe_ratio(
                    group_sum(g, f"{num_col}{pre_suffix}"),
                    group_sum(g, f"{den_col}{pre_suffix}"),
                )

            elif kind == "derived_sum":
                components = spec["components"]
                out[f"avg_{metric}_post"] = sum(
                    safe_ratio(
                        group_sum(g, metric_config[item]["num"]),
                        group_sum(g, metric_config[item]["den"]),
                    )
                    for item in components
                )
                out[f"avg_{metric}_pre"] = sum(
                    safe_ratio(
                        group_sum(g, f"{metric_config[item]['num']}{pre_suffix}"),
                        group_sum(g, f"{metric_config[item]['den']}{pre_suffix}"),
                    )
                    for item in components
                )

            elif kind in ["nominal", "sum"]:
                out[f"{metric}_post"] = group_sum(g, metric)
                out[f"{metric}_pre"] = group_sum(
                    g, f"{metric}{pre_suffix}"
                )

            elif kind == "ratio":
                num_col = spec["num"]
                den_col = spec["den"]
                out[f"avg_{metric}_post"] = (
                    safe_ratio(group_sum(g, num_col), group_sum(g, den_col))
                    * ratio_scale
                )
                out[f"avg_{metric}_pre"] = (
                    safe_ratio(
                        group_sum(g, f"{num_col}{pre_suffix}"),
                        group_sum(g, f"{den_col}{pre_suffix}"),
                    )
                    * ratio_scale
                )
            else:
                raise ValueError(f"Unsupported metric type: {kind}")

        return pd.Series(out)

    summary = (
        df.groupby(group_cols, dropna=False)
        .apply(summarize_group, include_groups=False)
        .reset_index()
    )

    for metric, spec in metric_config.items():
        kind = metric_type(spec)
        if kind in [
            "weighted_avg", "weighted_average", "weighted",
            "aggregate_mean", "derived_sum", "ratio",
        ]:
            post_col = f"avg_{metric}_post"
            pre_col = f"avg_{metric}_pre"
        else:
            post_col = f"{metric}_post"
            pre_col = f"{metric}_pre"

        if kind == "ratio":
            summary[f"{metric}_diff_pp"] = summary[post_col] - summary[pre_col]
        else:
            summary[f"{metric}_diff_nominal"] = summary[post_col] - summary[pre_col]
            summary[f"{metric}_diff_percent"] = (
                summary[f"{metric}_diff_nominal"] / summary[pre_col]
            ).replace([np.inf, -np.inf], np.nan)

    return summary


 build_merged_pre_post_reduction()

In [ ]:
def build_merged_pre_post_reduction(
    df_post,
    df_pre,
    reduction,
    key="vendor_code",
    pre_suffix="_0",
    franchise_col="franchise_name",
    wave_col="wave"
):
    post = df_post.copy()
    pre = df_pre.copy()
    red = reduction.copy()

    for df in [post, pre, red]:
        if key in df.columns:
            df[key] = (
                df[key]
                .astype(str)
                .str.replace(r"\.0$", "", regex=True)
                .str.strip()
            )

    # Conservar la wave más reciente de cada vendor.
    sort_cols = []
    ascending = []

    if franchise_col in red.columns:
        sort_cols.append(franchise_col)
        ascending.append(True)

    sort_cols.append(key)
    ascending.append(True)

    if "start_date" in red.columns:
        red["start_date"] = pd.to_datetime(
            red["start_date"],
            errors="coerce"
        )
        sort_cols.append("start_date")
        ascending.append(False)
    elif wave_col in red.columns:
        red["_wave_order"] = pd.to_numeric(
            red[wave_col]
            .astype(str)
            .str.extract(r"(\d+)")[0],
            errors="coerce"
        )
        sort_cols.append("_wave_order")
        ascending.append(False)

    red = red.sort_values(
        sort_cols,
        ascending=ascending,
        na_position="last"
    )

    dedup_subset = [key]
    if franchise_col in red.columns:
        dedup_subset = [franchise_col, key]

    red = red.drop_duplicates(
        subset=dedup_subset,
        keep="first"
    ).drop(columns=["_wave_order"], errors="ignore")

    pre = pre.rename(columns={
        col: f"{col}{pre_suffix}"
        for col in pre.columns
        if col != key and not col.endswith(pre_suffix)
    })

    df_merged = post.merge(
        pre,
        on=key,
        how="left",
        validate="m:1"
    )

    reduction_merge_keys = [key]
    if (
        franchise_col in df_merged.columns
        and franchise_col in red.columns
    ):
        reduction_merge_keys = [franchise_col, key]

    df_merged = df_merged.merge(
        red,
        on=reduction_merge_keys,
        how="left",
        suffixes=("", "_reduction"),
        validate="m:1"
    )

    duplicated_reduction_cols = [
        col for col in df_merged.columns
        if col.endswith("_reduction")
    ]

    df_merged = df_merged.drop(
        columns=duplicated_reduction_cols
    )
    df_merged = df_merged.loc[
        :,
        ~df_merged.columns.duplicated()
    ]

    return df_merged


In [ ]:
#

# def build_merged_pre_post_reduction(
#     df_post,
#     df_pre,
#     reduction,
#     key="vendor_code",
#     pre_suffix="_0",
#     wave_col="wave"
# ):
#     post = df_post.copy()
#     pre = df_pre.copy()
#     red = reduction.copy()

#     merge_keys = [wave_col, key]

#     for df_name, df in {
#         "df_post": post,
#         "df_pre": pre
#     }.items():
#         missing = [c for c in merge_keys if c not in df.columns]
#         if missing:
#             raise ValueError(f"{df_name} no tiene columnas requeridas: {missing}")

#     if key not in red.columns:
#         raise ValueError(f"reduction no tiene columna requerida: {key}")

#     # normalizar vendor_code
#     post[key] = post[key].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
#     pre[key] = pre[key].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
#     red[key] = red[key].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()

#     # normalizar wave
#     post[wave_col] = post[wave_col].astype(str).str.strip()
#     pre[wave_col] = pre[wave_col].astype(str).str.strip()

#     # renombrar columnas pre
#     pre = pre.rename(columns={
#         col: f"{col}{pre_suffix}"
#         for col in pre.columns
#         if col not in merge_keys and not col.endswith(pre_suffix)
#     })

#     # validar que pre sea único por wave + vendor_code
#     duplicated_pre = (
#         pre
#         .groupby(merge_keys)
#         .size()
#         .reset_index(name="n")
#         .query("n > 1")
#     )

#     if not duplicated_pre.empty:
#         raise ValueError(
#             "df_pre tiene más de una fila por wave + vendor_code. "
#             "Revisa estos casos:\n"
#             + duplicated_pre.head(20).to_string(index=False)
#         )

#     # merge post + pre
#     df_merged = post.merge(
#         pre,
#         on=merge_keys,
#         how="left",
#         validate="m:1"
#     )

#     # marcar si vendor_code está en reduction, sin hacer merge
#     reduction_vendor_codes = set(red[key].dropna().unique())

#     df_merged["is_in_reduction"] = df_merged[key].isin(reduction_vendor_codes)

#     return df_merged

 plot_slow_nonseamless_by_group()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch




def plot_slow_nonseamless_by_group(df_plot, title_suffix="", figsize=(8, 6),rotation=0):
    df_plot = df_plot.copy()

    df_plot["Franchise_Treatment"] = (
        df_plot["franchise_name"].astype(str)
        + " (T"
        + df_plot["treatment"].astype(int).astype(str)
        + ")"
    )

    df_plot = df_plot.sort_values(["franchise_name", "treatment"])

    fig, ax1 = plt.subplots(figsize=figsize)

    # ------------------------------------------------------------
    # Slow Orders Ratio - bars
    # ------------------------------------------------------------

    slow_orders_melted = df_plot.melt(
        id_vars=["Franchise_Treatment", "franchise_name", "treatment"],
        value_vars=["avg_slow_orders_ratio_pre", "avg_slow_orders_ratio_post"],
        var_name="Slow_Orders_Period",
        value_name="Average_Slow_Orders_Ratio"
    )

    slow_orders_melted["Slow_Orders_Period"] = slow_orders_melted["Slow_Orders_Period"].map({
        "avg_slow_orders_ratio_pre": "Pre-Intervention",
        "avg_slow_orders_ratio_post": "Post-Intervention"
    })

    # ------------------------------------------------------------
    # Non-Seamless Orders Ratio - dots
    # ------------------------------------------------------------

    non_seamless_melted = df_plot.melt(
        id_vars=["Franchise_Treatment", "franchise_name", "treatment"],
        value_vars=[
            "avg_non_seamless_orders_ratio_pre",
            "avg_non_seamless_orders_ratio_post"
        ],
        var_name="Non_Seamless_Period",
        value_name="Average_Non_Seamless_Orders_Ratio"
    )

    non_seamless_melted["Non_Seamless_Period"] = non_seamless_melted["Non_Seamless_Period"].map({
        "avg_non_seamless_orders_ratio_pre": "Pre-Intervention",
        "avg_non_seamless_orders_ratio_post": "Post-Intervention"
    })

    # ------------------------------------------------------------
    # Manual X-axis positioning
    # ------------------------------------------------------------

    x_positions = []
    x_labels = []
    x_coord_map = {}

    bar_width = 0.35
    group_spacing = 0.2
    major_group_spacing = 0.6

    current_x = 0
    unique_ft_groups = df_plot["Franchise_Treatment"].unique()

    for i, ft_group in enumerate(unique_ft_groups):

        treatment_value = int(
            df_plot.loc[
                df_plot["Franchise_Treatment"] == ft_group,
                "treatment"
            ].iloc[0]
        )

        franchise_label = df_plot.loc[
            df_plot["Franchise_Treatment"] == ft_group,
            "franchise_name"
        ].iloc[0]

        treatment_suffix = " (T)" if treatment_value == 1 else ""

        # ---------------- PRE ----------------

        pre_slow = slow_orders_melted[
            (slow_orders_melted["Franchise_Treatment"] == ft_group)
            & (slow_orders_melted["Slow_Orders_Period"] == "Pre-Intervention")
        ]["Average_Slow_Orders_Ratio"].iloc[0]

        ax1.bar(
            current_x,
            pre_slow,
            bar_width,
            color="skyblue",
            align="center",
            label="Slow Orders Pre" if i == 0 else ""
        )

        ax1.text(
            current_x,
            pre_slow + 0.5,
            f"{pre_slow:.0f}%",
            ha="center",
            va="bottom",
            color="black",
            fontsize=13
        )

        x_positions.append(current_x)
        x_labels.append(f"{franchise_label} - Pre{treatment_suffix}")
        x_coord_map[(ft_group, "Pre-Intervention")] = current_x

        current_x += bar_width + group_spacing

        # ---------------- POST ----------------

        post_slow = slow_orders_melted[
            (slow_orders_melted["Franchise_Treatment"] == ft_group)
            & (slow_orders_melted["Slow_Orders_Period"] == "Post-Intervention")
        ]["Average_Slow_Orders_Ratio"].iloc[0]

        ax1.bar(
            current_x,
            post_slow,
            bar_width,
            color="lightcoral",
            align="center",
            label="Slow Orders Post" if i == 0 else ""
        )

        ax1.text(
            current_x,
            post_slow + 0.5,
            f"{post_slow:.0f}%",
            ha="center",
            va="bottom",
            color="black",
            fontsize=13
        )

        x_positions.append(current_x)
        x_labels.append(f"{franchise_label} - Post{treatment_suffix}")
        x_coord_map[(ft_group, "Post-Intervention")] = current_x

        post_x = current_x

        current_x += bar_width + major_group_spacing

        # línea después de cada T1-Post
        if treatment_value == 1 and i < len(unique_ft_groups) - 1:
            ax1.axvline(
                x=post_x + (bar_width / 2) + (major_group_spacing / 2),
                color="grey",
                linestyle="--",
                linewidth=0.8
            )

    # ------------------------------------------------------------
    # Non-Seamless Orders Ratio - dots + line
    # ------------------------------------------------------------

    for j, ft_group in enumerate(unique_ft_groups):

        pre_seamless = non_seamless_melted[
            (non_seamless_melted["Franchise_Treatment"] == ft_group)
            & (non_seamless_melted["Non_Seamless_Period"] == "Pre-Intervention")
        ]["Average_Non_Seamless_Orders_Ratio"].iloc[0]

        post_seamless = non_seamless_melted[
            (non_seamless_melted["Franchise_Treatment"] == ft_group)
            & (non_seamless_melted["Non_Seamless_Period"] == "Post-Intervention")
        ]["Average_Non_Seamless_Orders_Ratio"].iloc[0]

        x_pre = x_coord_map[(ft_group, "Pre-Intervention")]
        x_post = x_coord_map[(ft_group, "Post-Intervention")]

        ax1.plot(
            [x_pre, x_post],
            [pre_seamless, post_seamless],
            color="forestgreen",
            linestyle="-",
            alpha=0.7,
            zorder=1
        )

        ax1.scatter(
            x_pre,
            pre_seamless,
            color="forestgreen",
            s=70,
            zorder=2,
            label="Non-Seamless Orders" if j == 0 else "",
            marker="o"
        )

        ax1.text(
            x_pre,
            pre_seamless + 0.5,
            f"{pre_seamless:.0f}%",
            ha="center",
            va="bottom",
            fontsize=13,
            color="forestgreen"
        )

        ax1.scatter(
            x_post,
            post_seamless,
            color="forestgreen",
            s=70,
            zorder=2,
            marker="o"
        )

        ax1.text(
            x_post,
            post_seamless + 0.5,
            f"{post_seamless:.0f}%",
            ha="center",
            va="bottom",
            fontsize=13,
            color="forestgreen"
        )

    # ------------------------------------------------------------
    # Format
    # ------------------------------------------------------------

    max_y_value = max(
        slow_orders_melted["Average_Slow_Orders_Ratio"].max(),
        non_seamless_melted["Average_Non_Seamless_Orders_Ratio"].max()
    )

    ax1.set_ylim(0, max_y_value + 2)

    ax1.set_xticks(x_positions)
    ax1.set_xticklabels(x_labels, rotation=rotation, fontsize=10)

    ax1.set_title(
        f"Average Slow and Non-Seamless Orders Ratio Pre vs Post {title_suffix}",
        pad=18
    )

    # ax1.set_xlabel("Franchise and Treatment Group")
    ax1.get_yaxis().set_visible(False)
    ax1.set_ylabel("")
    ax1.grid(axis="y", linestyle="--", alpha=0.7)

    legend_elements = [
        Patch(facecolor="skyblue", label="Slow Orders Pre"),
        Patch(facecolor="lightcoral", label="Slow Orders Post"),
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="forestgreen",
            label="Non-Seamless Orders",
            markerfacecolor="forestgreen",
            markersize=8
        )
    ]

    ax1.legend(
        handles=legend_elements,
        loc="upper left",
        # bbox_to_anchor=(0.75, 1),
        title="Metrics and Period"
    )

    plt.tight_layout(rect=[0, 0.03, 0.85, 0.98])
    plt.show()

 plot_ept_awt_ctp_by_group()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch




def plot_ept_awt_ctp_by_group(
    df_plot,
    title_suffix="",
    figsize=(14, 6),
    rotation=0
):
    df_plot = df_plot.copy()

    df_plot["Franchise_Treatment"] = (
        df_plot["franchise_name"].astype(str)
        + " (T"
        + df_plot["treatment"].astype(int).astype(str)
        + ")"
    )

    df_plot = df_plot.sort_values(["franchise_name", "treatment"])

    plt.figure(figsize=figsize)
    ax = plt.gca()

    # ------------------------------------------------------------
    # EPT + AWT stacked bars
    # ------------------------------------------------------------

    melted_ept_awt_df = df_plot.melt(
        id_vars=["Franchise_Treatment", "franchise_name", "treatment"],
        value_vars=["avg_ept_pre", "avg_awt_pre", "avg_ept_post", "avg_awt_post"],
        var_name="Metric_Period",
        value_name="Value"
    )

    melted_ept_awt_df["Metric"] = melted_ept_awt_df["Metric_Period"].apply(
        lambda x: "EPT" if "ept" in x else "AWT"
    )

    melted_ept_awt_df["Period"] = melted_ept_awt_df["Metric_Period"].apply(
        lambda x: "Pre-Intervention" if "pre" in x else "Post-Intervention"
    )

    x_positions = []
    x_labels = []
    x_coord_map = {}

    bar_width = 0.35
    group_spacing = 0.2
    major_group_spacing = 0.6

    current_x = 0
    unique_ft_groups = df_plot["Franchise_Treatment"].unique()

    for i, ft_group in enumerate(unique_ft_groups):
        ft_data = melted_ept_awt_df[
            melted_ept_awt_df["Franchise_Treatment"] == ft_group
        ]

        treatment_value = int(
            df_plot.loc[
                df_plot["Franchise_Treatment"] == ft_group,
                "treatment"
            ].iloc[0]
        )

        franchise_label = df_plot.loc[
            df_plot["Franchise_Treatment"] == ft_group,
            "franchise_name"
        ].iloc[0]

        treatment_suffix = " (T)" if treatment_value == 1 else ""

        # ---------------- PRE ----------------

        pre_data = ft_data[ft_data["Period"] == "Pre-Intervention"]

        if not pre_data.empty:
            ept_pre = pre_data[pre_data["Metric"] == "EPT"]["Value"].iloc[0]
            awt_pre = pre_data[pre_data["Metric"] == "AWT"]["Value"].iloc[0]

            ax.bar(
                current_x,
                ept_pre,
                bar_width,
                color="skyblue",
                align="center",
                label="EPT" if i == 0 else ""
            )

            ax.text(
                current_x,
                ept_pre / 2,
                f"{ept_pre:.1f}",
                ha="center",
                va="center",
                color="black",
                fontsize=13
            )

            ax.bar(
                current_x,
                awt_pre,
                bar_width,
                bottom=ept_pre,
                color="lightcoral",
                align="center",
                label="AWT" if i == 0 else ""
            )

            ax.text(
                current_x,
                ept_pre + awt_pre / 2,
                f"{awt_pre:.1f}",
                ha="center",
                va="center",
                color="black",
                fontsize=13
            )

            x_positions.append(current_x)
            x_labels.append(f"{franchise_label} - Pre{treatment_suffix}")
            x_coord_map[(ft_group, "Pre-Intervention")] = current_x

            current_x += bar_width + group_spacing

        # ---------------- POST ----------------

        post_data = ft_data[ft_data["Period"] == "Post-Intervention"]

        if not post_data.empty:
            ept_post = post_data[post_data["Metric"] == "EPT"]["Value"].iloc[0]
            awt_post = post_data[post_data["Metric"] == "AWT"]["Value"].iloc[0]

            ax.bar(
                current_x,
                ept_post,
                bar_width,
                color="skyblue",
                align="center"
            )

            ax.text(
                current_x,
                ept_post / 2,
                f"{ept_post:.1f}",
                ha="center",
                va="center",
                color="black",
                fontsize=13
            )

            ax.bar(
                current_x,
                awt_post,
                bar_width,
                bottom=ept_post,
                color="lightcoral",
                align="center"
            )

            ax.text(
                current_x,
                ept_post + awt_post / 2,
                f"{awt_post:.1f}",
                ha="center",
                va="center",
                color="black",
                fontsize=13
            )

            x_positions.append(current_x)
            x_labels.append(f"{franchise_label} - Post{treatment_suffix}")
            x_coord_map[(ft_group, "Post-Intervention")] = current_x

            post_x = current_x

            current_x += bar_width + major_group_spacing

            # línea después de cada T1 - Post
            if treatment_value == 1 and i < len(unique_ft_groups) - 1:
                ax.axvline(
                    x=post_x + (bar_width / 2) + (major_group_spacing / 2),
                    color="grey",
                    linestyle="--",
                    linewidth=0.8
                )

    # ------------------------------------------------------------
    # CTP dots + connected line
    # ------------------------------------------------------------

    ctp_melted = df_plot.melt(
        id_vars=["Franchise_Treatment", "franchise_name", "treatment"],
        value_vars=["avg_ctp_pre", "avg_ctp_post"],
        var_name="ctp_Period",
        value_name="Average_ctp"
    )

    ctp_melted["ctp_Period"] = ctp_melted["ctp_Period"].map({
        "avg_ctp_pre": "Pre-Intervention",
        "avg_ctp_post": "Post-Intervention"
    })

    for j, ft_group in enumerate(ctp_melted["Franchise_Treatment"].unique()):
        pre_value = ctp_melted[
            (ctp_melted["Franchise_Treatment"] == ft_group)
            & (ctp_melted["ctp_Period"] == "Pre-Intervention")
        ]["Average_ctp"].iloc[0]

        post_value = ctp_melted[
            (ctp_melted["Franchise_Treatment"] == ft_group)
            & (ctp_melted["ctp_Period"] == "Post-Intervention")
        ]["Average_ctp"].iloc[0]

        x_pre = x_coord_map[(ft_group, "Pre-Intervention")]
        x_post = x_coord_map[(ft_group, "Post-Intervention")]

        ax.plot(
            [x_pre, x_post],
            [pre_value, post_value],
            color="black",
            linestyle=":",
            alpha=0.7,
            zorder=1
        )

        ax.scatter(
            x_pre,
            pre_value,
            color="black",
            s=60,
            zorder=2,
            label="CTP" if j == 0 else ""
        )

        ax.text(
            x_pre,
            pre_value + 0.5,
            f"{pre_value:.1f}",
            ha="center",
            va="bottom",
            fontsize=13,
            color="black"
        )

        ax.scatter(
            x_post,
            post_value,
            color="black",
            s=60,
            zorder=2
        )

        ax.text(
            x_post,
            post_value + 0.5,
            f"{post_value:.1f}",
            ha="center",
            va="bottom",
            fontsize=13,
            color="black"
        )

    # ------------------------------------------------------------
    # Format
    # ------------------------------------------------------------

    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_labels, rotation=rotation,  fontsize=10)

    ax.set_title(f"Average EPT, AWT, and CTP Pre vs Post {title_suffix}")
    ax.grid(axis="y", linestyle="-", alpha=0.2)

    ax.get_yaxis().set_visible(False)
    ax.set_ylabel("")

    legend_elements = [
        Patch(facecolor="skyblue", label="EPT"),
        Patch(facecolor="lightcoral", label="AWT"),
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            label="CTP",
            markerfacecolor="black",
            markersize=10
        )
    ]

    ax.legend(
        handles=legend_elements,
        loc="upper left",
        # bbox_to_anchor=(0.8, 1),
        title="Metrics"
    )

    max_stacked_pre = (
        df_plot["avg_ept_pre"] + df_plot["avg_awt_pre"]
    ).max()

    max_stacked_post = (
        df_plot["avg_ept_post"] + df_plot["avg_awt_post"]
    ).max()

    max_ctp = df_plot[
        ["avg_ctp_pre", "avg_ctp_post"]
    ].max().max()

    max_y_value = max(max_stacked_pre, max_stacked_post, max_ctp)

    ax.set_ylim(0, max_y_value * 1.1)

    plt.tight_layout(rect=[0, 0.03, 0.85, 0.94])
    plt.show()

 plot_ept_awt_ctp_total_and_city()

In [ ]:
def plot_ept_awt_ctp_total_and_city(
    df_merged,
    metric_config,
    franchise_summary=None,
    city_col="ciudad",
    city_labels=None,
    figsize=(14, 6),
    rotation=0
):
    """
    Genera 3 gráficos:
    1. Total
    2. Santiago
    3. Regiones

    Requiere que exista build_franchise_summary().
    """

    if city_labels is None:
        city_labels = {
            1: "Santiago",
            0: "Regiones"
        }

    # ------------------------------------------------------------
    # 1. Summary total, si no viene precalculado
    # ------------------------------------------------------------

    if franchise_summary is None:
        franchise_summary = build_franchise_summary(
            df=df_merged,
            metric_config=metric_config,
            group_cols=("franchise_id", "franchise_name", "treatment"),
            post_weight_col="total_orders",
            pre_weight_col="total_orders_0"
        )

        franchise_summary = franchise_summary.round(2)

    # gráfico total
    plot_ept_awt_ctp_by_group(
        df_plot=franchise_summary,
        title_suffix="- Total",
        figsize=figsize,
        rotation=rotation
    )

    # ------------------------------------------------------------
    # 2. Summary por ciudad
    # ------------------------------------------------------------

    franchise_summary_city = build_franchise_summary(
        df=df_merged,
        metric_config=metric_config,
        group_cols=("franchise_id", "franchise_name", city_col, "treatment"),
        post_weight_col="total_orders",
        pre_weight_col="total_orders_0"
    )

    franchise_summary_city = franchise_summary_city.round(2)

    # ------------------------------------------------------------
    # 3. Gráficos por ciudad
    # ------------------------------------------------------------

    for city_flag in [1, 0]:
        df_city = franchise_summary_city[
            franchise_summary_city[city_col] == city_flag
        ]

        if df_city.empty:
            continue

        city_label = city_labels.get(city_flag, str(city_flag))

        plot_ept_awt_ctp_by_group(
            df_plot=df_city,
            title_suffix=f"- {city_label}",
            figsize=figsize,
            rotation=rotation
        )

    return franchise_summary_city

 build_diff_table()

In [ ]:
def build_diff_table(
    franchise_summary,
    index_cols=("wave", "franchise_id", "franchise_name"),
    group_col="analysis_group"
):
    metric_cols = [
        "ept_diff_nominal",
        "awt_diff_nominal",
        "tt_diff_nominal",
        "ctp_diff_nominal",
        "slow_orders_ratio_diff_pp",
        "non_seamless_orders_ratio_diff_pp",
        "awt_null_diff_pp",
        "awt_0min_diff_pp",
        "awt_1_3min_diff_pp",
        "awt_4plus_diff_pp",
        "partner_perfo_diff_nominal",
        "high_preptimes_diff_nominal",
        "awt10_diff_nominal",
        "prediction_model_diff_nominal",
        "partner_perfo_ratio_diff_pp",
        "high_preptimes_ratio_diff_pp",
        "awt10_ratio_diff_pp",
        "prediction_model_ratio_diff_pp"
    ]

    df = franchise_summary.copy()
    index_cols = list(index_cols)
    metric_cols = [col for col in metric_cols if col in df.columns]

    missing_cols = [
        col for col in index_cols + [group_col]
        if col not in df.columns
    ]
    if missing_cols:
        raise ValueError(f"Faltan columnas para pivotear: {missing_cols}")

    dupes = df[
        df.duplicated(index_cols + [group_col], keep=False)
    ].sort_values(index_cols + [group_col])

    if not dupes.empty:
        display(dupes[index_cols + [group_col]])
        raise ValueError(
            f"Hay duplicados en {index_cols + [group_col]}. "
            "El pivot requiere una fila única por combinación."
        )

    table = df.pivot(
        index=index_cols,
        columns=group_col,
        values=metric_cols
    )

    suffix_map = {
        "treatment": "_T",
        "control": "",
        "resto_de_chile": "_R"
    }

    new_columns = []

    for metric_name, analysis_group in table.columns:
        suffix = suffix_map.get(
            analysis_group,
            "_" + str(analysis_group)
        )

        clean_metric_name = (
            metric_name
            .replace("_diff_nominal", "_diff")
            .replace("_diff_pp", "_diff_pp")
            .replace("_orders_ratio_", "_")
        )

        new_columns.append(f"{clean_metric_name}{suffix}")

    table.columns = new_columns

    return table.reset_index().round(2)


 table_simplify()

In [ ]:
def table_simplify(df, source_df=None):
    id_cols = [
        col for col in ["wave", "franchise_id", "franchise_name", "ciudad"]
        if col in df.columns
    ]

    simplified = (
        df[id_cols].copy()
        if id_cols
        else pd.DataFrame(index=df.index)
    )

    if source_df is not None:
        stats_df = source_df.copy()
        valid_groups = [
            "treatment", "control", "resto_de_chile"
        ]
        stats_df = stats_df[
            stats_df["analysis_group"].isin(valid_groups)
        ].copy()

        group_cols = id_cols.copy()
        missing_group_cols = [
            col for col in group_cols
            if col not in stats_df.columns
        ]

        if missing_group_cols:
            raise ValueError(
                "source_df no tiene estas columnas necesarias: "
                f"{missing_group_cols}"
            )

        date_cols = [
            "inicio_pre", "fin_pre", "inicio_post", "fin_post"
        ]
        for col in date_cols:
            if col in stats_df.columns:
                stats_df[col] = pd.to_datetime(
                    stats_df[col],
                    errors="coerce"
                ).dt.date

        total_stats = (
            stats_df
            .groupby(group_cols, dropna=False)
            .agg(
                inicio_pre=("inicio_pre", "min"),
                fin_pre=("fin_pre", "max"),
                inicio_post=("inicio_post", "min"),
                fin_post=("fin_post", "max"),
                franchises=("franchise_name", "nunique"),
                vendors=("vendor_code", "nunique"),
                post_orders=("total_orders", "sum"),
                pre_orders=("total_orders_0", "sum")
            )
            .reset_index()
        )

        total_stats["dias_pre_activo"] = (
            pd.to_datetime(total_stats["fin_pre"])
            - pd.to_datetime(total_stats["inicio_pre"])
        ).dt.days + 1

        total_stats["dias_post_activo"] = (
            pd.to_datetime(total_stats["fin_post"])
            - pd.to_datetime(total_stats["inicio_post"])
        ).dt.days + 1

        group_stats = (
            stats_df
            .groupby(
                group_cols + ["analysis_group"],
                dropna=False
            )
            .agg(
                franchises=("franchise_name", "nunique"),
                vendors=("vendor_code", "nunique"),
                post_orders=("total_orders", "sum"),
                pre_orders=("total_orders_0", "sum")
            )
            .reset_index()
        )

        group_stats_wide = group_stats.pivot(
            index=group_cols,
            columns="analysis_group",
            values=[
                "franchises",
                "vendors",
                "post_orders",
                "pre_orders"
            ]
        )

        group_stats_wide.columns = [
            f"{metric}_{group}"
            for metric, group in group_stats_wide.columns
        ]
        group_stats_wide = group_stats_wide.reset_index()

        total_stats = total_stats.merge(
            group_stats_wide,
            on=group_cols,
            how="left"
        )

        simplified = simplified.merge(
            total_stats,
            on=group_cols,
            how="left"
        )

    control_available = (
        "ept_diff" in df.columns
        and df["ept_diff"].notna().any()
    )
    rest_available = (
        "ept_diff_R" in df.columns
        and df["ept_diff_R"].notna().any()
    )

    if control_available:
        simplified["reference_group"] = "control"
    elif rest_available:
        simplified["reference_group"] = "resto_de_chile"

    def pair_col(
        out_col,
        treatment_col,
        control_col,
        rest_col
    ):
        if treatment_col not in df.columns:
            return

        reference = pd.Series(
            np.nan,
            index=df.index,
            dtype=float
        )

        if control_col in df.columns:
            reference = reference.fillna(df[control_col])

        if rest_col in df.columns:
            reference = reference.fillna(df[rest_col])

        if reference.notna().any():
            simplified[out_col] = (
                df[treatment_col].round(2).astype(str)
                + " ("
                + reference.round(2).astype(str)
                + ")"
            )
        else:
            simplified[out_col] = df[treatment_col].round(2)

    metric_pairs = {
        "delta_ept": ("ept_diff_T", "ept_diff", "ept_diff_R"),
        "delta_awt": ("awt_diff_T", "awt_diff", "awt_diff_R"),
        "delta_tt": ("tt_diff_T", "tt_diff", "tt_diff_R"),
        "delta_ctp": ("ctp_diff_T", "ctp_diff", "ctp_diff_R"),
        "delta_slow_pp": (
            "slow_diff_pp_T", "slow_diff_pp", "slow_diff_pp_R"
        ),
        "delta_non_seamless_pp": (
            "non_seamless_diff_pp_T",
            "non_seamless_diff_pp",
            "non_seamless_diff_pp_R"
        ),
        "delta_awt_null_pp": (
            "awt_null_diff_pp_T", "awt_null_diff_pp", "awt_null_diff_pp_R"
        ),
        "delta_awt_0min_pp": (
            "awt_0min_diff_pp_T", "awt_0min_diff_pp", "awt_0min_diff_pp_R"
        ),
        "delta_awt_1_3min_pp": (
            "awt_1_3min_diff_pp_T", "awt_1_3min_diff_pp", "awt_1_3min_diff_pp_R"
        ),
        "delta_awt_4plus_pp": (
            "awt_4plus_diff_pp_T", "awt_4plus_diff_pp", "awt_4plus_diff_pp_R"
        ),
        "delta_partner_perfo_pp": (
            "partner_perfo_ratio_diff_pp_T",
            "partner_perfo_ratio_diff_pp",
            "partner_perfo_ratio_diff_pp_R"
        ),
        "delta_high_preptimes_pp": (
            "high_preptimes_ratio_diff_pp_T",
            "high_preptimes_ratio_diff_pp",
            "high_preptimes_ratio_diff_pp_R"
        ),
        "delta_awt10_pp": (
            "awt10_ratio_diff_pp_T",
            "awt10_ratio_diff_pp",
            "awt10_ratio_diff_pp_R"
        ),
        "delta_prediction_model_pp": (
            "prediction_model_ratio_diff_pp_T",
            "prediction_model_ratio_diff_pp",
            "prediction_model_ratio_diff_pp_R"
        )
    }

    for out_col, cols in metric_pairs.items():
        pair_col(out_col, *cols)

    return simplified


 table_simplify_treatment()

In [ ]:
def table_simplify_treatment(df, source_df=None):
    id_cols = [
        col for col in [
            "wave", "flag", "franchise_id", "franchise_name", "ciudad"
        ]
        if col in df.columns
    ]

    simplified = df[id_cols].copy() if id_cols else pd.DataFrame(index=df.index)

    # --------------------------------------------------
    # 1. Stats extra SOLO treatment desde source_df
    # --------------------------------------------------
    if source_df is not None:
        stats_df = source_df.copy()

        stats_df = stats_df[
            stats_df["analysis_group"] == "treatment"
        ].copy()

        group_cols = id_cols.copy()

        missing_group_cols = [
            col for col in group_cols
            if col not in stats_df.columns
        ]

        if missing_group_cols:
            raise ValueError(
                f"source_df no tiene estas columnas necesarias: {missing_group_cols}"
            )

        date_cols = [
            "inicio_pre",
            "fin_pre",
            "inicio_post",
            "fin_post"
        ]

        existing_date_cols = [
            col for col in date_cols
            if col in stats_df.columns
        ]

        for col in existing_date_cols:
            stats_df[col] = pd.to_datetime(stats_df[col], errors="coerce").dt.date

        agg_dict = {}

        if "inicio_pre" in stats_df.columns:
            agg_dict["inicio_pre"] = ("inicio_pre", "min")

        if "fin_pre" in stats_df.columns:
            agg_dict["fin_pre"] = ("fin_pre", "max")

        if "inicio_post" in stats_df.columns:
            agg_dict["inicio_post"] = ("inicio_post", "min")

        if "fin_post" in stats_df.columns:
            agg_dict["fin_post"] = ("fin_post", "max")

        if "franchise_name" in stats_df.columns:
            agg_dict["franchises"] = ("franchise_name", "nunique")

        if "vendor_code" in stats_df.columns:
            agg_dict["vendors"] = ("vendor_code", "nunique")

        if "total_orders" in stats_df.columns:
            agg_dict["post_orders"] = ("total_orders", "sum")

        if "total_orders_0" in stats_df.columns:
            agg_dict["pre_orders"] = ("total_orders_0", "sum")

        # El total CL se repite por vendor: se toma MAX, nunca SUM.
        if "cl_total_orders" in stats_df.columns:
            agg_dict["post_cl_orders"] = ("cl_total_orders", "max")

        if "cl_total_orders_0" in stats_df.columns:
            agg_dict["pre_cl_orders"] = ("cl_total_orders_0", "max")

        treatment_stats = (
            stats_df
            .groupby(group_cols, dropna=False)
            .agg(**agg_dict)
            .reset_index()
        )

        if {"inicio_pre", "fin_pre"}.issubset(treatment_stats.columns):
            treatment_stats["dias_pre_activo"] = (
                pd.to_datetime(treatment_stats["fin_pre"])
                - pd.to_datetime(treatment_stats["inicio_pre"])
            ).dt.days + 1

        if {"inicio_post", "fin_post"}.issubset(treatment_stats.columns):
            treatment_stats["dias_post_activo"] = (
                pd.to_datetime(treatment_stats["fin_post"])
                - pd.to_datetime(treatment_stats["inicio_post"])
            ).dt.days + 1

        if {"pre_orders", "pre_cl_orders"}.issubset(treatment_stats.columns):
            treatment_stats["pre_orders_cl_share"] = np.where(
                treatment_stats["pre_cl_orders"] > 0,
                treatment_stats["pre_orders"]
                / treatment_stats["pre_cl_orders"],
                np.nan
            )

        if {"post_orders", "post_cl_orders"}.issubset(treatment_stats.columns):
            treatment_stats["post_orders_cl_share"] = np.where(
                treatment_stats["post_cl_orders"] > 0,
                treatment_stats["post_orders"]
                / treatment_stats["post_cl_orders"],
                np.nan
            )

        simplified = simplified.merge(
            treatment_stats,
            on=group_cols,
            how="left"
        )

    # --------------------------------------------------
    # 2. Deltas SOLO treatment
    # --------------------------------------------------
    def treatment_col(out_col, col):
        if col in df.columns:
            simplified[out_col] = df[col].round(2)

    treatment_col("delta_ept", "ept_diff_T")
    treatment_col("delta_awt", "awt_diff_T")
    # treatment_col("delta_tt", "tt_diff_T")
    treatment_col("delta_ctp", "ctp_diff_T")

    treatment_col("delta_non_seamless_pp", "non_seamless_diff_pp_T")
    # treatment_col("delta_slow_pp", "slow_diff_pp_T")
    treatment_col("delta_partner_perfo_pp", "partner_perfo_ratio_diff_pp_T")
    treatment_col("delta_high_preptimes_pp", "high_preptimes_ratio_diff_pp_T")
    treatment_col("delta_awt10_pp", "awt10_ratio_diff_pp_T")
    treatment_col("delta_prediction_model_pp", "prediction_model_ratio_diff_pp_T")

    treatment_col("delta_awt_null_pp", "awt_null_diff_pp_T")
    treatment_col("delta_awt_0min_pp", "awt_0min_diff_pp_T")
    treatment_col("delta_awt_1_3min_pp", "awt_1_3min_diff_pp_T")
    treatment_col("delta_awt_4plus_pp", "awt_4plus_diff_pp_T")


    return simplified


 format_wave_treatment_table()

In [ ]:
def format_wave_treatment_table(delta_table_treatment):
    """
    Formatea la Holy Table.

    - Tabla por wave: una columna por wave.
    - Tabla wave + flag: encabezado multinivel real (wave, flag).
      No concatena nombres del tipo "wave | flag".
    """

    df = delta_table_treatment.copy()
    has_flag = "flag" in df.columns

    df["_wave_order"] = pd.to_numeric(
        df["wave"].astype(str).str.extract(r"(\d+)")[0],
        errors="coerce",
    )

    sort_cols = ["_wave_order", "wave"]
    if has_flag:
        sort_cols.append("flag")

    df = (
        df.sort_values(sort_cols)
        .drop(columns="_wave_order")
        .copy()
    )

    month_map = {
        1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
        5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
        9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec",
    }

    def fmt_date(value):
        value = pd.to_datetime(value, errors="coerce")
        if pd.isna(value):
            return ""
        return f"{value.day:02d}/{month_map[value.month]}"

    def fmt_range(start_col, end_col):
        return (
            df[start_col].apply(fmt_date)
            + "-"
            + df[end_col].apply(fmt_date)
        )

    if {"inicio_pre", "fin_pre"}.issubset(df.columns):
        df["Pre"] = fmt_range("inicio_pre", "fin_pre")

    if {"inicio_post", "fin_post"}.issubset(df.columns):
        df["Post"] = fmt_range("inicio_post", "fin_post")

    col_map = {
        "franchises": "franchises",
        "vendors": "vendors",
        "dias_post_activo": "días evaluados",
        "post_orders": "Post #Orders",
        "pre_orders": "Pre #Orders",
        "pre_orders_cl_share": "% Orders CL Pre",
        "post_orders_cl_share": "% Orders CL Post",
        "delta_ept": "Δ ept (min)",
        "delta_awt": "Δ awt (min)",
        "delta_ctp": "Δ pickup time (min)",
        "delta_high_preptimes_pp": "Δ high preptimes (pp)",
        "delta_awt10_pp": "Δ AWT10 causal (pp)",
        "delta_awt_null_pp": "Δ awt_null (pp)",
        "delta_awt_0min_pp": "Δ awt_0min (pp)",
        "delta_awt_1_3min_pp": "Δ awt_1_3min (pp)",
        "delta_awt_4plus_pp": "Δ awt_4plus (pp)",
        "Pre": "Pre",
        "Post": "Post",
    }

    existing_cols = [
        column for column in col_map
        if column in df.columns
    ]

    identifier_cols = ["wave"]
    if has_flag:
        identifier_cols.append("flag")

    out = df[identifier_cols + existing_cols].copy()

    def fmt_int(value):
        if pd.isna(value):
            return ""
        return f"{value:.0f}"

    def fmt_1(value):
        if pd.isna(value):
            return ""
        return f"{value:.1f}"

    def fmt_2(value):
        if pd.isna(value):
            return ""
        return f"{value:.2f}"

    def fmt_pct(value):
        if pd.isna(value):
            return ""
        return f"{value * 100:.1f}%"

    int_cols = [
        "franchises",
        "vendors",
        "dias_post_activo",
        "post_orders",
        "pre_orders",
    ]

    one_decimal_cols = [
        "delta_ept",
        "delta_awt",
        "delta_ctp",
        "delta_high_preptimes_pp",
        "delta_awt10_pp",
        "delta_awt_null_pp",
        "delta_awt_0min_pp",
        "delta_awt_1_3min_pp",
        "delta_awt_4plus_pp",
    ]

    two_decimal_cols = [
        "delta_tt",
        "delta_partner_perfo_pp",
        "delta_prediction_model_pp",
    ]

    pct_cols = [
        "pre_orders_cl_share",
        "post_orders_cl_share",
    ]

    for column in int_cols:
        if column in out.columns:
            out[column] = pd.to_numeric(
                out[column], errors="coerce"
            ).map(fmt_int)

    for column in one_decimal_cols:
        if column in out.columns:
            out[column] = pd.to_numeric(
                out[column], errors="coerce"
            ).map(fmt_1)

    for column in two_decimal_cols:
        if column in out.columns:
            out[column] = pd.to_numeric(
                out[column], errors="coerce"
            ).map(fmt_2)

    for column in pct_cols:
        if column in out.columns:
            out[column] = pd.to_numeric(
                out[column], errors="coerce"
            ).map(fmt_pct)

    out = out.rename(columns=col_map)

    if has_flag:
        duplicate_mask = out.duplicated(["wave", "flag"], keep=False)
        if duplicate_mask.any():
            duplicates = out.loc[duplicate_mask, ["wave", "flag"]]
            raise ValueError(
                "Hay más de una fila por wave + flag:\n"
                + duplicates.to_string(index=False)
            )

        out = out.set_index(["wave", "flag"]).T

        # Así se visualiza como:
        # metric | Week32 | Week32 | Week34 ...
        # flag   | criterio 1 | criterio 2 ...
        out.columns.names = ["metric", "flag"]
        out.index.name = None

    else:
        duplicate_mask = out.duplicated(["wave"], keep=False)
        if duplicate_mask.any():
            duplicates = out.loc[duplicate_mask, ["wave"]]
            raise ValueError(
                "Hay más de una fila por wave:\n"
                + duplicates.to_string(index=False)
            )

        out = out.set_index("wave").T
        out.columns.name = None
        out.index.name = "metric"

    return out


 build_wave_summary()

In [ ]:
def build_wave_summary(
    df_eval,
    metric_config,
    city_filter=None,
    analysis_groups=(
        "treatment",
        "control",
        "resto_de_chile"
    )
):
    df = df_eval.copy()

    # resto_de_chile es una referencia nacional: no se fuerza dentro
    # de los cortes Santiago/Regiones.
    if city_filter is not None:
        df = df[
            df["analysis_group"].isin(["treatment", "control"])
        ].copy()
        df = df[df["ciudad"] == city_filter].copy()
    else:
        df = df[
            df["analysis_group"].isin(analysis_groups)
        ].copy()

    if df.empty:
        raise ValueError(
            "No quedaron filas para construir el wave summary."
        )

    summary = build_franchise_summary(
        df=df,
        metric_config=metric_config,
        group_cols=("wave", "analysis_group"),
        post_weight_col="total_orders",
        pre_weight_col="total_orders_0"
    )

    return summary.round(2)


def build_wave_summary_city(
    df_eval,
    metric_config
):
    df = df_eval.copy()

    df = df[
        df["analysis_group"].isin(["treatment", "control"])
    ].copy()

    summary = build_franchise_summary(
        df=df,
        metric_config=metric_config,
        group_cols=("wave", "ciudad", "analysis_group"),
        post_weight_col="total_orders",
        pre_weight_col="total_orders_0"
    )

    return summary.round(2)


def build_wave_diff_table(wave_summary):
    return build_diff_table(
        franchise_summary=wave_summary,
        index_cols=("wave",),
        group_col="analysis_group"
    )


def build_wave_city_diff_table(wave_summary_city):
    return build_diff_table(
        franchise_summary=wave_summary_city,
        index_cols=("wave", "ciudad"),
        group_col="analysis_group"
    )



def build_wave_flag_eval(
    df_eval,
    flag_col="flag",
    missing_flag_label="SIN_FLAG"
):
    """
    Crea el universo para comparar cada wave+flag.

    - treatment conserva la flag real de cada vendor.
    - control y resto_de_chile se replican dentro de cada flag de su wave
      para que funcionen como la misma referencia de tendencia.
    """
    required_cols = {"wave", "analysis_group", flag_col}
    missing_cols = required_cols.difference(df_eval.columns)

    if missing_cols:
        raise ValueError(
            f"Faltan columnas para wave+flag: {sorted(missing_cols)}"
        )

    df = df_eval.copy()
    treatment = df[
        df["analysis_group"].eq("treatment")
    ].copy()

    if treatment.empty:
        raise ValueError(
            "No hay filas treatment para construir la tabla wave+flag."
        )

    clean_flag = treatment[flag_col].astype("string").str.strip()
    treatment[flag_col] = clean_flag.mask(
        clean_flag.isna()
        | clean_flag.eq("")
        | clean_flag.str.lower().eq("nan"),
        missing_flag_label
    )

    wave_flags = (
        treatment[["wave", flag_col]]
        .drop_duplicates()
    )

    references = df[
        df["analysis_group"].isin(["control", "resto_de_chile"])
    ].copy()
    references = references.drop(columns=[flag_col], errors="ignore")
    references = references.merge(
        wave_flags,
        on="wave",
        how="inner",
        validate="m:m"
    )

    return pd.concat(
        [treatment, references],
        ignore_index=True,
        sort=False
    )


def build_wave_flag_summary(df_eval_wave_flag, metric_config):
    return build_franchise_summary(
        df=df_eval_wave_flag,
        metric_config=metric_config,
        group_cols=("wave", "flag", "analysis_group"),
        post_weight_col="total_orders",
        pre_weight_col="total_orders_0"
    ).round(2)


def build_wave_flag_diff_table(wave_flag_summary):
    return build_diff_table(
        franchise_summary=wave_flag_summary,
        index_cols=("wave", "flag"),
        group_col="analysis_group"
    )

def build_treatment_delta_transpose(wave_summary):
    df = (
        wave_summary
        .query("analysis_group == 'treatment'")
        .drop(columns=["analysis_group"])
        .set_index("wave")
    )

    delta_cols = [
        col for col in df.columns
        if col.endswith("_diff_nominal")
        or col.endswith("_diff_pp")
    ]

    out = df[delta_cols].T.round(2)

    out.index = (
        out.index
        .str.replace(
            "_orders_ratio_diff_pp",
            "_diff_pp",
            regex=False
        )
        .str.replace(
            "_ratio_diff_pp",
            "_diff_pp",
            regex=False
        )
        .str.replace(
            "_diff_nominal",
            "_diff",
            regex=False
        )
    )

    out.columns.name = None
    out.index.name = "metric"

    return out


def add_wave_did_cols(wave_diff):
    """
    Agrega comparaciones solo cuando existe la referencia.
    control se mantiene como DID; resto_de_chile se informa como
    referencia de tendencia, no como control causal.
    """
    df = wave_diff.copy()

    metric_pairs = {
        "ept": "ept_diff",
        "awt": "awt_diff",
        "tt": "tt_diff",
        "ctp": "ctp_diff",
        "slow": "slow_diff_pp",
        "non_seamless": "non_seamless_diff_pp",
        "awt_null": "awt_null_diff_pp",
        "awt_0min": "awt_0min_diff_pp",
        "awt_1_3min": "awt_1_3min_diff_pp",
        "awt_4plus": "awt_4plus_diff_pp",
        "partner_perfo": "partner_perfo_ratio_diff_pp",
        "high_preptimes": "high_preptimes_ratio_diff_pp",
        "awt10": "awt10_ratio_diff_pp",
        "prediction_model": "prediction_model_ratio_diff_pp"
    }

    for metric, base_col in metric_pairs.items():
        treatment_col = f"{base_col}_T"

        if treatment_col not in df.columns:
            continue

        if base_col in df.columns:
            suffix = "_pp" if "diff_pp" in base_col else ""
            df[f"{metric}_did{suffix}"] = (
                df[treatment_col] - df[base_col]
            )

        rest_col = f"{base_col}_R"
        if rest_col in df.columns:
            suffix = "_pp" if "diff_pp" in base_col else ""
            df[f"{metric}_vs_resto_de_chile{suffix}"] = (
                df[treatment_col] - df[rest_col]
            )

    return df.round(2)


def plot_wave_diff_bars(
    wave_diff,
    treatment_col,
    control_col=None,
    title=None,
    ylabel="post - pre"
):
    if treatment_col not in wave_diff.columns:
        print(
            f"Columna faltante para graficar: {treatment_col}"
        )
        return

    cols = ["wave", treatment_col]
    rename_cols = {treatment_col: "treatment"}

    if control_col is not None:
        if control_col in wave_diff.columns:
            cols.append(control_col)
            rename_cols[control_col] = "control"
        elif f"{control_col}_R" in wave_diff.columns:
            rest_col = f"{control_col}_R"
            cols.append(rest_col)
            rename_cols[rest_col] = "resto_de_chile"

    plot_df = (
        wave_diff[cols]
        .copy()
        .rename(columns=rename_cols)
        .set_index("wave")
    )

    ax = plot_df.plot(kind="bar", figsize=(10, 5))
    ax.axhline(0, linewidth=1)
    ax.set_title(title or treatment_col)
    ax.set_xlabel("wave")
    ax.set_ylabel(ylabel)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


 prep_summary_for_old_plots()

In [ ]:
def prep_summary_for_old_plots(summary):
    df_plot = summary.copy()

    if "treatment" not in df_plot.columns:
        df_plot["treatment"] = (
            df_plot["analysis_group"].eq("treatment").astype(int)
        )

    # Importante: si hay wave, meterla en el label para no mezclar W3 Burger King con W5 Burger King
    if "wave" in df_plot.columns:
        df_plot["franchise_name"] = (
            df_plot["wave"].astype(str)
            + " | "
            + df_plot["franchise_name"].astype(str)
        )

    return df_plot

 keys()

In [ ]:
def normalize_vendor_code(s):
    return (
        s.astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )


def add_keys(df):
    df = df.copy()
    df["vendor_code"] = normalize_vendor_code(df["vendor_code"])
    df["wave"] = df["wave"].astype(str).str.strip()
    df["analysis_group"] = df["analysis_group"].astype(str).str.strip()

    # No depende de franchise_name: long tail también puede ser treatment.
    df["cohort_key"] = (
        df["wave"] + " | " + df["vendor_code"]
    )

    # Los pseudo-vendors nacionales vienen separados por vertical.
    # Treatment/control mantienen su clave histórica por vendor.
    aggregate_scope = df["analysis_group"].isin([
        "resto_de_chile", "all_chile"
    ])
    vertical_scope = (
        df.get("vertical_type", pd.Series(
            "__NULL_VERTICAL__", index=df.index
        ))
        .astype("string")
        .fillna("__NULL_VERTICAL__")
        .str.strip()
        .str.lower()
    )
    df["_unit_vertical_scope"] = np.where(
        aggregate_scope, vertical_scope, "__VENDOR__"
    )
    df["unit_key"] = (
        df["wave"]
        + " | " + df["analysis_group"]
        + " | " + df["vendor_code"]
        + " | " + df["_unit_vertical_scope"]
    )
    return df


def consolidate_city_rows(df):
    """
    Consolida todas las ciudades de un vendor-periodo sin perder órdenes.

    La ciudad canónica es la de mayor volumen combinado Pre+Post. Los campos
    de ubicación/nombre vienen de esa fila; numeradores y denominadores se
    suman y los promedios se recalculan.
    """
    work = add_keys(df)

    required = {
        "ept_sum", "ept_orders",
        "awt_sum", "awt_orders",
        "pickup_time_sum", "pickup_time_orders",
    }
    missing = required.difference(work.columns)
    if missing:
        raise ValueError(
            "No se pueden consolidar ciudades correctamente. "
            f"Faltan numeradores/bases: {sorted(missing)}"
        )

    work["_city_key"] = (
        work["city_name"].astype("string").fillna("__NULL_CITY__")
    )
    work["total_orders"] = pd.to_numeric(
        work["total_orders"], errors="coerce"
    )

    city_volume = (
        work.groupby(["unit_key", "_city_key"], dropna=False)["total_orders"]
        .sum(min_count=1)
        .reset_index(name="orders_pre_post")
        .sort_values(
            ["unit_key", "orders_pre_post", "_city_key"],
            ascending=[True, False, True],
        )
    )
    primary_city = city_volume.drop_duplicates("unit_key", keep="first")

    anchor = (
        work.merge(
            primary_city[["unit_key", "_city_key"]],
            on=["unit_key", "_city_key"],
            how="inner",
        )
        .sort_values(
            ["unit_key", "total_orders"],
            ascending=[True, False],
        )
        .drop_duplicates("unit_key", keep="first")
        .set_index("unit_key")
    )

    sum_cols = [
        col for col in work.columns
        if (
            col == "total_orders"
            or col == "orders_with_fir"
            or (col.endswith("_orders") and col != "cl_total_orders")
            or col.endswith("_sum")
            or col in {
                "partner_perfo", "high_preptimes",
                "awt10", "prediction_model",
            }
        )
    ]
    sum_cols = list(dict.fromkeys(sum_cols))
    work[sum_cols] = work[sum_cols].apply(
        lambda series: pd.to_numeric(series, errors="coerce")
    )

    location_cols = [
        "franchise_id", "franchise_name", "store_name",
        "city_name", "vertical_type",
    ]
    rows = []

    for (unit_key, period), group in work.groupby(
        ["unit_key", "period"], dropna=False, sort=False
    ):
        record = group.iloc[0].to_dict()
        anchor_row = anchor.loc[unit_key]

        for col in location_cols:
            if col in work.columns:
                record[col] = anchor_row[col]

        for col in sum_cols:
            record[col] = group[col].sum(min_count=1)

        if "cl_total_orders" in group.columns:
            record["cl_total_orders"] = pd.to_numeric(
                group["cl_total_orders"], errors="coerce"
            ).max()

        total = record.get("total_orders")
        record["fir"] = safe_ratio(record.get("orders_with_fir"), total)
        record["slow_orders_ratio"] = safe_ratio(
            record.get("slow_orders"), total
        )
        record["non_seamless_orders_ratio"] = safe_ratio(
            record.get("non_seamless_orders"), total
        )

        for count_col in [
            col for col in sum_cols
            if col.startswith("awt_")
            and col.endswith("_orders")
            and col != "awt_orders"
        ]:
            bucket = count_col[len("awt_"):-len("_orders")]
            if bucket == "null":
                share_col = "awt_null"
            elif bucket.endswith("plus"):
                share_col = f"awt_{bucket}"
            else:
                share_col = f"awt_{bucket}min"
            record[share_col] = safe_ratio(
                record.get(count_col), total
            )

        record["ept"] = safe_ratio(
            record.get("ept_sum"), record.get("ept_orders")
        )
        record["awt"] = safe_ratio(
            record.get("awt_sum"), record.get("awt_orders")
        )
        record["ctp"] = safe_ratio(
            record.get("pickup_time_sum"),
            record.get("pickup_time_orders"),
        )
        record["tt"] = record["ept"] + record["awt"]

        for impact in [
            "partner_perfo", "high_preptimes",
            "awt10", "prediction_model",
        ]:
            record[f"{impact}_ratio"] = safe_ratio(
                record.get(impact), total
            )

        rows.append(record)

    consolidated = pd.DataFrame(rows).drop(columns=["_city_key"], errors="ignore")

    def joined_cities(series):
        values = sorted({
            str(value) for value in series.dropna()
            if str(value) != "__NULL_CITY__"
        })
        return " | ".join(values)

    audit = (
        work.groupby("unit_key", dropna=False)
        .agg(
            wave=("wave", "first"),
            analysis_group=("analysis_group", "first"),
            vendor_code=("vendor_code", "first"),
            cities_before=("city_name", joined_cities),
            city_count=("_city_key", "nunique"),
            rows_before=("period", "size"),
            rows_pre=("period", lambda s: (s == "pre").sum()),
            rows_post=("period", lambda s: (s == "post").sum()),
            orders_total=("total_orders", "sum"),
        )
        .reset_index()
        .merge(
            primary_city[["unit_key", "_city_key"]].rename(
                columns={"_city_key": "selected_city"}
            ),
            on="unit_key",
            how="left",
            validate="1:1",
        )
    )
    audit = audit[
        (audit["city_count"] > 1)
        | (audit["rows_pre"] > 1)
        | (audit["rows_post"] > 1)
    ].reset_index(drop=True)

    return consolidated, audit


 format_analysis_group_table()

In [ ]:
def format_analysis_group_table(
    df_eval,
    metric_config,
    analysis_groups=("treatment", "control", "resto_de_chile")
):
    df = df_eval.copy()

    df = df[
        df["analysis_group"].isin(analysis_groups)
    ].copy()

    # --------------------------------------------------
    # 1. Summary agregado SOLO por control/treatment
    # --------------------------------------------------
    summary = build_franchise_summary(
        df=df,
        metric_config=metric_config,
        group_cols=("analysis_group",),
        post_weight_col="total_orders",
        pre_weight_col="total_orders_0"
    )

    # --------------------------------------------------
    # 2. Stats extra por control/treatment
    # --------------------------------------------------
    stats = (
        df
        .groupby("analysis_group", dropna=False)
        .agg(
            franchises=("franchise_name", "nunique"),
            vendors=("vendor_code", "nunique"),
            post_orders=("total_orders", "sum"),
            pre_orders=("total_orders_0", "sum")
        )
        .reset_index()
    )

    if "resto_de_chile" in set(analysis_groups):
        # control está contenido en resto_de_chile: no son grupos disjuntos.
        stats["share_orders"] = np.nan
    else:
        total_post_orders = stats["post_orders"].sum()
        stats["share_orders"] = np.where(
            total_post_orders > 0,
            stats["post_orders"] / total_post_orders,
            np.nan,
        )

    summary = summary.merge(
        stats,
        on="analysis_group",
        how="left"
    )

    # --------------------------------------------------
    # 3. Columnas finales
    # --------------------------------------------------
    metric_map = {
        "franchises": "franchises",
        "vendors": "vendors",
        "share_orders": "share orders",

        "ept_diff_nominal": "Δ ept (min)",
        "awt_diff_nominal": "Δ awt (min)",
        "ctp_diff_nominal": "Δ pickup time (min)",
        "slow_orders_ratio_diff_pp": "Δ slow (pp)",
        "non_seamless_orders_ratio_diff_pp": "Δ non_seamless (pp)",
        "awt_null_diff_pp": "Δ awt_null (pp)",
        "awt_0min_diff_pp": "Δ awt_0min (pp)",
        "awt_1_3min_diff_pp": "Δ awt_1_3min (pp)",
        "awt_4plus_diff_pp": "Δ awt_4plus (pp)",
        "high_preptimes_ratio_diff_pp": "Δ high preptimes (pp)",
        "awt10_ratio_diff_pp": "Δ AWT10 causal (pp)",

        "tt_diff_nominal": "Δ ept + awt (min)",
        "partner_perfo_ratio_diff_pp": "Δ partner performance (pp)",
        "prediction_model_ratio_diff_pp": "Δ prediction model (pp)",

        "post_orders": "Post #Orders",
        "pre_orders": "Pre #Orders"
    }

    existing_cols = [
        col for col in metric_map
        if col in summary.columns
    ]

    out = summary[
        ["analysis_group"] + existing_cols
    ].copy()

    # --------------------------------------------------
    # 4. Formato
    # --------------------------------------------------
    int_cols = [
        "franchises",
        "vendors",
        "post_orders",
        "pre_orders"
    ]

    one_decimal_cols = [
        "ept_diff_nominal",
        "awt_diff_nominal",
        "ctp_diff_nominal",
        "slow_orders_ratio_diff_pp",
        "non_seamless_orders_ratio_diff_pp",
        "awt_null_diff_pp",
        "awt_0min_diff_pp",
        "awt_1_3min_diff_pp",
        "awt_4plus_diff_pp",
        "high_preptimes_ratio_diff_pp",
        "awt10_ratio_diff_pp"
    ]

    two_decimal_cols = [
        "tt_diff_nominal",
        "partner_perfo_ratio_diff_pp",
        "prediction_model_ratio_diff_pp"
    ]

    for col in int_cols:
        if col in out.columns:
            out[col] = (
                pd.to_numeric(out[col], errors="coerce")
                .round(0)
                .astype("Int64")
                .astype(str)
            )

    for col in one_decimal_cols:
        if col in out.columns:
            out[col] = (
                pd.to_numeric(out[col], errors="coerce")
                .round(1)
                .map(lambda x: "" if pd.isna(x) else f"{x:.1f}")
            )

    for col in two_decimal_cols:
        if col in out.columns:
            out[col] = (
                pd.to_numeric(out[col], errors="coerce")
                .round(2)
                .map(lambda x: "" if pd.isna(x) else f"{x:.2f}")
            )

    if "share_orders" in out.columns:
        out["share_orders"] = (
            pd.to_numeric(out["share_orders"], errors="coerce")
            .map(lambda x: "" if pd.isna(x) else f"{x * 100:.1f}%")
        )

    # --------------------------------------------------
    # 5. Transponer: filas = métricas, columnas = control/treatment
    # --------------------------------------------------
    out = out.rename(columns=metric_map)

    final = (
        out
        .set_index("analysis_group")
        .T
    )

    final = final[
        [col for col in ["treatment", "control", "resto_de_chile"] if col in final.columns]
    ]

    final.columns.name = None
    final.index.name = "metric"

    return final


 table_simplify_analysis_group()

In [ ]:
import numpy as np
import pandas as pd

def table_simplify_analysis_group(df, source_df=None):
    id_cols = [
        col for col in ["wave", "franchise_id", "franchise_name", "ciudad"]
        if col in df.columns
    ]

    metric_map = {
        "delta_ept": {
            "control": "ept_diff",
            "treatment": "ept_diff_T"
        },
        "delta_awt": {
            "control": "awt_diff",
            "treatment": "awt_diff_T"
        },
        "delta_tt": {
            "control": "tt_diff",
            "treatment": "tt_diff_T"
        },
        "delta_ctp": {
            "control": "ctp_diff",
            "treatment": "ctp_diff_T"
        },
        "delta_slow_pp": {
            "control": "slow_diff_pp",
            "treatment": "slow_diff_pp_T"
        },
        "delta_non_seamless_pp": {
            "control": "non_seamless_diff_pp",
            "treatment": "non_seamless_diff_pp_T"
        },
        "delta_awt_null_pp": {
            "control": "awt_null_diff_pp",
            "treatment": "awt_null_diff_pp_T"
        },
        "delta_awt_0min_pp": {
            "control": "awt_0min_diff_pp",
            "treatment": "awt_0min_diff_pp_T"
        },
        "delta_awt_1_3min_pp": {
            "control": "awt_1_3min_diff_pp",
            "treatment": "awt_1_3min_diff_pp_T"
        },
        "delta_awt_4plus_pp": {
            "control": "awt_4plus_diff_pp",
            "treatment": "awt_4plus_diff_pp_T"
        },
        "delta_partner_perfo_pp": {
            "control": "partner_perfo_ratio_diff_pp",
            "treatment": "partner_perfo_ratio_diff_pp_T"
        },
        "delta_high_preptimes_pp": {
            "control": "high_preptimes_ratio_diff_pp",
            "treatment": "high_preptimes_ratio_diff_pp_T"
        },
        "delta_awt10_pp": {
            "control": "awt10_ratio_diff_pp",
            "treatment": "awt10_ratio_diff_pp_T"
        },
        "delta_prediction_model_pp": {
            "control": "prediction_model_ratio_diff_pp",
            "treatment": "prediction_model_ratio_diff_pp_T"
        }
    }

    frames = []

    for analysis_group in ["control", "treatment"]:
        tmp = df[id_cols].copy()
        tmp["analysis_group"] = analysis_group

        for out_col, cols in metric_map.items():
            source_col = cols[analysis_group]

            if source_col in df.columns:
                tmp[out_col] = df[source_col].round(2)

        frames.append(tmp)

    simplified = pd.concat(frames, ignore_index=True)

    # --------------------------------------------------
    # Stats extra desde source_df
    # share_orders = post_orders del grupo / post_orders total de la wave
    # --------------------------------------------------
    if source_df is not None:
        stats_df = source_df.copy()

        stats_df = stats_df[
            stats_df["analysis_group"].isin(["control", "treatment"])
        ].copy()

        group_cols = id_cols.copy()

        missing_group_cols = [
            col for col in group_cols
            if col not in stats_df.columns
        ]

        if missing_group_cols:
            raise ValueError(
                f"source_df no tiene estas columnas necesarias: {missing_group_cols}"
            )

        agg_dict = {}

        if "franchise_name" in stats_df.columns:
            agg_dict["franchises"] = ("franchise_name", "nunique")

        if "vendor_code" in stats_df.columns:
            agg_dict["vendors"] = ("vendor_code", "nunique")

        if "total_orders" in stats_df.columns:
            agg_dict["post_orders"] = ("total_orders", "sum")

        if "total_orders_0" in stats_df.columns:
            agg_dict["pre_orders"] = ("total_orders_0", "sum")

        stats = (
            stats_df
            .groupby(group_cols + ["analysis_group"], dropna=False)
            .agg(**agg_dict)
            .reset_index()
        )

        if "post_orders" in stats.columns:
            total_post_orders = (
                stats
                .groupby(group_cols, dropna=False)["post_orders"]
                .transform("sum")
            )

            stats["share_orders"] = np.where(
                total_post_orders > 0,
                stats["post_orders"] / total_post_orders,
                np.nan
            )

        simplified = simplified.merge(
            stats,
            on=group_cols + ["analysis_group"],
            how="left"
        )

    return simplified


def format_wave_analysis_group_table(delta_table_group):
    df = delta_table_group.copy()

    id_cols = [
        col for col in ["wave", "franchise_id", "franchise_name", "ciudad"]
        if col in df.columns
    ]

    # ordenar waves tipo W1, W2, W3...
    if "wave" in df.columns:
        df["_wave_order"] = (
            df["wave"]
            .astype(str)
            .str.extract(r"(\d+)")
            .astype(float)
        )

        df = (
            df
            .sort_values(["_wave_order", "analysis_group"])
            .drop(columns="_wave_order")
            .copy()
        )

    col_map = {
        "franchises": "franchises",
        "vendors": "vendors",
        "share_orders": "share orders",

        "delta_ept": "Δ ept (min)",
        "delta_awt": "Δ awt (min)",
        "delta_ctp": "Δ pickup time (min)",
        "delta_slow_pp": "Δ slow (pp)",
        "delta_non_seamless_pp": "Δ non_seamless (pp)",
        "delta_awt_null_pp": "Δ awt_null (pp)",
        "delta_awt_0min_pp": "Δ awt_0min (pp)",
        "delta_awt_1_3min_pp": "Δ awt_1_3min (pp)",
        "delta_awt_4plus_pp": "Δ awt_4plus (pp)",
        "delta_high_preptimes_pp": "Δ high preptimes (pp)",
        "delta_awt10_pp": "Δ AWT10 causal (pp)",

        "delta_tt": "Δ ept + awt (min)",
        "delta_partner_perfo_pp": "Δ partner performance (pp)",
        "delta_prediction_model_pp": "Δ prediction model (pp)",

        "post_orders": "Post #Orders",
        "pre_orders": "Pre #Orders"
    }

    existing_cols = [
        col for col in col_map
        if col in df.columns
    ]

    out = df[id_cols + ["analysis_group"] + existing_cols].copy()

    def fmt_int(x):
        if pd.isna(x):
            return ""
        return f"{x:.0f}"

    def fmt_1(x):
        if pd.isna(x):
            return ""
        return f"{x:.1f}"

    def fmt_2(x):
        if pd.isna(x):
            return ""
        return f"{x:.2f}"

    def fmt_pct(x):
        if pd.isna(x):
            return ""
        return f"{x * 100:.1f}%"

    int_cols = [
        "franchises",
        "vendors",
        "post_orders",
        "pre_orders"
    ]

    one_decimal_cols = [
        "delta_ept",
        "delta_awt",
        "delta_ctp",
        "delta_slow_pp",
        "delta_non_seamless_pp",
        "delta_awt_null_pp",
        "delta_awt_0min_pp",
        "delta_awt_1_3min_pp",
        "delta_awt_4plus_pp",
        "delta_high_preptimes_pp",
        "delta_awt10_pp"
    ]

    two_decimal_cols = [
        "delta_tt",
        "delta_partner_perfo_pp",
        "delta_prediction_model_pp"
    ]

    for col in int_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce").map(fmt_int)

    for col in one_decimal_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce").map(fmt_1)

    for col in two_decimal_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce").map(fmt_2)

    if "share_orders" in out.columns:
        out["share_orders"] = pd.to_numeric(out["share_orders"], errors="coerce").map(fmt_pct)

    out = out.rename(columns=col_map)

    metric_order = [
        col_map[col]
        for col in existing_cols
    ]

    out = out.melt(
        id_vars=id_cols + ["analysis_group"],
        value_vars=metric_order,
        var_name="metric",
        value_name="value"
    )

    out["metric"] = pd.Categorical(
        out["metric"],
        categories=metric_order,
        ordered=True
    )

    final = (
        out
        .pivot_table(
            index=id_cols + ["metric"],
            columns="analysis_group",
            values="value",
            aggfunc="first",
            sort=False
        )
    )

    ordered_cols = [
        col for col in ["control", "treatment"]
        if col in final.columns
    ]

    final = final[ordered_cols]

    final.columns.name = None
    final.index.names = id_cols + ["metric"]

    return final


In [ ]:
df_pre_post_k = add_keys(df_pre_post)

df_pre_post_consolidated, city_consolidation_audit = (
    consolidate_city_rows(df_pre_post_k)
)

pre = df_pre_post_consolidated[
    df_pre_post_consolidated["period"].eq("pre")
].copy()
post = df_pre_post_consolidated[
    df_pre_post_consolidated["period"].eq("post")
].copy()


### df_merged

## Auditoría de cohorte y consolidación por ciudad

La consolidación elige una ciudad canónica usando el mayor volumen combinado
Pre+Post. Luego suma órdenes, impactos, numeradores y bases de todas las
ciudades y recalcula los promedios. No se elimina información. Por ejemplo, si
`439914` tiene mayor volumen en Peñaflor, las órdenes de Santiago se incorporan
a Peñaflor y la fila secundaria desaparece solo después de la suma.

Las Holy Tables usan exclusivamente vendors observados en ambos periodos. Las
tablas de auditoría conservan los configurados y los casos perdidos.


In [ ]:
reduction_latest = reduction.copy()

if "flag" not in reduction_latest.columns:
    raise ValueError("Falta la columna 'flag' en la tabla reduction.")

reduction_latest["flag"] = (
    reduction_latest["flag"]
    .fillna("SIN_FLAG")
    .astype(str)
    .str.strip()
    .replace({"": "SIN_FLAG", "nan": "SIN_FLAG"})
)
reduction_latest["vendor_code"] = normalize_vendor_code(
    reduction_latest["vendor_code"]
)
reduction_latest["wave"] = (
    reduction_latest["wave"].astype(str).str.strip()
)
reduction_latest["start_date"] = pd.to_datetime(
    reduction_latest["start_date"], errors="coerce"
)

reduction_latest = (
    reduction_latest
    .dropna(subset=["start_date"])
    .sort_values(
        ["vendor_code", "start_date"],
        ascending=[True, False],
    )
    .drop_duplicates("vendor_code", keep="first")
    .reset_index(drop=True)
)
reduction_latest["treatment_key"] = (
    reduction_latest["wave"].astype(str)
    + "|" + reduction_latest["vendor_code"].astype(str)
)


acá hacer que el left join tome la latest wave

In [ ]:
print("Casos consolidados por ciudad:", len(city_consolidation_audit))
display(city_consolidation_audit)


In [ ]:
# Ejemplo esperado: si 439914 tiene más órdenes en Peñaflor,
# selected_city será Peñaflor y las órdenes de Santiago se sumarán allí.
display(
    city_consolidation_audit[
        city_consolidation_audit["vendor_code"].eq("439914")
    ]
)


In [ ]:
# La consolidación ya dejó una sola fila por unit_key + period.
assert not pre.duplicated("unit_key").any()
assert not post.duplicated("unit_key").any()


In [ ]:
df_merged_all = post.merge(
    pre,
    on="unit_key",
    how="outer",
    suffixes=("", "_0"),
    validate="1:1",
    indicator=True,
)


In [ ]:
df_merged_all = df_merged_all.copy()

# Para casos solo-Pre, recuperar las dimensiones desde las columnas _0.
identity_cols = [
    "wave", "analysis_group", "franchise_id", "franchise_name",
    "vendor_code", "store_name", "city_name", "vertical_type",
    "inicio_pre", "fin_pre", "inicio_post", "fin_post", "cohort_key",
]

for col in identity_cols:
    pre_col = f"{col}_0"
    if pre_col not in df_merged_all.columns:
        continue
    if col not in df_merged_all.columns:
        df_merged_all[col] = df_merged_all[pre_col]
    else:
        df_merged_all[col] = df_merged_all[col].where(
            df_merged_all[col].notna(), df_merged_all[pre_col]
        )

df_merged_all["vendor_code"] = normalize_vendor_code(
    df_merged_all["vendor_code"]
)
df_merged_all["active_post"] = df_merged_all["_merge"].isin(
    ["left_only", "both"]
)
df_merged_all["active_pre"] = df_merged_all["_merge"].isin(
    ["right_only", "both"]
)
df_merged_all["active_both"] = df_merged_all["_merge"].eq("both")
df_merged_all["comparison_status"] = np.select(
    [
        df_merged_all["active_both"],
        df_merged_all["active_pre"] & ~df_merged_all["active_post"],
        ~df_merged_all["active_pre"] & df_merged_all["active_post"],
    ],
    ["COMPARABLE", "MISSING_POST", "MISSING_PRE"],
    default="MISSING_BOTH",
)

# Solo estos casos alimentan summaries y Holy Tables.
df_merged = df_merged_all[
    df_merged_all["active_both"]
].drop(columns=["_merge"]).copy()

df_merged["treatment_key"] = (
    df_merged["wave"].astype(str)
    + "|" + df_merged["vendor_code"].astype(str)
)


In [ ]:
df_merged

In [ ]:
vendors_in_selected_waves = set(reduction_latest["vendor_code"])
treatment_keys = set(reduction_latest["treatment_key"])

df_merged["in_selected_wave"] = df_merged["vendor_code"].isin(
    vendors_in_selected_waves
)
df_merged["treatment"] = df_merged["treatment_key"].isin(
    treatment_keys
)

derived_group = np.select(
    [df_merged["treatment"], ~df_merged["in_selected_wave"]],
    ["treatment", "control"],
    default="exclude",
)

# La query nueva manda. derived_group solo es fallback para CSVs antiguos.
sql_group = df_merged["analysis_group"].astype(str).str.strip()
df_merged["analysis_group"] = sql_group.where(
    sql_group.isin(["treatment", "control", "resto_de_chile", "all_chile"]),
    derived_group,
)

flag_lookup = (
    reduction_latest[["wave", "vendor_code", "flag"]]
    .drop_duplicates(["wave", "vendor_code"], keep="first")
)
df_merged = (
    df_merged.drop(columns=["flag"], errors="ignore")
    .merge(
        flag_lookup,
        on=["wave", "vendor_code"],
        how="left",
        validate="m:1",
    )
)
df_merged.loc[
    df_merged["analysis_group"].ne("treatment"), "flag"
] = pd.NA
df_merged.loc[
    df_merged["analysis_group"].eq("treatment")
    & df_merged["flag"].isna(),
    "flag",
] = "SIN_FLAG"


In [ ]:
# ------------------------------------------------------------
# AUDITORÍA 1: cohorte configurada de treatment
# ------------------------------------------------------------
treatment_activity = (
    df_pre_post_consolidated[
        df_pre_post_consolidated["analysis_group"].eq("treatment")
    ]
    .pivot_table(
        index=["wave", "vendor_code"],
        columns="period",
        values="total_orders",
        aggfunc="sum",
    )
    .rename(columns={"pre": "orders_pre", "post": "orders_post"})
    .reset_index()
)

configured_cols = [
    col for col in [
        "wave", "vendor_code", "franchise_name", "flag", "start_date"
    ]
    if col in reduction_latest.columns
]

cohort_audit_detail = (
    reduction_latest[configured_cols]
    .drop_duplicates(["wave", "vendor_code"])
    .merge(
        treatment_activity,
        on=["wave", "vendor_code"],
        how="left",
        validate="1:1",
    )
)
for col in ["orders_pre", "orders_post"]:
    if col not in cohort_audit_detail.columns:
        cohort_audit_detail[col] = np.nan

cohort_audit_detail["active_pre"] = (
    cohort_audit_detail["orders_pre"].fillna(0) > 0
)
cohort_audit_detail["active_post"] = (
    cohort_audit_detail["orders_post"].fillna(0) > 0
)
cohort_audit_detail["active_both"] = (
    cohort_audit_detail["active_pre"]
    & cohort_audit_detail["active_post"]
)
cohort_audit_detail["comparison_status"] = np.select(
    [
        cohort_audit_detail["active_both"],
        cohort_audit_detail["active_pre"]
        & ~cohort_audit_detail["active_post"],
        ~cohort_audit_detail["active_pre"]
        & cohort_audit_detail["active_post"],
    ],
    ["COMPARABLE", "MISSING_POST", "MISSING_PRE"],
    default="MISSING_BOTH",
)

cohort_audit_summary = (
    cohort_audit_detail.groupby("wave", dropna=False)
    .agg(
        configured_vendors=("vendor_code", "nunique"),
        active_pre=("active_pre", "sum"),
        active_post=("active_post", "sum"),
        active_both=("active_both", "sum"),
        missing_pre=("active_pre", lambda s: (~s).sum()),
        missing_post=("active_post", lambda s: (~s).sum()),
    )
    .reset_index()
)

cohort_lost_cases = cohort_audit_detail[
    ~cohort_audit_detail["active_both"]
].copy()


# ------------------------------------------------------------
# AUDITORÍA 2: unión observada para treatment/control/resto/all_chile
# ------------------------------------------------------------
cohort_audit_observed = (
    df_merged_all.groupby(["wave", "analysis_group"], dropna=False)
    .agg(
        vendors_union=("vendor_code", "nunique"),
        active_pre=("active_pre", "sum"),
        active_post=("active_post", "sum"),
        active_both=("active_both", "sum"),
        only_pre=("comparison_status", lambda s: (s == "MISSING_POST").sum()),
        only_post=("comparison_status", lambda s: (s == "MISSING_PRE").sum()),
    )
    .reset_index()
)

df_eval = df_merged[
    df_merged["analysis_group"].isin(
        ["treatment", "control", "resto_de_chile"]
    )
].copy()

print("AUDITORÍA — COHORTE CONFIGURADA")
display(cohort_audit_summary)
print("AUDITORÍA — CASOS PERDIDOS (no entran a Holy Tables)")
display(cohort_lost_cases)
print("AUDITORÍA — ACTIVIDAD OBSERVADA POR GRUPO")
display(cohort_audit_observed)


In [ ]:
df_eval

city vs región flag

In [ ]:
df_eval['ciudad'] = (df_eval['city_name'] == 'Santiago').astype(int)

treatment vs no treatment flag

In [ ]:
# df_merged['treatment'] = df_merged['ept_new'].notna().astype(int)
# df_merged['treatment'] = df_merged['is_in_reduction'].notna().astype(int)

### ratios and diffs

In [ ]:
# ============================================================
# BUCKETS AWT ÚNICOS PARA TODO EL NOTEBOOK
# ============================================================

# Los buckets detallados siguen viniendo desde la query y permiten
# reconstruir los cuatro buckets finales sin volver a consultar BigQuery.
AWT_RAW_BUCKETS = [
    f"awt_{minute}min"
    for minute in range(1, 21)
] + ["awt_21plus"]

AWT_RAW_SHARE_TO_COUNT = {
    **{
        f"awt_{minute}min": f"awt_{minute}_orders"
        for minute in range(1, 21)
    },
    "awt_21plus": "awt_21plus_orders",
}

# Estos cuatro buckets alimentan métricas, Holy Tables y la
# primera vista resumida de gráficos.
AWT_DISTRIBUTION_BUCKETS = [
    "awt_null",
    "awt_0min",
    "awt_1_3min",
    "awt_4plus",
]

AWT_SHARE_TO_COUNT = {
    "awt_null": "awt_null_orders",
    "awt_0min": "awt_0_orders",
    "awt_1_3min": "awt_1_3_orders",
    "awt_4plus": "awt_4plus_orders",
}

AWT_BUCKET_LABELS = {
    "awt_null": "AWT null",
    "awt_0min": "AWT 0 min",
    "awt_1_3min": "AWT 1–3 min",
    "awt_4plus": "AWT 4+ min",
}

AWT_CURVE_BUCKETS = AWT_DISTRIBUTION_BUCKETS.copy()

# Segunda vista para los gráficos detallados del final.
# 11+ agrupa (10, 11], ..., (19, 20] y [20, +inf).
AWT_DETAILED_BUCKETS = (
    ["awt_null", "awt_0min"]
    + [f"awt_{minute}min" for minute in range(1, 11)]
    + ["awt_11plus"]
)

AWT_DETAILED_COUNT_COLUMNS = {
    "awt_null": ["awt_null_orders"],
    "awt_0min": ["awt_0_orders"],
    **{
        f"awt_{minute}min": [f"awt_{minute}_orders"]
        for minute in range(1, 11)
    },
    "awt_11plus": (
        [f"awt_{minute}_orders" for minute in range(11, 21)]
        + ["awt_21plus_orders"]
    ),
}

AWT_DETAILED_BUCKET_LABELS = {
    "awt_null": "Null",
    "awt_0min": "0",
    **{
        f"awt_{minute}min": str(minute)
        for minute in range(1, 11)
    },
    "awt_11plus": "11+",
}

awt_raw_count_cols = [
    AWT_RAW_SHARE_TO_COUNT[bucket]
    for bucket in AWT_RAW_BUCKETS
]

awt_raw_share_cols = AWT_RAW_BUCKETS.copy()

numeric_cols = [
    # tiempos
    "ept", "ept_0",
    "awt", "awt_0",
    "tt", "tt_0",
    "ctp", "ctp_0",

    # órdenes y bases
    "total_orders", "total_orders_0",
    "ept_sum", "ept_sum_0",
    "ept_orders", "ept_orders_0",
    "awt_sum", "awt_sum_0",
    "awt_orders", "awt_orders_0",
    "pickup_time_sum", "pickup_time_sum_0",
    "pickup_time_orders", "pickup_time_orders_0",
    "cl_total_orders", "cl_total_orders_0",
    "slow_orders", "slow_orders_0",
    "non_seamless_orders", "non_seamless_orders_0",
    "awt_null_orders", "awt_null_orders_0",
    "awt_0_orders", "awt_0_orders_0",

    # reasons
    "partner_perfo", "partner_perfo_0",
    "high_preptimes", "high_preptimes_0",
    "awt10", "awt10_0",
    "prediction_model", "prediction_model_0",
]

numeric_cols += [
    col
    for base_col in awt_raw_count_cols + awt_raw_share_cols
    for col in (base_col, f"{base_col}_0")
]

numeric_cols = [
    col for col in numeric_cols
    if col in df_merged.columns
]

df_merged[numeric_cols] = df_merged[numeric_cols].apply(
    lambda series: pd.to_numeric(series, errors="coerce")
)


# Construir cantidades y shares de los cuatro buckets finales.
for suffix in ["", "_0"]:

    one_to_three_cols = [
        f"awt_{minute}_orders{suffix}"
        for minute in range(1, 4)
    ]

    four_plus_cols = [
        f"awt_{minute}_orders{suffix}"
        for minute in range(4, 21)
    ] + [f"awt_21plus_orders{suffix}"]

    missing_1_3 = [
        col for col in one_to_three_cols
        if col not in df_merged.columns
    ]

    missing_4plus = [
        col for col in four_plus_cols
        if col not in df_merged.columns
    ]

    if missing_1_3 or missing_4plus:
        raise KeyError(
            "Faltan counts detallados de AWT para construir los buckets. "
            f"1–3: {missing_1_3}; 4+: {missing_4plus}"
        )

    df_merged[f"awt_1_3_orders{suffix}"] = (
        df_merged[one_to_three_cols]
        .sum(axis=1, min_count=1)
    )

    df_merged[f"awt_4plus_orders{suffix}"] = (
        df_merged[four_plus_cols]
        .sum(axis=1, min_count=1)
    )

    denominator = df_merged[f"total_orders{suffix}"].replace(0, np.nan)

    df_merged[f"awt_null{suffix}"] = (
        df_merged[f"awt_null_orders{suffix}"] / denominator
    )

    df_merged[f"awt_0min{suffix}"] = (
        df_merged[f"awt_0_orders{suffix}"] / denominator
    )

    df_merged[f"awt_1_3min{suffix}"] = (
        df_merged[f"awt_1_3_orders{suffix}"] / denominator
    )

    df_merged[f"awt_4plus{suffix}"] = (
        df_merged[f"awt_4plus_orders{suffix}"] / denominator
    )


In [ ]:
# tiempos
df_merged["ept_diff"] = df_merged["ept"] - df_merged["ept_0"]
df_merged["awt_diff"] = df_merged["awt"] - df_merged["awt_0"]
df_merged["tt_diff"] = df_merged["tt"] - df_merged["tt_0"]
df_merged["ctp_diff"] = df_merged["ctp"] - df_merged["ctp_0"]

# slow ratio en escala 0-100
df_merged["slow_orders_ratio_0"] = (
    df_merged["slow_orders_0"] / df_merged["total_orders_0"] * 100
)

df_merged["slow_orders_ratio"] = (
    df_merged["slow_orders"] / df_merged["total_orders"] * 100
)

df_merged["slow_orders_ratio_diff_pp"] = (
    df_merged["slow_orders_ratio"] - df_merged["slow_orders_ratio_0"]
)

# non-seamless
df_merged["non_seamless_orders_diff"] = (
    df_merged["non_seamless_orders"] - df_merged["non_seamless_orders_0"]
)

df_merged["non_seamless_orders_ratio_0"] = (
    df_merged["non_seamless_orders_0"] / df_merged["total_orders_0"] * 100
)

df_merged["non_seamless_orders_ratio"] = (
    df_merged["non_seamless_orders"] / df_merged["total_orders"] * 100
)

df_merged["non_seamless_orders_ratio_diff_pp"] = (
    df_merged["non_seamless_orders_ratio"] - df_merged["non_seamless_orders_ratio_0"]
)

# Partner Performance remanente para reconciliar el total.
df_merged["other_partner_perfo"] = (
    df_merged["partner_perfo"]
    - df_merged["high_preptimes"]
    - df_merged["awt10"]
)
df_merged["other_partner_perfo_0"] = (
    df_merged["partner_perfo_0"]
    - df_merged["high_preptimes_0"]
    - df_merged["awt10_0"]
)

# Métricas non-seamless reasons.
new_reason_metrics = [
    "partner_perfo",
    "high_preptimes",
    "awt10",
    "other_partner_perfo",
    "prediction_model",
]

for metric in new_reason_metrics:
    # diferencia nominal post - pre
    df_merged[f"{metric}_diff"] = (
        df_merged[metric] - df_merged[f"{metric}_0"]
    )

    # ratio pre en escala 0-100
    df_merged[f"{metric}_ratio_0"] = (
        df_merged[f"{metric}_0"] / df_merged["total_orders_0"] * 100
    )

    # ratio post en escala 0-100
    df_merged[f"{metric}_ratio"] = (
        df_merged[metric] / df_merged["total_orders"] * 100
    )

    # diferencia en puntos porcentuales
    df_merged[f"{metric}_ratio_diff_pp"] = (
        df_merged[f"{metric}_ratio"] - df_merged[f"{metric}_ratio_0"]
    )

# limpieza por divisiones con 0
ratio_cols = [
    "slow_orders_ratio_0",
    "slow_orders_ratio",
    "slow_orders_ratio_diff_pp",

    "non_seamless_orders_ratio_0",
    "non_seamless_orders_ratio",
    "non_seamless_orders_ratio_diff_pp",

    "partner_perfo_ratio_0",
    "partner_perfo_ratio",
    "partner_perfo_ratio_diff_pp",

    "high_preptimes_ratio_0",
    "high_preptimes_ratio",
    "high_preptimes_ratio_diff_pp",

    "awt10_ratio_0",
    "awt10_ratio",
    "awt10_ratio_diff_pp",

    "other_partner_perfo_ratio_0",
    "other_partner_perfo_ratio",
    "other_partner_perfo_ratio_diff_pp",

    "prediction_model_ratio_0",
    "prediction_model_ratio",
    "prediction_model_ratio_diff_pp"
]

df_merged[ratio_cols] = df_merged[ratio_cols].replace([np.inf, -np.inf], np.nan)

In [ ]:
df_merged

In [ ]:
df_merged.groupby("franchise_name").agg(
    stores=("vendor_code", "nunique"),
    treated_stores=("treatment", "sum"),
    pre_orders=("total_orders_0", "sum"),
    post_orders=("total_orders", "sum")
).query("treated_stores == 0")

# df_merged[df_merged["franchise_name"].eq("Churromania")][[
#     "vendor_code",
#     "store_name",
#     "total_orders_0",
#     "total_orders",
#     "ept_new",
#     "treatment"
# ]]

In [ ]:
# treatment/control + referencia nacional
df_eval = df_merged[
    df_merged["analysis_group"].isin([
        "treatment",
        "control",
        "resto_de_chile",
    ])
].copy()

# resto_de_chile ya viene agregado a nivel nacional y no debe caer
# artificialmente dentro de Regiones.
df_eval["ciudad"] = np.select(
    [
        df_eval["analysis_group"].eq("resto_de_chile"),
        df_eval["city_name"].eq("Santiago"),
    ],
    [
        np.nan,
        1,
    ],
    default=0,
)

numeric_cols = [
    "ept", "ept_0",
    "awt", "awt_0",
    "tt", "tt_0",
    "ctp", "ctp_0",
    "total_orders", "total_orders_0",
    "ept_sum", "ept_sum_0",
    "ept_orders", "ept_orders_0",
    "awt_sum", "awt_sum_0",
    "awt_orders", "awt_orders_0",
    "pickup_time_sum", "pickup_time_sum_0",
    "pickup_time_orders", "pickup_time_orders_0",
    "cl_total_orders", "cl_total_orders_0",
    "slow_orders", "slow_orders_0",
    "non_seamless_orders", "non_seamless_orders_0",
    "awt_null_orders", "awt_null_orders_0",
    "awt_0_orders", "awt_0_orders_0",
    "awt_1_3_orders", "awt_1_3_orders_0",
    "awt_4plus_orders", "awt_4plus_orders_0",
    "awt_null", "awt_null_0",
    "awt_0min", "awt_0min_0",
    "awt_1_3min", "awt_1_3min_0",
    "awt_4plus", "awt_4plus_0",
    "partner_perfo", "partner_perfo_0",
    "high_preptimes", "high_preptimes_0",
    "awt10", "awt10_0",
    "prediction_model", "prediction_model_0",
]

numeric_cols += [
    col
    for base_col in awt_raw_count_cols + awt_raw_share_cols
    for col in (base_col, f"{base_col}_0")
]

numeric_cols = [
    column
    for column in numeric_cols
    if column in df_eval.columns
]

df_eval[numeric_cols] = df_eval[numeric_cols].apply(
    lambda series: pd.to_numeric(series, errors="coerce")
)

df_eval


###  impact general summary

In [ ]:
metric_config = {
    "ept": {
        "type": "aggregate_mean",
        "num": "ept_sum",
        "den": "ept_orders",
    },
    "awt": {
        "type": "aggregate_mean",
        "num": "awt_sum",
        "den": "awt_orders",
    },
    "ctp": {
        "type": "aggregate_mean",
        "num": "pickup_time_sum",
        "den": "pickup_time_orders",
    },
    "tt": {
        "type": "derived_sum",
        "components": ["ept", "awt"],
    },
    "slow_orders_ratio": {
        "type": "ratio",
        "num": "slow_orders",
        "den": "total_orders",
    },
    "non_seamless_orders_ratio": {
        "type": "ratio",
        "num": "non_seamless_orders",
        "den": "total_orders",
    },

    # Distribución AWT: cuatro buckets no superpuestos.
    "awt_null": {
        "type": "ratio",
        "num": "awt_null_orders",
        "den": "total_orders",
    },
    "awt_0min": {
        "type": "ratio",
        "num": "awt_0_orders",
        "den": "total_orders",
    },
    "awt_1_3min": {
        "type": "ratio",
        "num": "awt_1_3_orders",
        "den": "total_orders",
    },
    "awt_4plus": {
        "type": "ratio",
        "num": "awt_4plus_orders",
        "den": "total_orders",
    },

    # Reasons raw.
    "partner_perfo": "sum",
    "high_preptimes": "sum",
    "awt10": "sum",
    "other_partner_perfo": "sum",
    "prediction_model": "sum",

    # Reasons normalizados por órdenes.
    "partner_perfo_ratio": {
        "type": "ratio",
        "num": "partner_perfo",
        "den": "total_orders",
    },
    "high_preptimes_ratio": {
        "type": "ratio",
        "num": "high_preptimes",
        "den": "total_orders",
    },
    "awt10_ratio": {
        "type": "ratio",
        "num": "awt10",
        "den": "total_orders",
    },
    "other_partner_perfo_ratio": {
        "type": "ratio",
        "num": "other_partner_perfo",
        "den": "total_orders",
    },
    "prediction_model_ratio": {
        "type": "ratio",
        "num": "prediction_model",
        "den": "total_orders",
    },
}

numeric_cols_for_eval = [
    "ept", "ept_0",
    "awt", "awt_0",
    "tt", "tt_0",
    "ctp", "ctp_0",
    "total_orders", "total_orders_0",
    "ept_sum", "ept_sum_0",
    "ept_orders", "ept_orders_0",
    "awt_sum", "awt_sum_0",
    "awt_orders", "awt_orders_0",
    "pickup_time_sum", "pickup_time_sum_0",
    "pickup_time_orders", "pickup_time_orders_0",
    "cl_total_orders", "cl_total_orders_0",
    "slow_orders", "slow_orders_0",
    "non_seamless_orders", "non_seamless_orders_0",
    "awt_null_orders", "awt_null_orders_0",
    "awt_0_orders", "awt_0_orders_0",
    "awt_1_3_orders", "awt_1_3_orders_0",
    "awt_4plus_orders", "awt_4plus_orders_0",
    "awt_null", "awt_null_0",
    "awt_0min", "awt_0min_0",
    "awt_1_3min", "awt_1_3min_0",
    "awt_4plus", "awt_4plus_0",
    "partner_perfo", "partner_perfo_0",
    "high_preptimes", "high_preptimes_0",
    "awt10", "awt10_0",
    "other_partner_perfo", "other_partner_perfo_0",
    "prediction_model", "prediction_model_0",
]

numeric_cols_for_eval += [
    col
    for base_col in awt_raw_count_cols + awt_raw_share_cols
    for col in (base_col, f"{base_col}_0")
]

numeric_cols_for_eval = [
    col for col in numeric_cols_for_eval
    if col in df_eval.columns
]

df_eval[numeric_cols_for_eval] = df_eval[
    numeric_cols_for_eval
].apply(
    lambda series: pd.to_numeric(series, errors="coerce")
)

wave_summary = build_wave_summary(
    df_eval=df_eval,
    metric_config=metric_config,
)

wave_summary_santiago = build_wave_summary(
    df_eval=df_eval,
    metric_config=metric_config,
    city_filter=1,
)

wave_summary_regiones = build_wave_summary(
    df_eval=df_eval,
    metric_config=metric_config,
    city_filter=0,
)

wave_summary_city = build_wave_summary_city(
    df_eval=df_eval,
    metric_config=metric_config,
)

wave_summary


### Generadores de Holy Tables por dimensión y country level

In [ ]:
# ============================================================
# HOLY TABLES REUTILIZABLES
# ============================================================

HOLY_RESULT_METRICS = [
    ("ept_diff_nominal", "Δ ept (min)", 1),
    ("awt_diff_nominal", "Δ awt (min)", 1),
    ("ctp_diff_nominal", "Δ pickup time (min)", 1),
    ("high_preptimes_ratio_diff_pp", "Δ high preptimes (pp)", 1),
    ("awt10_ratio_diff_pp", "Δ AWT10 causal (pp)", 1),
    ("awt_null_diff_pp", "Δ awt_null (pp)", 1),
    ("awt_0min_diff_pp", "Δ awt_0min (pp)", 1),
    ("awt_1_3min_diff_pp", "Δ awt_1_3min (pp)", 1),
    ("awt_4plus_diff_pp", "Δ awt_4plus (pp)", 1),
]


COUNTRY_METRIC_SPECS = [
    ("ept", "ept", "ept_sum", "ept_orders", 1.0, "min"),
    ("awt", "awt", "awt_sum", "awt_orders", 1.0, "min"),
    (
        "ctp",
        "pickup time",
        "pickup_time_sum",
        "pickup_time_orders",
        1.0,
        "min",
    ),
    (
        "high_preptimes",
        "high preptimes",
        "high_preptimes",
        "total_orders",
        100.0,
        "pp",
    ),
    (
        "awt10",
        "AWT10 causal",
        "awt10",
        "total_orders",
        100.0,
        "pp",
    ),
    (
        "awt_null",
        "awt_null",
        "awt_null_orders",
        "total_orders",
        100.0,
        "pp",
    ),
    (
        "awt_0min",
        "awt_0min",
        "awt_0_orders",
        "total_orders",
        100.0,
        "pp",
    ),
    (
        "awt_1_3min",
        "awt_1_3min",
        "awt_1_3_orders",
        "total_orders",
        100.0,
        "pp",
    ),
    (
        "awt_4plus",
        "awt_4plus",
        "awt_4plus_orders",
        "total_orders",
        100.0,
        "pp",
    ),
]


def _holy_valid_flag_mask(series):
    clean = series.astype("string").str.strip()
    return (
        clean.notna()
        & clean.ne("")
        & ~clean.str.lower().isin([
            "nan",
            "none",
            "sin_flag",
            "<na>",
        ])
    )


def _fmt_holy_date(value):
    month_map = {
        1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
        5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
        9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec",
    }
    value = pd.to_datetime(value, errors="coerce")
    if pd.isna(value):
        return ""
    return f"{value.day:02d}/{month_map[value.month]}"


def _fmt_holy_value(value, kind):
    if pd.isna(value):
        return ""
    if kind == "int":
        return f"{value:.0f}"
    if kind == "pct":
        return f"{value * 100:.1f}%"
    if kind == "pct_already_scaled":
        return f"{value:.1f}%"
    if kind == "decimal_1":
        return f"{value:.1f}"
    if kind == "decimal_2":
        return f"{value:.2f}"
    return str(value)


def build_grouped_treatment_holy_table(
    source_df,
    metric_config,
    group_cols,
    drop_missing_flags=False,
):
    """
    Holy Table treatment-only para cualquier combinación de dimensiones.

    La cohorte viene desde df_eval, por lo que exige actividad Pre y Post.
    % Orders CL usa el total CL real de la wave como denominador.
    """
    group_cols = list(group_cols)
    df = source_df[
        source_df["analysis_group"].eq("treatment")
    ].copy()

    if drop_missing_flags and "flag" in group_cols:
        df = df[_holy_valid_flag_mask(df["flag"])].copy()

    if df.empty:
        raise ValueError(
            f"No quedaron filas treatment para {group_cols}."
        )

    summary = build_franchise_summary(
        df=df,
        metric_config=metric_config,
        group_cols=tuple(group_cols + ["analysis_group"]),
        post_weight_col="total_orders",
        pre_weight_col="total_orders_0",
    ).drop(columns="analysis_group")

    metadata = (
        df.groupby(group_cols, dropna=False)
        .agg(
            inicio_pre=("inicio_pre", "min"),
            fin_pre=("fin_pre", "max"),
            inicio_post=("inicio_post", "min"),
            fin_post=("fin_post", "max"),
            franchises=("franchise_name", "nunique"),
            vendors=("vendor_code", "nunique"),
            pre_orders=("total_orders_0", "sum"),
            post_orders=("total_orders", "sum"),
            pre_cl_orders=("cl_total_orders_0", "max"),
            post_cl_orders=("cl_total_orders", "max"),
        )
        .reset_index()
    )

    metadata["days"] = (
        pd.to_datetime(metadata["fin_post"])
        - pd.to_datetime(metadata["inicio_post"])
    ).dt.days + 1

    metadata["pre_orders_cl_share"] = np.where(
        metadata["pre_cl_orders"] > 0,
        metadata["pre_orders"] / metadata["pre_cl_orders"],
        np.nan,
    )
    metadata["post_orders_cl_share"] = np.where(
        metadata["post_cl_orders"] > 0,
        metadata["post_orders"] / metadata["post_cl_orders"],
        np.nan,
    )
    metadata["Pre"] = (
        metadata["inicio_pre"].map(_fmt_holy_date)
        + "-"
        + metadata["fin_pre"].map(_fmt_holy_date)
    )
    metadata["Post"] = (
        metadata["inicio_post"].map(_fmt_holy_date)
        + "-"
        + metadata["fin_post"].map(_fmt_holy_date)
    )

    out = metadata.merge(
        summary,
        on=group_cols,
        how="left",
        validate="1:1",
    )

    row_specs = [
        ("franchises", "franchises", "int"),
        ("vendors", "vendors", "int"),
        ("days", "días evaluados", "int"),
        ("post_orders", "Post #Orders", "int"),
        ("pre_orders", "Pre #Orders", "int"),
        (
            "pre_orders_cl_share",
            "% Orders CL Pre",
            "pct",
        ),
        (
            "post_orders_cl_share",
            "% Orders CL Post",
            "pct",
        ),
    ]
    row_specs += [
        (source_col, label, f"decimal_{decimals}")
        for source_col, label, decimals in HOLY_RESULT_METRICS
        if source_col in out.columns
    ]
    row_specs += [
        ("Pre", "Pre", "text"),
        ("Post", "Post", "text"),
    ]

    value_labels = []
    for source_col, label, kind in row_specs:
        if source_col not in out.columns:
            continue
        if kind == "text":
            out[label] = out[source_col].fillna("").astype(str)
        else:
            out[label] = pd.to_numeric(
                out[source_col], errors="coerce"
            ).map(lambda value, k=kind: _fmt_holy_value(value, k))
        value_labels.append(label)

    long = out[group_cols + value_labels].melt(
        id_vars=group_cols,
        value_vars=value_labels,
        var_name="metric",
        value_name="value",
    )
    long["metric"] = pd.Categorical(
        long["metric"],
        categories=value_labels,
        ordered=True,
    )

    final = long.pivot_table(
        index="metric",
        columns=group_cols,
        values="value",
        aggfunc="first",
        sort=False,
        observed=False,
    )
    final = final.reindex(value_labels)
    final.index.name = "metric"
    return final


def _ensure_country_awt_counts(df):
    df = df.copy()
    if "awt_1_3_orders" not in df.columns:
        cols = [f"awt_{minute}_orders" for minute in range(1, 4)]
        missing = [col for col in cols if col not in df.columns]
        if missing:
            raise KeyError(f"Faltan buckets AWT 1–3: {missing}")
        df["awt_1_3_orders"] = df[cols].apply(
            pd.to_numeric, errors="coerce"
        ).sum(axis=1, min_count=1)

    if "awt_4plus_orders" not in df.columns:
        cols = [
            f"awt_{minute}_orders"
            for minute in range(4, 21)
        ] + ["awt_21plus_orders"]
        missing = [col for col in cols if col not in df.columns]
        if missing:
            raise KeyError(f"Faltan buckets AWT 4+: {missing}")
        df["awt_4plus_orders"] = df[cols].apply(
            pd.to_numeric, errors="coerce"
        ).sum(axis=1, min_count=1)
    return df


def _aggregate_country_periods(df, group_cols):
    group_cols = list(group_cols)
    sum_cols = ["total_orders"]
    for _, _, num_col, den_col, _, _ in COUNTRY_METRIC_SPECS:
        sum_cols.extend([num_col, den_col])
    sum_cols = list(dict.fromkeys(sum_cols))

    missing = [col for col in sum_cols if col not in df.columns]
    if missing:
        raise KeyError(
            "Faltan columnas para la tabla country level: "
            f"{missing}"
        )

    work = df.copy()
    work[sum_cols] = work[sum_cols].apply(
        pd.to_numeric, errors="coerce"
    )

    out = (
        work.groupby(group_cols, dropna=False)[sum_cols]
        .sum(min_count=1)
        .reset_index()
    )

    for key, _, num_col, den_col, scale, _ in COUNTRY_METRIC_SPECS:
        out[key] = np.where(
            out[den_col] > 0,
            out[num_col] / out[den_col] * scale,
            np.nan,
        )
    return out


def build_wave_country_tables(source_raw, scope_label="CL"):
    """
    Devuelve dos tablas:

    1. inputs: una fila por wave + métrica con todos los numeradores,
       denominadores, niveles, deltas, peso, DID y fórmula del aporte.
    2. contributions: matriz compacta con solo los aportes estimados.

    Definiciones:
    - CL observado = all_chile, incluidas otras waves.
    - CL limpio = treatment de la wave + resto sin ninguna wave.
    - aporte = peso treatment Post en CL × (Δ treatment - Δ resto).

    Country level usa todas las órdenes observadas en cada período para
    reconciliar con CL; no exige una cohorte vendor balanceada.
    """
    raw = _ensure_country_awt_counts(source_raw)
    needed_groups = {"treatment", "resto_de_chile", "all_chile"}
    present = set(raw["analysis_group"].dropna().astype(str))
    missing_groups = needed_groups - present

    if missing_groups:
        raise ValueError(
            "Vuelve a ejecutar la query y cargar su CSV. "
            f"Faltan grupos: {sorted(missing_groups)}"
        )

    raw = raw[raw["analysis_group"].isin(needed_groups)].copy()

    by_group = _aggregate_country_periods(
        raw,
        ["wave", "analysis_group", "period"],
    )
    clean = _aggregate_country_periods(
        raw[raw["analysis_group"].isin([
            "treatment",
            "resto_de_chile",
        ])],
        ["wave", "period"],
    )

    def row_for(frame, wave, period, analysis_group=None):
        mask = frame["wave"].astype(str).eq(str(wave))
        mask &= frame["period"].eq(period)

        if analysis_group is not None:
            mask &= frame["analysis_group"].eq(analysis_group)

        rows = frame[mask]
        return None if rows.empty else rows.iloc[0]

    metadata = (
        raw[raw["analysis_group"].eq("all_chile")]
        .groupby("wave", dropna=False)
        .agg(
            inicio_pre=("inicio_pre", "min"),
            fin_pre=("fin_pre", "max"),
            inicio_post=("inicio_post", "min"),
            fin_post=("fin_post", "max"),
        )
        .reset_index()
    )

    treatment_metadata = (
        raw[raw["analysis_group"].eq("treatment")]
        .groupby("wave", dropna=False)
        .agg(
            franchises=("franchise_name", "nunique"),
            vendors=("vendor_code", "nunique"),
        )
        .reset_index()
    )
    metadata = metadata.merge(
        treatment_metadata,
        on="wave",
        how="left",
        validate="1:1",
    )

    input_records = []
    contribution_records = []

    for meta in metadata.itertuples(index=False):
        wave = meta.wave
        all_pre = row_for(by_group, wave, "pre", "all_chile")
        all_post = row_for(by_group, wave, "post", "all_chile")
        treatment_pre = row_for(by_group, wave, "pre", "treatment")
        treatment_post = row_for(by_group, wave, "post", "treatment")
        rest_pre = row_for(
            by_group, wave, "pre", "resto_de_chile"
        )
        rest_post = row_for(
            by_group, wave, "post", "resto_de_chile"
        )
        clean_pre = row_for(clean, wave, "pre")
        clean_post = row_for(clean, wave, "post")

        required_rows = [
            all_pre,
            all_post,
            treatment_pre,
            treatment_post,
            rest_pre,
            rest_post,
            clean_pre,
            clean_post,
        ]

        if any(row is None for row in required_rows):
            raise ValueError(
                f"{wave}: faltan filas Pre/Post para country level."
            )

        all_orders_pre = all_pre["total_orders"]
        all_orders_post = all_post["total_orders"]
        treatment_orders_pre = treatment_pre["total_orders"]
        treatment_orders_post = treatment_post["total_orders"]
        rest_orders_pre = rest_pre["total_orders"]
        rest_orders_post = rest_post["total_orders"]

        other_wave_orders_pre = max(
            all_orders_pre
            - treatment_orders_pre
            - rest_orders_pre,
            0,
        )
        other_wave_orders_post = max(
            all_orders_post
            - treatment_orders_post
            - rest_orders_post,
            0,
        )

        common = {
            "wave": wave,
            "universo_country": scope_label,
            "pre_inicio": str(pd.to_datetime(meta.inicio_pre).date()),
            "pre_fin": str(pd.to_datetime(meta.fin_pre).date()),
            "post_inicio": str(pd.to_datetime(meta.inicio_post).date()),
            "post_fin": str(pd.to_datetime(meta.fin_post).date()),
            "dias_evaluados": (
                pd.Timestamp(meta.fin_post)
                - pd.Timestamp(meta.inicio_post)
            ).days + 1,
            "franchises_treatment_observadas": meta.franchises,
            "vendors_treatment_observados": meta.vendors,
            "orders_treatment_pre": treatment_orders_pre,
            "orders_treatment_post": treatment_orders_post,
            "orders_resto_limpio_pre": rest_orders_pre,
            "orders_resto_limpio_post": rest_orders_post,
            "orders_otras_waves_pre": other_wave_orders_pre,
            "orders_otras_waves_post": other_wave_orders_post,
            "orders_cl_pre": all_orders_pre,
            "orders_cl_post": all_orders_post,
            "share_orders_treatment_cl_pre": safe_ratio(
                treatment_orders_pre, all_orders_pre
            ),
            "share_orders_treatment_cl_post": safe_ratio(
                treatment_orders_post, all_orders_post
            ),
            "share_orders_otras_waves_cl_pre": safe_ratio(
                other_wave_orders_pre, all_orders_pre
            ),
            "share_orders_otras_waves_cl_post": safe_ratio(
                other_wave_orders_post, all_orders_post
            ),
            "cobertura_cl_limpio_pre": safe_ratio(
                treatment_orders_pre + rest_orders_pre,
                all_orders_pre,
            ),
            "cobertura_cl_limpio_post": safe_ratio(
                treatment_orders_post + rest_orders_post,
                all_orders_post,
            ),
        }

        for (
            key,
            label,
            num_col,
            den_col,
            scale,
            unit,
        ) in COUNTRY_METRIC_SPECS:
            delta_treatment = (
                treatment_post[key] - treatment_pre[key]
            )
            delta_rest = rest_post[key] - rest_pre[key]
            did = delta_treatment - delta_rest

            treatment_weight_post = safe_ratio(
                treatment_post[den_col],
                all_post[den_col],
            )
            contribution = (
                treatment_weight_post * did
                if pd.notna(treatment_weight_post)
                else np.nan
            )

            delta_all = all_post[key] - all_pre[key]
            delta_clean = clean_post[key] - clean_pre[key]

            formula = ""
            if all(pd.notna(value) for value in [
                treatment_weight_post,
                delta_treatment,
                delta_rest,
            ]):
                formula = (
                    f"{treatment_weight_post:.8f} × "
                    f"({delta_treatment:.8f} - {delta_rest:.8f})"
                )

            input_records.append({
                **common,
                "metric": label,
                "unit": unit,
                "numerator_column": num_col,
                "denominator_column": den_col,
                "scale": scale,

                "treatment_pre_numerator": treatment_pre[num_col],
                "treatment_pre_denominator": treatment_pre[den_col],
                "treatment_pre_value": treatment_pre[key],
                "treatment_post_numerator": treatment_post[num_col],
                "treatment_post_denominator": treatment_post[den_col],
                "treatment_post_value": treatment_post[key],
                "delta_treatment": delta_treatment,

                "resto_pre_numerator": rest_pre[num_col],
                "resto_pre_denominator": rest_pre[den_col],
                "resto_pre_value": rest_pre[key],
                "resto_post_numerator": rest_post[num_col],
                "resto_post_denominator": rest_post[den_col],
                "resto_post_value": rest_post[key],
                "delta_resto": delta_rest,

                "all_chile_pre_numerator": all_pre[num_col],
                "all_chile_pre_denominator": all_pre[den_col],
                "all_chile_pre_value": all_pre[key],
                "all_chile_post_numerator": all_post[num_col],
                "all_chile_post_denominator": all_post[den_col],
                "all_chile_post_value": all_post[key],
                "delta_cl_observado": delta_all,

                "cl_limpio_pre_numerator": clean_pre[num_col],
                "cl_limpio_pre_denominator": clean_pre[den_col],
                "cl_limpio_pre_value": clean_pre[key],
                "cl_limpio_post_numerator": clean_post[num_col],
                "cl_limpio_post_denominator": clean_post[den_col],
                "cl_limpio_post_value": clean_post[key],
                "delta_cl_limpio": delta_clean,

                "treatment_post_weight_cl": treatment_weight_post,
                "treatment_post_weight_cl_pct": (
                    treatment_weight_post * 100
                    if pd.notna(treatment_weight_post)
                    else np.nan
                ),
                "did_treatment_vs_resto": did,
                "formula_aporte": formula,
                "aporte_estimado_wave_cl": contribution,
            })

            contribution_records.append({
                "wave": wave,
                "metric": f"{label} ({unit})",
                "aporte_estimado_wave_cl": contribution,
            })

    inputs = pd.DataFrame(input_records)
    numeric_columns = inputs.select_dtypes(include=[np.number]).columns
    inputs[numeric_columns] = inputs[numeric_columns].round(6)

    metric_order = [
        f"{label} ({unit})"
        for _, label, _, _, _, unit in COUNTRY_METRIC_SPECS
    ]
    contributions_long = pd.DataFrame(contribution_records)
    contributions_long["metric"] = pd.Categorical(
        contributions_long["metric"],
        categories=metric_order,
        ordered=True,
    )

    contributions = contributions_long.pivot_table(
        index="metric",
        columns="wave",
        values="aporte_estimado_wave_cl",
        aggfunc="first",
        sort=False,
        observed=False,
    ).reindex(metric_order)
    contributions = contributions.round(2)
    contributions.index.name = "metric"
    contributions.columns.name = None

    return inputs, contributions


In [ ]:
wave_treatment_deltas_T = build_treatment_delta_transpose(wave_summary)

wave_treatment_deltas_T

# Data Visualization

# 🚀 Results

### diff table control vs. treatment

In [ ]:
wave_diff = build_wave_diff_table(wave_summary)
wave_diff = add_wave_did_cols(wave_diff)

wave_diff

In [ ]:
wave_diff_santiago = build_wave_diff_table(wave_summary_santiago)
wave_diff_santiago = add_wave_did_cols(wave_diff_santiago)

print("SUMMARY SANTIAGO")
wave_diff_santiago

In [ ]:
wave_diff_regiones = build_wave_diff_table(wave_summary_regiones)
wave_diff_regiones = add_wave_did_cols(wave_diff_regiones)

print("SUMMARY REGIONES")
wave_diff_regiones

* incorporar pre y post
* delta ept
* incorporar data monthly

In [ ]:
delta_table = table_simplify(wave_diff)
delta_table

In [ ]:
delta_table = table_simplify(wave_diff, df_eval)
delta_table

### 🔥THE HOLY TABLE

Se generan dos tablas: una por `wave` y otra por `wave + flag`. Ambas incluyen `% Orders CL Pre` y `% Orders CL Post`.

In [ ]:
# ============================================================
# HOLY TABLE 1: agregación por wave
# ============================================================
delta_table_treatment = table_simplify_treatment(
    df=wave_diff,
    source_df=df_eval
)

final_table = format_wave_treatment_table(
    delta_table_treatment
)

print("THE HOLY TABLE — WAVE")
display(final_table)

In [ ]:
final_table_simple = final_table.copy()

In [ ]:
# ============================================================
# HOLY TABLE 2: agregación por wave + flag
# ============================================================
df_eval_wave_flag = build_wave_flag_eval(df_eval)
df_eval_wave_flag = df_eval_wave_flag[
    _holy_valid_flag_mask(df_eval_wave_flag["flag"])
].copy()

wave_flag_summary = build_wave_flag_summary(
    df_eval_wave_flag=df_eval_wave_flag,
    metric_config=metric_config
)

wave_flag_diff = build_wave_flag_diff_table(
    wave_flag_summary
)
wave_flag_diff = add_wave_did_cols(wave_flag_diff)

delta_table_treatment_wave_flag = table_simplify_treatment(
    df=wave_flag_diff,
    source_df=df_eval_wave_flag
)

final_table_wave_flag = format_wave_treatment_table(
    delta_table_treatment_wave_flag
)

print("THE HOLY TABLE — WAVE + FLAG")
display(final_table_wave_flag)

In [ ]:
general_table_treatment = (
    format_analysis_group_table(
        df_eval=df_eval,
        metric_config=metric_config,
        analysis_groups=("treatment",)
    )
    .rename(columns={"treatment": "General"})
    .drop(index="share orders", errors="ignore")
)

general_table_treatment

#### regiones

In [ ]:
delta_table_treatment_regiones = table_simplify_treatment(
    df=wave_diff_regiones,
    source_df=df_eval[df_eval["ciudad"] == 0]
)

format_wave_treatment_table(delta_table_treatment_regiones)

#### santiago

In [ ]:
delta_table_treatment_santiago = table_simplify_treatment(
    df=wave_diff_santiago,
    source_df=df_eval[df_eval["ciudad"] == 1]
)

format_wave_treatment_table(delta_table_treatment_santiago)

## Control vs. Treatment

In [ ]:
final_table_analysis_group = format_analysis_group_table(
    df_eval=df_eval,
    metric_config=metric_config
)

final_table_analysis_group


### Holy Tables adicionales

- `final_table`: wave.
- `final_table_wave_flag`: wave + flag, sin flags nulas o `SIN_FLAG`.
- `final_table_wave_santiago` y `final_table_wave_regiones`: wave, separados por ciudad.
- `final_table_wave_country_inputs`: auditoría completa del cálculo nacional.
- `final_table_wave_country_contributions`: solo aportes estimados.
- `final_table_wave_flag_santiago` y `final_table_wave_flag_regiones`: wave + flag, separados por ciudad y sin flags nulas.

En country level, `resto limpio` excluye vendors de todas las waves. La
tabla muestra por separado el CL observado, el CL limpio y el aporte
estimado de la wave contra la tendencia del resto.


In [ ]:
# ============================================================
# HOLY TABLE 3: ciudad + wave
# ============================================================
df_eval_city = df_eval.copy()
df_eval_city["city"] = np.select(
    [
        df_eval_city["ciudad"].eq(1),
        df_eval_city["ciudad"].eq(0),
    ],
    ["Santiago", "Regiones"],
    default="SIN_CIUDAD",
)

final_table_city_wave = build_grouped_treatment_holy_table(
    source_df=df_eval_city,
    metric_config=metric_config,
    group_cols=["city", "wave"],
)

final_table_wave_santiago = (
    final_table_city_wave
    .xs("Santiago", level="city", axis=1)
    .copy()
)

final_table_wave_regiones = (
    final_table_city_wave
    .xs("Regiones", level="city", axis=1)
    .copy()
)

print("THE HOLY TABLE — SANTIAGO + WAVE")
display(final_table_wave_santiago)

print("THE HOLY TABLE — REGIONES + WAVE")
display(final_table_wave_regiones)


# ============================================================
# HOLY TABLE 3B: wave solo vertical restaurants
# ============================================================
if "vertical_type" not in df_eval.columns:
    raise KeyError(
        "Falta vertical_type en df_eval. Vuelve a ejecutar la query "
        "y las celdas de carga/consolidación antes de esta celda."
    )

df_eval_restaurants = df_eval[
    df_eval["vertical_type"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("restaurants")
].copy()

final_table_wave_restaurants = build_grouped_treatment_holy_table(
    source_df=df_eval_restaurants,
    metric_config=metric_config,
    group_cols=["wave"],
)

print("THE HOLY TABLE — WAVE | SOLO RESTAURANTS")
display(final_table_wave_restaurants)


# ============================================================
# HOLY TABLE 4A: inputs auditables de country level
# HOLY TABLE 4B: solo aportes estimados
# ============================================================
(
    final_table_wave_country_inputs,
    final_table_wave_country_contributions,
) = build_wave_country_tables(
    source_raw=df_pre_post_consolidated,
)

print("COUNTRY LEVEL — INPUTS COMPLETOS DEL CÁLCULO")
display(final_table_wave_country_inputs)

print("COUNTRY LEVEL — SOLO APORTES ESTIMADOS")
display(final_table_wave_country_contributions)


# ============================================================
# HOLY TABLE 4C/4D: country level solo restaurants
# ============================================================
country_restaurants_raw = df_pre_post_consolidated[
    df_pre_post_consolidated["vertical_type"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("restaurants")
].copy()

restaurant_treatment_waves = set(
    country_restaurants_raw.loc[
        country_restaurants_raw["analysis_group"].eq("treatment"),
        "wave",
    ].astype(str)
)
country_restaurants_raw = country_restaurants_raw[
    country_restaurants_raw["wave"]
    .astype(str)
    .isin(restaurant_treatment_waves)
].copy()

restaurant_country_groups = set(
    country_restaurants_raw["analysis_group"]
    .dropna()
    .astype(str)
)
required_restaurant_country_groups = {
    "treatment", "resto_de_chile", "all_chile"
}
missing_restaurant_country_groups = (
    required_restaurant_country_groups
    - restaurant_country_groups
)

if missing_restaurant_country_groups:
    raise ValueError(
        "El CSV no contiene resto_de_chile/all_chile desglosado "
        "para restaurants. Vuelve a ejecutar la query generada por "
        "esta versión y carga el CSV nuevo. Faltan: "
        f"{sorted(missing_restaurant_country_groups)}"
    )

(
    final_table_wave_country_restaurants_inputs,
    final_table_wave_country_restaurants_contributions,
) = build_wave_country_tables(
    source_raw=country_restaurants_raw,
    scope_label="CL restaurants",
)

print("COUNTRY LEVEL RESTAURANTS — INPUTS COMPLETOS")
display(final_table_wave_country_restaurants_inputs)

print("COUNTRY LEVEL RESTAURANTS — SOLO APORTES ESTIMADOS")
display(final_table_wave_country_restaurants_contributions)


# ============================================================
# HOLY TABLE 5: ciudad + wave + flag
# ============================================================
final_table_city_wave_flag = build_grouped_treatment_holy_table(
    source_df=df_eval_city,
    metric_config=metric_config,
    group_cols=["city", "wave", "flag"],
    drop_missing_flags=True,
)

final_table_wave_flag_santiago = (
    final_table_city_wave_flag
    .xs("Santiago", level="city", axis=1)
    .copy()
)

final_table_wave_flag_regiones = (
    final_table_city_wave_flag
    .xs("Regiones", level="city", axis=1)
    .copy()
)

print("THE HOLY TABLE — SANTIAGO + WAVE + FLAG")
display(final_table_wave_flag_santiago)

print("THE HOLY TABLE — REGIONES + WAVE + FLAG")
display(final_table_wave_flag_regiones)


# Appendix de auditoría

Estas tablas dejan visibles los niveles Pre y Post que alimentan las Holy Tables.

- AWT0: espera exactamente igual a 0.
- AWT1: espera mayor que 0 y menor que 1 minuto.
- AWT2: 1 a menos de 2 minutos; la misma regla continúa hasta AWT20.
- AWT21+: 20 minutos o más.
- Los shares usan siempre órdenes totales del mismo grupo y período como denominador.
- AWT2plus se conserva solamente como agregado histórico; no se suma junto a los buckets detallados.


In [ ]:
import numpy as np
import pandas as pd


def _valid_flag_mask(series):
    clean = series.astype("string").str.strip()
    return (
        clean.notna()
        & clean.ne("")
        & ~clean.str.lower().isin([
            "nan",
            "none",
            "sin_flag",
        ])
    )


def build_holy_input_appendix(
    summary,
    source_df,
    group_cols,
):
    """
    Una fila por grupo con los valores Pre y Post que generan los
    deltas de la Holy Table. Los porcentajes quedan en escala 0–100.
    """

    summary_treatment = summary[
        summary["analysis_group"].eq("treatment")
    ].copy()

    source_treatment = source_df[
        source_df["analysis_group"].eq("treatment")
    ].copy()

    metadata = (
        source_treatment
        .groupby(group_cols, dropna=False)
        .agg(
            inicio_pre=("inicio_pre", "min"),
            fin_pre=("fin_pre", "max"),
            inicio_post=("inicio_post", "min"),
            fin_post=("fin_post", "max"),
            franchises=("franchise_name", "nunique"),
            vendors=("vendor_code", "nunique"),
            orders_pre=("total_orders_0", "sum"),
            orders_post=("total_orders", "sum"),
            orders_cl_pre=("cl_total_orders_0", "max"),
            orders_cl_post=("cl_total_orders", "max"),
        )
        .reset_index()
    )

    metadata["days_pre"] = (
        pd.to_datetime(metadata["fin_pre"])
        - pd.to_datetime(metadata["inicio_pre"])
    ).dt.days + 1

    metadata["days_post"] = (
        pd.to_datetime(metadata["fin_post"])
        - pd.to_datetime(metadata["inicio_post"])
    ).dt.days + 1

    metadata["orders_cl_share_pre_pct"] = np.where(
        metadata["orders_cl_pre"] > 0,
        metadata["orders_pre"] / metadata["orders_cl_pre"] * 100,
        np.nan,
    )

    metadata["orders_cl_share_post_pct"] = np.where(
        metadata["orders_cl_post"] > 0,
        metadata["orders_post"] / metadata["orders_cl_post"] * 100,
        np.nan,
    )

    metric_columns = [
        "avg_ept_pre",
        "avg_ept_post",
        "avg_awt_pre",
        "avg_awt_post",
        "avg_ctp_pre",
        "avg_ctp_post",
        "partner_perfo_pre",
        "partner_perfo_post",
        "avg_partner_perfo_ratio_pre",
        "avg_partner_perfo_ratio_post",
        "other_partner_perfo_pre",
        "other_partner_perfo_post",
        "avg_other_partner_perfo_ratio_pre",
        "avg_other_partner_perfo_ratio_post",
        "high_preptimes_pre",
        "high_preptimes_post",
        "avg_high_preptimes_ratio_pre",
        "avg_high_preptimes_ratio_post",
        "awt10_pre",
        "awt10_post",
        "avg_awt10_ratio_pre",
        "avg_awt10_ratio_post",
        "avg_awt_null_pre",
        "avg_awt_null_post",
        "avg_awt_0min_pre",
        "avg_awt_0min_post",
        "avg_awt_1_3min_pre",
        "avg_awt_1_3min_post",
        "avg_awt_4plus_pre",
        "avg_awt_4plus_post",
    ]

    metric_columns = [
        column
        for column in metric_columns
        if column in summary_treatment.columns
    ]

    out = metadata.merge(
        summary_treatment[group_cols + metric_columns],
        on=group_cols,
        how="left",
        validate="1:1",
    )

    rename_map = {
        "inicio_pre": "Pre inicio",
        "fin_pre": "Pre fin",
        "inicio_post": "Post inicio",
        "fin_post": "Post fin",
        "days_pre": "Pre días",
        "days_post": "Post días",
        "franchises": "Franchises",
        "vendors": "Vendors",
        "orders_pre": "Pre #Orders",
        "orders_post": "Post #Orders",
        "orders_cl_pre": "Pre #Orders CL",
        "orders_cl_post": "Post #Orders CL",
        "orders_cl_share_pre_pct": "% Orders CL Pre",
        "orders_cl_share_post_pct": "% Orders CL Post",
        "avg_ept_pre": "EPT Pre (min)",
        "avg_ept_post": "EPT Post (min)",
        "avg_awt_pre": "AWT Pre (min)",
        "avg_awt_post": "AWT Post (min)",
        "avg_ctp_pre": "Pickup Pre (min)",
        "avg_ctp_post": "Pickup Post (min)",
        "partner_perfo_pre": "Partner Performance Pre (impacto atribuido)",
        "partner_perfo_post": "Partner Performance Post (impacto atribuido)",
        "avg_partner_perfo_ratio_pre": "Partner Performance Pre (%)",
        "avg_partner_perfo_ratio_post": "Partner Performance Post (%)",
        "other_partner_perfo_pre": "Other Partner Performance Pre (impacto atribuido)",
        "other_partner_perfo_post": "Other Partner Performance Post (impacto atribuido)",
        "avg_other_partner_perfo_ratio_pre": "Other Partner Performance Pre (%)",
        "avg_other_partner_perfo_ratio_post": "Other Partner Performance Post (%)",
        "high_preptimes_pre": "High preptimes Pre (impacto atribuido)",
        "high_preptimes_post": "High preptimes Post (impacto atribuido)",
        "avg_high_preptimes_ratio_pre": "High preptimes Pre (%)",
        "avg_high_preptimes_ratio_post": "High preptimes Post (%)",
        "awt10_pre": "AWT10 causal Pre (impacto atribuido)",
        "awt10_post": "AWT10 causal Post (impacto atribuido)",
        "avg_awt10_ratio_pre": "AWT10 causal Pre (%)",
        "avg_awt10_ratio_post": "AWT10 causal Post (%)",
        "avg_awt_null_pre": "AWT null Pre (%)",
        "avg_awt_null_post": "AWT null Post (%)",
        "avg_awt_0min_pre": "AWT 0 min Pre (%)",
        "avg_awt_0min_post": "AWT 0 min Post (%)",
        "avg_awt_1_3min_pre": "AWT 1–3 min Pre (%)",
        "avg_awt_1_3min_post": "AWT 1–3 min Post (%)",
        "avg_awt_4plus_pre": "AWT 4+ min Pre (%)",
        "avg_awt_4plus_post": "AWT 4+ min Post (%)",
    }

    out = out.rename(columns=rename_map)

    for metric_name in [
        "Partner Performance",
        "Other Partner Performance",
    ]:
        pre_col = f"{metric_name} Pre (%)"
        post_col = f"{metric_name} Post (%)"

        if {pre_col, post_col}.issubset(out.columns):
            out[f"Δ {metric_name} (pp)"] = (
                out[post_col] - out[pre_col]
            )

    return out.round(2)


def build_awt_distribution_appendix(
    source_df,
    group_cols,
    drop_missing_flags=False,
):
    """
    Tabla larga usando exclusivamente los cuatro buckets oficiales:
    Null, 0 min, 1–3 min y 4+ min.
    """

    df = source_df[
        source_df["analysis_group"].eq("treatment")
    ].copy()

    if drop_missing_flags and "flag" in group_cols:
        df = df[_valid_flag_mask(df["flag"])].copy()

    rows = []

    for group_key, group_df in df.groupby(
        group_cols,
        dropna=False,
        sort=False,
    ):
        if len(group_cols) == 1 and not isinstance(group_key, tuple):
            group_key = (group_key,)

        identifiers = dict(zip(group_cols, group_key))
        total_pre = group_df["total_orders_0"].sum(min_count=1)
        total_post = group_df["total_orders"].sum(min_count=1)

        for bucket_order, bucket in enumerate(
            AWT_DISTRIBUTION_BUCKETS
        ):
            count_col = AWT_SHARE_TO_COUNT[bucket]
            pre_count_col = f"{count_col}_0"

            pre_orders = group_df[pre_count_col].sum(min_count=1)
            post_orders = group_df[count_col].sum(min_count=1)

            rows.append({
                **identifiers,
                "bucket": bucket,
                "bucket_label": AWT_BUCKET_LABELS[bucket],
                "bucket_order": bucket_order,
                "pre_orders": pre_orders,
                "post_orders": post_orders,
                "pre_total_orders": total_pre,
                "post_total_orders": total_post,
                "pre_share_pct": (
                    pre_orders / total_pre * 100
                    if pd.notna(total_pre) and total_pre > 0
                    else np.nan
                ),
                "post_share_pct": (
                    post_orders / total_post * 100
                    if pd.notna(total_post) and total_post > 0
                    else np.nan
                ),
            })

    out = pd.DataFrame(rows)
    out["change_pp"] = (
        out["post_share_pct"] - out["pre_share_pct"]
    )

    return (
        out.sort_values(group_cols + ["bucket_order"])
        .reset_index(drop=True)
        .round(4)
    )


def build_awt_detailed_distribution_appendix(
    source_df,
    group_cols,
    drop_missing_flags=False,
):
    """
    Distribución treatment con eje Null, 0, 1, ..., 10, 11+.
    11+ suma todos los buckets individuales desde 11 en adelante.
    """
    group_cols = list(group_cols)
    df = source_df[
        source_df["analysis_group"].eq("treatment")
    ].copy()

    if drop_missing_flags and "flag" in group_cols:
        df = df[_valid_flag_mask(df["flag"])].copy()

    post_count_cols = list(dict.fromkeys(
        count_col
        for bucket in AWT_DETAILED_BUCKETS
        for count_col in AWT_DETAILED_COUNT_COLUMNS[bucket]
    ))
    pre_count_cols = [f"{column}_0" for column in post_count_cols]
    required = [
        "total_orders", "total_orders_0",
        *post_count_cols, *pre_count_cols,
    ]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise KeyError(
            "Faltan columnas para AWT detallado Null–11+: "
            + ", ".join(missing)
        )

    df[required] = df[required].apply(
        lambda series: pd.to_numeric(series, errors="coerce")
    )
    rows = []

    for group_key, group_df in df.groupby(
        group_cols, dropna=False, sort=False
    ):
        if len(group_cols) == 1 and not isinstance(group_key, tuple):
            group_key = (group_key,)

        identifiers = dict(zip(group_cols, group_key))
        total_pre = group_df["total_orders_0"].sum(min_count=1)
        total_post = group_df["total_orders"].sum(min_count=1)

        for bucket_order, bucket in enumerate(AWT_DETAILED_BUCKETS):
            post_cols = AWT_DETAILED_COUNT_COLUMNS[bucket]
            pre_cols = [f"{column}_0" for column in post_cols]

            pre_orders = (
                group_df[pre_cols]
                .sum(axis=1, min_count=1)
                .sum(min_count=1)
            )
            post_orders = (
                group_df[post_cols]
                .sum(axis=1, min_count=1)
                .sum(min_count=1)
            )

            rows.append({
                **identifiers,
                "bucket": bucket,
                "bucket_label": AWT_DETAILED_BUCKET_LABELS[bucket],
                "bucket_order": bucket_order,
                "pre_orders": pre_orders,
                "post_orders": post_orders,
                "pre_total_orders": total_pre,
                "post_total_orders": total_post,
                "pre_share_pct": (
                    pre_orders / total_pre * 100
                    if pd.notna(total_pre) and total_pre > 0
                    else np.nan
                ),
                "post_share_pct": (
                    post_orders / total_post * 100
                    if pd.notna(total_post) and total_post > 0
                    else np.nan
                ),
            })

    out = pd.DataFrame(rows)
    out["change_pp"] = (
        out["post_share_pct"] - out["pre_share_pct"]
    )
    return (
        out.sort_values(group_cols + ["bucket_order"])
        .reset_index(drop=True)
        .round(4)
    )


# Inputs exactos de las dos Holy Tables.
appendix_holy_wave = build_holy_input_appendix(
    summary=wave_summary,
    source_df=df_eval,
    group_cols=["wave"],
)

appendix_holy_wave_flag = build_holy_input_appendix(
    summary=wave_flag_summary,
    source_df=df_eval_wave_flag,
    group_cols=["wave", "flag"],
)


# Distribuciones AWT con los mismos cuatro buckets.
appendix_awt_wave = build_awt_distribution_appendix(
    source_df=df_eval,
    group_cols=["wave"],
)

appendix_awt_wave_flag = build_awt_distribution_appendix(
    source_df=df_eval_wave_flag,
    group_cols=["wave", "flag"],
    drop_missing_flags=True,
)

appendix_awt_flag = build_awt_distribution_appendix(
    source_df=df_eval,
    group_cols=["flag"],
    drop_missing_flags=True,
)


# Distribuciones detalladas para los gráficos Null, 0, 1, ..., 10, 11+.
appendix_awt_detailed_wave = build_awt_detailed_distribution_appendix(
    source_df=df_eval,
    group_cols=["wave"],
)

appendix_awt_detailed_wave_flag = build_awt_detailed_distribution_appendix(
    source_df=df_eval_wave_flag,
    group_cols=["wave", "flag"],
    drop_missing_flags=True,
)

appendix_awt_detailed_flag = build_awt_detailed_distribution_appendix(
    source_df=df_eval,
    group_cols=["flag"],
    drop_missing_flags=True,
)


# Summary agregado por flag real.
flag_eval = df_eval[
    df_eval["analysis_group"].eq("treatment")
    & _valid_flag_mask(df_eval["flag"])
].copy()

flag_summary = build_franchise_summary(
    df=flag_eval,
    metric_config=metric_config,
    group_cols=("flag", "analysis_group"),
    post_weight_col="total_orders",
    pre_weight_col="total_orders_0",
).round(2)


print("APPENDIX — INPUTS HOLY TABLE POR WAVE")
display(appendix_holy_wave)

print("APPENDIX — INPUTS HOLY TABLE POR WAVE + FLAG")
display(appendix_holy_wave_flag)

print("APPENDIX — DISTRIBUCIÓN AWT POR WAVE")
display(appendix_awt_wave)

print("APPENDIX — DISTRIBUCIÓN AWT POR WAVE + FLAG")
display(appendix_awt_wave_flag)

print("APPENDIX — DISTRIBUCIÓN AWT POR FLAG")
display(appendix_awt_flag)


# Pegar tablas en sheets

In [ ]:
!pip install -q gspread-dataframe

import gspread
import google.auth
import numpy as np
import pandas as pd

from google.colab import auth
from gspread_dataframe import set_with_dataframe
from gspread.exceptions import WorksheetNotFound


auth.authenticate_user()

credentials, _ = google.auth.default(scopes=[
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
])

gc = gspread.authorize(credentials)

sheet_id = "1OLY68XweFeVXIUnT9CfImdf-sjAcmVJNLDcSKJMUJM0"
spreadsheet = gc.open_by_key(sheet_id)


tablas_a_exportar = {
    # Tablas finales generales
    "final_table": final_table,
    "final_table_wave_flag": final_table_wave_flag,
    "final_table_wave_restaurants": final_table_wave_restaurants,
    "final_table_analysis_group": final_table_analysis_group,

    # Tablas finales separadas por ciudad
    "final_table_wave_santiago": final_table_wave_santiago,
    "final_table_wave_regiones": final_table_wave_regiones,
    "final_table_wave_flag_santiago": final_table_wave_flag_santiago,
    "final_table_wave_flag_regiones": final_table_wave_flag_regiones,

    # Country level
    "final_table_wave_country_inputs": final_table_wave_country_inputs,
    "final_table_wave_country_contributions": final_table_wave_country_contributions,
    "final_table_wave_country_restaurants_inputs": final_table_wave_country_restaurants_inputs,
    "final_table_wave_country_restaurants_contributions": final_table_wave_country_restaurants_contributions,

    # ========================================================
    # TABLAS TÉCNICAS — DESCOMENTAR SOLO CUANDO SE NECESITEN
    # ========================================================
    # "wave_summary": wave_summary,
    # "wave_flag_summary": wave_flag_summary,
    # "flag_summary": flag_summary,
    # "wave_diff": wave_diff,
    # "wave_flag_diff": wave_flag_diff,
    # "appendix_holy_wave": appendix_holy_wave,
    # "appendix_holy_wave_flag": appendix_holy_wave_flag,
    # "appendix_awt_wave": appendix_awt_wave,
    # "appendix_awt_wave_flag": appendix_awt_wave_flag,
    # "appendix_awt_flag": appendix_awt_flag,
    # "appendix_awt_detailed_wave": appendix_awt_detailed_wave,
    # "appendix_awt_detailed_wave_flag": appendix_awt_detailed_wave_flag,
    # "appendix_awt_detailed_flag": appendix_awt_detailed_flag,
    # "cohort_audit_summary": cohort_audit_summary,
    # "cohort_lost_cases": cohort_lost_cases,
    # "cohort_audit_detail": cohort_audit_detail,
    # "cohort_audit_observed": cohort_audit_observed,
    # "city_merge_audit": city_consolidation_audit,
}


def prepare_table_for_sheets(table):
    """
    Convierte una tabla a una matriz lista para Sheets.

    Las tablas con columnas MultiIndex conservan tantos encabezados
    como dimensiones tengan.
    """

    clean = (
        table.copy()
        .replace([np.inf, -np.inf], np.nan)
        .fillna("")
    )

    if isinstance(clean.columns, pd.MultiIndex):
        header_rows = []

        for level in range(clean.columns.nlevels):
            level_name = (
                "metric"
                if level == 0
                else clean.columns.names[level] or f"nivel_{level + 1}"
            )
            header_rows.append(
                [level_name]
                + [
                    str(value)
                    for value in clean.columns.get_level_values(level)
                ]
            )

        body = [
            [str(index)] + row.tolist()
            for index, row in clean.iterrows()
        ]

        export_df = pd.DataFrame(header_rows + body)
        return export_df, False, clean.columns.nlevels

    if (
        not isinstance(clean.index, pd.RangeIndex)
        or clean.index.name is not None
    ):
        clean = clean.reset_index()

    return clean, True, 1
table_descriptions = {
    "final_table": (
        "Holy Table treatment por wave; cohorte con órdenes Pre y Post."
    ),
    "final_table_wave_flag": (
        "Holy Table treatment por wave y flag; excluye flags nulas/SIN_FLAG."
    ),
    "final_table_wave_restaurants": (
        "Holy Table treatment por wave filtrada solo a vertical_type restaurants."
    ),
    "final_table_wave_santiago": (
        "Holy Table treatment de Santiago por wave."
    ),
    "final_table_wave_regiones": (
        "Holy Table treatment de Regiones por wave."
    ),
    "final_table_wave_flag_santiago": (
        "Holy Table treatment de Santiago por wave y flag; sin flags nulas."
    ),
    "final_table_wave_flag_regiones": (
        "Holy Table treatment de Regiones por wave y flag; sin flags nulas."
    ),
    "final_table_wave_country_inputs": (
        "Inputs auditables: numeradores, denominadores, niveles, deltas, peso, DID y fórmula."
    ),
    "final_table_wave_country_contributions": (
        "Tabla compacta con únicamente los aportes estimados de cada wave a CL."
    ),
    "final_table_wave_country_restaurants_inputs": (
        "Inputs auditables del impacto nacional usando solo el universo restaurants."
    ),
    "final_table_wave_country_restaurants_contributions": (
        "Aportes estimados de cada wave al movimiento nacional de restaurants."
    ),
    "final_table_analysis_group": (
        "Deltas agregados de treatment, control y resto; control está dentro del resto."
    ),
    "wave_summary": "Niveles Pre/Post y deltas técnicos por wave y grupo.",
    "wave_flag_summary": "Niveles Pre/Post y deltas técnicos por wave, flag y grupo.",
    "flag_summary": "Niveles Pre/Post y deltas treatment agregados por flag.",
    "wave_diff": "Deltas treatment/control/resto y DID por wave.",
    "wave_flag_diff": "Deltas y DID por wave + flag.",
    "appendix_holy_wave": "Inputs numéricos que alimentan la Holy Table por wave.",
    "appendix_holy_wave_flag": "Inputs de la Holy Table wave + flag.",
    "appendix_awt_wave": "Distribución AWT Null/0/1–3/4+ por wave.",
    "appendix_awt_wave_flag": "Distribución AWT por wave + flag.",
    "appendix_awt_flag": "Distribución AWT treatment agregada por flag.",
    "appendix_awt_detailed_wave": "Distribución AWT Null/0/1/.../10/11+ por wave.",
    "appendix_awt_detailed_wave_flag": "Distribución AWT detallada por wave + flag.",
    "appendix_awt_detailed_flag": "Distribución AWT detallada agregada por flag.",
    "cohort_audit_summary": "Resumen de vendors configurados y comparables por wave.",
    "cohort_lost_cases": "Vendors treatment excluidos por faltar Pre o Post.",
    "cohort_audit_detail": "Detalle completo de actividad de la cohorte configurada.",
    "cohort_audit_observed": "Actividad observada Pre/Post por wave y grupo.",
    "city_merge_audit": "Vendors cuyas órdenes de varias ciudades fueron consolidadas.",
}

print("TABLAS QUE SE EXPORTARÁN")
print("-" * 80)
for table_name in tablas_a_exportar:
    description = table_descriptions.get(
        table_name,
        "Tabla auxiliar del análisis.",
    )
    print(f"• {table_name}: {description}")
print("-" * 80)


for nombre_hoja, tabla in tablas_a_exportar.items():

    if tabla is None:
        print(f"⚠️ {nombre_hoja}: tabla vacía")
        continue

    tabla_export, include_header, freeze_rows = (
        prepare_table_for_sheets(tabla)
    )

    try:
        worksheet = spreadsheet.worksheet(nombre_hoja)
        worksheet.clear()

    except WorksheetNotFound:
        worksheet = spreadsheet.add_worksheet(
            title=nombre_hoja,
            rows=max(len(tabla_export) + 10, 100),
            cols=max(len(tabla_export.columns) + 5, 20),
        )

    set_with_dataframe(
        worksheet,
        tabla_export,
        include_index=False,
        include_column_header=include_header,
        resize=True,
    )

    worksheet.freeze(rows=freeze_rows)

    print(
        f"✅ {nombre_hoja}: "
        f"{len(tabla_export):,} filas × "
        f"{len(tabla_export.columns):,} columnas | "
        f"{table_descriptions.get(nombre_hoja, 'Tabla auxiliar.')}"
    )

print("✅ Exportación terminada")


# plot

In [ ]:
import gspread
from google.colab import auth
import google.auth

auth.authenticate_user()

credentials, _ = google.auth.default(scopes=[
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
])

gc = gspread.authorize(credentials)

sheet_id = "1OLY68XweFeVXIUnT9CfImdf-sjAcmVJNLDcSKJMUJM0"
sheet_name = "df_4groups"

worksheet = gc.open_by_key(sheet_id).worksheet(sheet_name)
data = worksheet.get_all_values()
df_4groups = pd.DataFrame(data[1:], columns=data[0])

df_4groups.head()

In [ ]:
# df_4groups = df_4groups[df_4groups["month_label"] != "2026-08"]
# df_4groups = df_4groups[df_4groups["month_label"] != "2025-08"]
# df_4groups = df_4groups[df_4groups["month_label"] != "2025-09"]
# df_4groups = df_4groups[df_4groups["month_label"] != "2025-10"]
# df_4groups = df_4groups[df_4groups["month_label"] != "2025-11"]
# df_4groups = df_4groups[df_4groups["month_label"] != "2025-12"]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter


# ============================================================
# PREPARAR DATA
# ============================================================

df_plot = df_4groups.copy()

# Convertir métricas
df_plot["avg_ept_minutes"] = pd.to_numeric(
    df_plot["avg_ept_minutes"],
    errors="coerce"
)

# Convertir mes. Funciona con fechas o etiquetas tipo Aug-26
df_plot["month"] = pd.to_datetime(
    df_plot["month_label"],
    errors="coerce"
)

# Si month_label viene como "Aug-26", usa esto:
if df_plot["month"].isna().all():
    df_plot["month"] = pd.to_datetime(
        df_plot["month_label"],
        format="%b-%y",
        errors="coerce"
    )

# Clasificar los dos ejes del análisis
df_plot["fir_group"] = np.where(
    df_plot["group_name"].str.contains(
        r">=?\s*90|90%",
        case=False,
        regex=True,
        na=False
    ),
    "FIR ≥90%",
    "FIR <10%"
)

# Ojo: primero identificamos "Not reduced",
# porque también contiene la palabra "reduced"
df_plot["intervention_group"] = np.where(
    df_plot["group_name"].str.contains(
        r"not reduced|sin intervención",
        case=False,
        regex=True,
        na=False
    ),
    "Sin intervención",
    "Con intervención"
)

df_plot = (
    df_plot
    .dropna(subset=["month", "avg_ept_minutes"])
    .sort_values(["intervention_group", "fir_group", "month"])
)

# Opcional: excluir agosto de 2026 si todavía está incompleto
# df_plot = df_plot[df_plot["month"] != pd.Timestamp("2026-08-01")]


# ============================================================
# CONFIGURACIÓN VISUAL
# ============================================================

colors = {
    "Sin intervención": "#8A94A6",
    "Con intervención": "#E54B4B",
}

linestyles = {
    "FIR <10%": "--",
    "FIR ≥90%": "-",
}

markers = {
    "FIR <10%": "o",
    "FIR ≥90%": "D",
}

plot_order = [
    ("Sin intervención", "FIR <10%"),
    ("Sin intervención", "FIR ≥90%"),
    ("Con intervención", "FIR <10%"),
    ("Con intervención", "FIR ≥90%"),
]


# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(14, 7.5))

for intervention, fir in plot_order:

    part = df_plot[
        (df_plot["intervention_group"] == intervention)
        & (df_plot["fir_group"] == fir)
    ].sort_values("month")

    if part.empty:
        continue

    ax.plot(
        part["month"],
        part["avg_ept_minutes"],
        color=colors[intervention],
        linestyle=linestyles[fir],
        marker=markers[fir],
        linewidth=2.8,
        markersize=6.5,
        markeredgecolor="white",
        markeredgewidth=1.2,
        alpha=1 if intervention == "Con intervención" else 0.78,
        zorder=3,
    )

    # Etiqueta directa al final de cada línea
    last = part.iloc[-1]

    ax.annotate(
        (
            f"{intervention} · {fir}\n"
            f"{last['avg_ept_minutes']:.1f} min"
        ),
        xy=(last["month"], last["avg_ept_minutes"]),
        xytext=(10, 0),
        textcoords="offset points",
        color=colors[intervention],
        fontsize=9.5,
        fontweight="semibold",
        va="center",
        annotation_clip=False,
    )


# ============================================================
# TÍTULOS Y EJES
# ============================================================

fig.text(
    0.075,
    0.965,
    "Evolución mensual del EPT según uso de FIR e intervención",
    fontsize=20,
    fontweight="bold",
    color="#172033",
)

fig.text(
    0.075,
    0.925,
    "Color = intervención  ·  Tipo de línea = nivel de adopción de FIR",
    fontsize=11.5,
    color="#667085",
)

ax.set_xlabel("")
ax.set_ylabel(
    "EPT promedio (minutos)",
    fontsize=11,
    fontweight="semibold",
    color="#344054",
)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda value, _: f"{value:.1f}")
)

ax.grid(
    axis="y",
    color="#D0D5DD",
    linewidth=0.8,
    alpha=0.55,
)

ax.grid(axis="x", visible=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#D0D5DD")
ax.spines["bottom"].set_color("#D0D5DD")

ax.tick_params(
    axis="both",
    colors="#667085",
    labelsize=10,
)

# Dejar espacio para las etiquetas a la derecha
months = sorted(df_plot["month"].dropna().unique())

if months:
    ax.set_xlim(
        pd.Timestamp(months[0]) - pd.Timedelta(days=10),
        pd.Timestamp(months[-1]) + pd.Timedelta(days=65),
    )


# ============================================================
# LEYENDA CON SIGNIFICADO VISUAL
# ============================================================

legend_elements = [
    Line2D(
        [0], [0],
        color=colors["Con intervención"],
        linewidth=3,
        label="Con intervención",
    ),
    Line2D(
        [0], [0],
        color=colors["Sin intervención"],
        linewidth=3,
        label="Sin intervención",
    ),
    Line2D(
        [0], [0],
        color="#344054",
        linestyle="-",
        marker="D",
        linewidth=2.5,
        label="FIR ≥90%",
    ),
    Line2D(
        [0], [0],
        color="#344054",
        linestyle="--",
        marker="o",
        linewidth=2.5,
        label="FIR <10%",
    ),
]

ax.legend(
    handles=legend_elements,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.11),
    ncol=4,
    frameon=False,
    fontsize=10,
)

fig.text(
    0.075,
    0.025,
    "Cada punto representa el EPT promedio mensual del grupo.",
    fontsize=9.5,
    color="#98A2B3",
)

plt.tight_layout(rect=[0.06, 0.06, 0.99, 0.89])
plt.show()

In [ ]:
orders_col = "orders"

df_4groups[orders_col] = pd.to_numeric(df_4groups[orders_col], errors='coerce').fillna(0)

table_order_share = (
    df_4groups
    .groupby("group_name", as_index=False)[orders_col]
    .sum()
    .rename(columns={"group_name": "Grupo"})
)

table_order_share["Share de orders"] = (
    table_order_share[orders_col]
    / table_order_share[orders_col].sum()
).map(lambda x: f"{x:.1%}")

table_order_share = table_order_share[
    ["Grupo", "Share de orders"]
]

display(table_order_share)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import PercentFormatter


# ============================================================
# CONFIGURACIÓN
# ============================================================

ORDERS_COL = "orders"  # Cambiar si tu columna tiene otro nombre


# ============================================================
# PREPARAR DATA
# ============================================================

df_share = df_4groups.copy()

df_share["month"] = pd.to_datetime(
    df_share["month_label"],
    errors="coerce"
)

# Para etiquetas tipo "Aug-26"
if df_share["month"].isna().all():
    df_share["month"] = pd.to_datetime(
        df_share["month_label"],
        format="%b-%y",
        errors="coerce"
    )

df_share[ORDERS_COL] = pd.to_numeric(
    df_share[ORDERS_COL],
    errors="coerce"
).fillna(0)

# Clasificación FIR
df_share["fir_group"] = np.where(
    df_share["group_name"].str.contains(
        r"(?:>=|≥|>)\s*90|90%",
        case=False,
        regex=True,
        na=False
    ),
    "FIR ≥90%",
    "FIR <10%"
)

# Clasificación intervención
df_share["intervention_group"] = np.where(
    df_share["group_name"].str.contains(
        r"not reduced|sin intervención",
        case=False,
        regex=True,
        na=False
    ),
    "Sin intervención",
    "Con intervención"
)

df_share["plot_group"] = (
    df_share["intervention_group"]
    + " · "
    + df_share["fir_group"]
)

df_share = df_share.dropna(subset=["month"])


# ============================================================
# CALCULAR SHARE MENSUAL
# ============================================================

group_order = [
    "Sin intervención · FIR <10%",
    "Sin intervención · FIR ≥90%",
    "Con intervención · FIR <10%",
    "Con intervención · FIR ≥90%",
]

orders_monthly = (
    df_share
    .pivot_table(
        index="month",
        columns="plot_group",
        values=ORDERS_COL,
        aggfunc="sum",
        fill_value=0
    )
    .reindex(columns=group_order, fill_value=0)
    .sort_index()
)

share_monthly = orders_monthly.div(
    orders_monthly.sum(axis=1).replace(0, np.nan),
    axis=0
)

# Opcional: excluir agosto si sigue incompleto
# share_monthly = share_monthly[
#     share_monthly.index != pd.Timestamp("2026-08-01")
# ]


# ============================================================
# PLOT
# ============================================================

colors = {
    "Sin intervención · FIR <10%": "#98A2B3",
    "Sin intervención · FIR ≥90%": "#475467",
    "Con intervención · FIR <10%": "#F6A57A",
    "Con intervención · FIR ≥90%": "#E54B4B",
}

fig, ax = plt.subplots(figsize=(14, 7.5))

bottom = np.zeros(len(share_monthly))

for group in group_order:

    values = share_monthly[group].fillna(0).values

    bars = ax.bar(
        share_monthly.index,
        values,
        bottom=bottom,
        width=20,
        color=colors[group],
        edgecolor="white",
        linewidth=0.8,
        label=group,
    )

    # Mostrar etiqueta solo cuando el segmento sea suficientemente grande
    for bar, value, base in zip(bars, values, bottom):
        if value >= 0.04:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                base + value / 2,
                f"{value:.1%}",
                ha="center",
                va="center",
                fontsize=9,
                fontweight="semibold",
                color="white",
            )

    bottom += values


# ============================================================
# FORMATO
# ============================================================

fig.text(
    0.075,
    0.965,
    "Composición mensual de órdenes por uso de FIR e intervención",
    fontsize=20,
    fontweight="bold",
    color="#172033",
)

fig.text(
    0.075,
    0.925,
    "Cada barra representa el 100% de las órdenes incluidas en los cuatro grupos",
    fontsize=11.5,
    color="#667085",
)

ax.set_xlabel("")
ax.set_ylabel(
    "Share de órdenes",
    fontsize=11,
    fontweight="semibold",
    color="#344054",
)

ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(PercentFormatter(1))

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))

ax.grid(
    axis="y",
    color="#D0D5DD",
    linewidth=0.8,
    alpha=0.45,
)

ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#D0D5DD")
ax.spines["bottom"].set_color("#D0D5DD")

ax.tick_params(
    axis="both",
    colors="#667085",
    labelsize=10,
)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.10),
    ncol=2,
    frameon=False,
    fontsize=10,
)

fig.text(
    0.075,
    0.025,
    "Denominador: total de órdenes pertenecientes a los cuatro grupos en cada mes.",
    fontsize=9.5,
    color="#98A2B3",
)

plt.tight_layout(rect=[0.06, 0.06, 0.99, 0.88])
plt.show()

In [ ]:
metrics = [
    "avg_ept_minutes",
    "highpreptime_pct",
    "awt10_pct",
    "partner_performance_pct",
]

print(
    "Período:",
    df_4groups["month_start"].min(),
    "a",
    df_4groups["month_start"].max()
)

duplicates = (
    df_4groups
    .groupby(["month_start", "group_name"])
    .size()
    .loc[lambda x: x > 1]
)

print("\nDuplicados month-group:")
print(duplicates if not duplicates.empty else "No hay duplicados")

for metric in metrics:
    print(f"\n===== {metric} =====")
    display(
        df_4groups.pivot(
            index="month_start",
            columns="group_name",
            values=metric,
        )
    )

# Gráficos finales de impacto

Las curvas usan shares sobre todas las órdenes del grupo e incluyen `AWT null`,
`AWT0`, `AWT1` y la cola completa. Post es sólido y dominante; Pre queda
punteado y más opaco. `AWT10 causal` se mantiene separado de `awt_10min`, que es
solo el bucket de distribución entre 9 y 10 minutos.


In [ ]:
import math
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter


# ============================================================
# FORMATOS Y DETALLES DE CADA PANEL
# ============================================================

def fmt_plot_date(value):
    value = pd.to_datetime(value, errors="coerce")
    return "N/A" if pd.isna(value) else value.strftime("%d/%m")


def fmt_plot_number(value):
    value = pd.to_numeric(value, errors="coerce")
    return "N/A" if pd.isna(value) else f"{value:,.0f}"


def fmt_plot_pct(value):
    value = pd.to_numeric(value, errors="coerce")
    return "N/A" if pd.isna(value) else f"{value:.2f}%"


def natural_wave_key(value):
    match = re.search(r"(\d+)", str(value))
    return (
        int(match.group(1)) if match else float("inf"),
        str(value),
    )


def detail_line_from_holy_row(row, include_wave=False):
    prefix = f"{row['wave']} · " if include_wave else ""

    return (
        f"{prefix}Pre {fmt_plot_date(row['Pre inicio'])}–"
        f"{fmt_plot_date(row['Pre fin'])} | "
        f"Post {fmt_plot_date(row['Post inicio'])}–"
        f"{fmt_plot_date(row['Post fin'])}\n"
        f"Vendors: {fmt_plot_number(row['Vendors'])} | "
        f"Franquicias: {fmt_plot_number(row['Franchises'])} | "
        f"Orders Pre: {fmt_plot_number(row['Pre #Orders'])} "
        f"({fmt_plot_pct(row['% Orders CL Pre'])} CL) | "
        f"Orders Post: {fmt_plot_number(row['Post #Orders'])} "
        f"({fmt_plot_pct(row['% Orders CL Post'])} CL)"
    )


# Detalles por wave.
plot_info_wave = {
    str(row["wave"]): detail_line_from_holy_row(row)
    for _, row in appendix_holy_wave.iterrows()
}


# Detalles por wave + flag.
appendix_awt_wave_flag_plot = appendix_awt_wave_flag.copy()
appendix_awt_wave_flag_plot["panel"] = (
    appendix_awt_wave_flag_plot["wave"].astype(str)
    + "\n"
    + appendix_awt_wave_flag_plot["flag"].astype(str)
)

appendix_awt_detailed_wave_flag_plot = (
    appendix_awt_detailed_wave_flag.copy()
)
appendix_awt_detailed_wave_flag_plot["panel"] = (
    appendix_awt_detailed_wave_flag_plot["wave"].astype(str)
    + "\n"
    + appendix_awt_detailed_wave_flag_plot["flag"].astype(str)
)

plot_info_wave_flag = {}

for _, row in appendix_holy_wave_flag.iterrows():
    panel = f"{row['wave']}\n{row['flag']}"
    plot_info_wave_flag[panel] = detail_line_from_holy_row(row)


# Detalles por flag. Cada wave conserva su período y volumen propio.
plot_info_flag = {}

for flag, group in appendix_holy_wave_flag.groupby(
    "flag",
    dropna=False,
):
    group = group.copy()
    group["_sort_date"] = pd.to_datetime(
        group["Post inicio"],
        errors="coerce",
    )
    group = group.sort_values("_sort_date")

    plot_info_flag[str(flag)] = "\n".join(
        detail_line_from_holy_row(row, include_wave=True)
        for _, row in group.iterrows()
    )


# ============================================================
# GRÁFICO ESTÁNDAR
# ============================================================

def plot_awt_distribution_facets(
    distribution_df,
    facet_col,
    title,
    plot_info,
    ncols=1,
    curve_buckets=None,
    bucket_labels=None,
):
    """
    Un panel por wave, wave+flag o flag.
    Siempre muestra:
    - los buckets solicitados para la vista;
    - cifras Pre y Post;
    - fechas, orders, vendors, franquicias y % Orders CL.
    """

    df = distribution_df.copy()
    curve_buckets = list(
        AWT_CURVE_BUCKETS
        if curve_buckets is None
        else curve_buckets
    )
    bucket_labels = (
        AWT_BUCKET_LABELS
        if bucket_labels is None
        else bucket_labels
    )

    required_columns = {
        facet_col,
        "bucket",
        "bucket_label",
        "bucket_order",
        "pre_share_pct",
        "post_share_pct",
    }

    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise KeyError(
            "Faltan columnas para el gráfico AWT: "
            + ", ".join(sorted(missing_columns))
        )

    df = df[df["bucket"].isin(curve_buckets)].copy()

    for column in [
        "bucket_order",
        "pre_share_pct",
        "post_share_pct",
    ]:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )

    df = df.dropna(subset=[facet_col, "bucket_order"])
    df[facet_col] = df[facet_col].astype(str)

    if facet_col == "wave":
        facets = sorted(
            df[facet_col].unique(),
            key=natural_wave_key,
        )
    elif facet_col == "panel":
        facets = sorted(
            df[facet_col].unique(),
            key=lambda value: (
                natural_wave_key(str(value).split("\n", 1)[0]),
                str(value),
            ),
        )
    else:
        facets = sorted(df[facet_col].unique())

    if not facets:
        print(f"No hay datos para graficar por {facet_col}.")
        return

    ncols = min(ncols, len(facets))
    nrows = math.ceil(len(facets) / ncols)

    detail_lines = [
        len(plot_info.get(str(facet), "").splitlines())
        for facet in facets
    ]

    max_detail_lines = max(detail_lines + [1])
    panel_height = 6.5 + 0.42 * max_detail_lines

    panel_width = max(11.5, 1.05 * len(curve_buckets))

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(panel_width * ncols, panel_height * nrows),
        sharey=True,
        squeeze=False,
    )

    axes = axes.ravel()

    all_values = pd.concat(
        [df["pre_share_pct"], df["post_share_pct"]],
        ignore_index=True,
    ).replace([np.inf, -np.inf], np.nan)

    global_max = all_values.max()
    if pd.isna(global_max) or global_max <= 0:
        global_max = 1

    # Espacio para ambas cifras, incluso en el mayor punto.
    y_upper = global_max * 1.35

    pre_handle = None
    post_handle = None

    for ax, facet in zip(axes, facets):

        part = (
            df[df[facet_col].eq(str(facet))]
            .sort_values("bucket_order")
            .set_index("bucket")
            .reindex(curve_buckets)
            .reset_index()
        )

        x = np.arange(len(part))

        pre_handle, = ax.plot(
            x,
            part["pre_share_pct"],
            color="#94A3B8",
            linewidth=2.4,
            linestyle="--",
            marker="o",
            markersize=7,
            markeredgecolor="white",
            markeredgewidth=1,
            label="Pre",
            zorder=2,
        )

        post_handle, = ax.plot(
            x,
            part["post_share_pct"],
            color="#2563EB",
            linewidth=3.2,
            linestyle="-",
            marker="o",
            markersize=8,
            markeredgecolor="white",
            markeredgewidth=1,
            label="Post",
            zorder=3,
        )

        # Cifras de ambas líneas. Gris = Pre; azul = Post.
        for x_value, pre_value, post_value in zip(
            x,
            part["pre_share_pct"],
            part["post_share_pct"],
        ):
            close_values = (
                pd.notna(pre_value)
                and pd.notna(post_value)
                and abs(pre_value - post_value) < y_upper * 0.055
            )

            if pd.notna(pre_value):
                pre_offset = -16 if close_values else 10
                ax.annotate(
                    f"{pre_value:.1f}%",
                    xy=(x_value, pre_value),
                    xytext=(-7, pre_offset),
                    textcoords="offset points",
                    ha="right",
                    va="top" if pre_offset < 0 else "bottom",
                    fontsize=9,
                    fontweight="semibold",
                    color="#64748B",
                    annotation_clip=False,
                )

            if pd.notna(post_value):
                ax.annotate(
                    f"{post_value:.1f}%",
                    xy=(x_value, post_value),
                    xytext=(7, 11),
                    textcoords="offset points",
                    ha="left",
                    va="bottom",
                    fontsize=9,
                    fontweight="bold",
                    color="#2563EB",
                    annotation_clip=False,
                )

        facet_title = str(facet).replace("\n", " · ")
        details = plot_info.get(str(facet), "")

        ax.set_title(
            facet_title,
            fontsize=13,
            fontweight="bold",
            color="#172033",
            pad=76 + max_detail_lines * 10,
        )

        ax.text(
            0.5,
            1.025,
            details,
            transform=ax.transAxes,
            ha="center",
            va="bottom",
            fontsize=7.5,
            color="#667085",
            linespacing=1.35,
            wrap=True,
        )

        ax.set_xticks(x)
        ax.set_xticklabels(
            [bucket_labels[bucket] for bucket in curve_buckets],
            fontsize=10,
            fontweight="semibold",
        )

        ax.yaxis.set_major_formatter(
            FuncFormatter(lambda value, _: f"{value:.1f}%")
        )

        ax.set_ylim(0, y_upper)
        ax.set_xlim(-0.45, len(part) - 0.55)

        ax.grid(
            axis="y",
            color="#D0D5DD",
            linewidth=0.8,
            alpha=0.45,
        )

        ax.set_axisbelow(True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color("#D0D5DD")
        ax.spines["bottom"].set_color("#D0D5DD")
        ax.tick_params(axis="both", colors="#667085")

    for ax in axes[len(facets):]:
        ax.set_visible(False)

    fig.suptitle(
        title,
        fontsize=18,
        fontweight="bold",
        color="#172033",
        y=0.998,
    )

    fig.supxlabel(
        "Bucket de AWT",
        fontsize=11,
        color="#475467",
        y=0.01,
    )

    fig.supylabel(
        "Share sobre órdenes totales",
        fontsize=11,
        color="#344054",
        x=0.01,
    )

    fig.legend(
        handles=[post_handle, pre_handle],
        labels=["Post", "Pre"],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.968),
        ncol=2,
        frameon=False,
        fontsize=10,
    )

    plt.tight_layout(
        rect=[0.035, 0.04, 0.99, 0.90],
        h_pad=8,
        w_pad=3,
    )

    plt.show()


# ============================================================
# GRÁFICOS FINALES
# ============================================================

plot_awt_distribution_facets(
    distribution_df=appendix_awt_wave,
    facet_col="wave",
    title="Distribución AWT Pre vs Post — por wave",
    plot_info=plot_info_wave,
    ncols=1,
)

plot_awt_distribution_facets(
    distribution_df=appendix_awt_wave_flag_plot,
    facet_col="panel",
    title="Distribución AWT Pre vs Post — por wave y flag",
    plot_info=plot_info_wave_flag,
    ncols=1,
)

plot_awt_distribution_facets(
    distribution_df=appendix_awt_flag,
    facet_col="flag",
    title="Distribución AWT Pre vs Post — por flag",
    plot_info=plot_info_flag,
    ncols=1,
)


# Vista adicional detallada: Null, 0, 1, ..., 10, 11+.
plot_awt_distribution_facets(
    distribution_df=appendix_awt_detailed_wave,
    facet_col="wave",
    title="Distribución AWT detallada Pre vs Post — por wave",
    plot_info=plot_info_wave,
    ncols=1,
    curve_buckets=AWT_DETAILED_BUCKETS,
    bucket_labels=AWT_DETAILED_BUCKET_LABELS,
)

plot_awt_distribution_facets(
    distribution_df=appendix_awt_detailed_wave_flag_plot,
    facet_col="panel",
    title="Distribución AWT detallada Pre vs Post — por wave y flag",
    plot_info=plot_info_wave_flag,
    ncols=1,
    curve_buckets=AWT_DETAILED_BUCKETS,
    bucket_labels=AWT_DETAILED_BUCKET_LABELS,
)

plot_awt_distribution_facets(
    distribution_df=appendix_awt_detailed_flag,
    facet_col="flag",
    title="Distribución AWT detallada Pre vs Post — por flag",
    plot_info=plot_info_flag,
    ncols=1,
    curve_buckets=AWT_DETAILED_BUCKETS,
    bucket_labels=AWT_DETAILED_BUCKET_LABELS,
)


# AWT0 por batches — última wave WeekX

Carga `Sheet1` del archivo de métricas, identifica la última wave `WeekX` excluyendo `Mini`, y grafica los vendors por tramos de share AWT0. Azul corresponde a `AWT0 reduction`; gris, a los demás criterios. `Sin dato` mantiene visibles las pérdidas del cruce.


In [ ]:
# ============================================================
# AWT0 SHARE POR BATCH — ÚLTIMA WAVE WEEKX
# ============================================================

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display


SHEET_ID_AWT0 = "1bq1G_luDEW4g-K4ePYBmMu68e021JIyPWznrqPLGewo"
SHEET_TAB_AWT0 = "Sheet1"


# ------------------------------------------------------------
# 1. Cargar Sheet1
# ------------------------------------------------------------

# Normalmente gc ya existe porque se creó al cargar reduction.
# El fallback permite ejecutar esta sección de forma independiente.
if "gc" not in globals():
    import gspread
    from google.colab import auth
    import google.auth

    auth.authenticate_user()

    credentials, _ = google.auth.default(scopes=[
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive",
    ])

    gc = gspread.authorize(credentials)


worksheet_awt0 = (
    gc.open_by_key(SHEET_ID_AWT0)
    .worksheet(SHEET_TAB_AWT0)
)

values_awt0 = worksheet_awt0.get_all_values()

if len(values_awt0) < 2:
    raise ValueError(
        "Sheet1 está vacío o no contiene filas de datos."
    )

headers_awt0 = [
    str(column).strip()
    for column in values_awt0[0]
]

if len(headers_awt0) != len(set(headers_awt0)):
    duplicated_headers = pd.Series(headers_awt0)[
        pd.Series(headers_awt0).duplicated(keep=False)
    ].unique().tolist()

    raise ValueError(
        "Sheet1 tiene encabezados duplicados: "
        f"{duplicated_headers}"
    )

df_awt0_source = pd.DataFrame(
    values_awt0[1:],
    columns=headers_awt0,
)

print(
    f"Sheet cargado: {SHEET_TAB_AWT0} | "
    f"{len(df_awt0_source):,} filas"
)


# ------------------------------------------------------------
# 2. Helpers de limpieza
# ------------------------------------------------------------

def normalize_column_key(value):
    value = (
        str(value)
        .strip()
        .lower()
        .replace("%", " pct ")
    )

    return re.sub(
        r"_+",
        "_",
        re.sub(r"[^a-z0-9]+", "_", value),
    ).strip("_")


def normalize_vendor_code(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


# ------------------------------------------------------------
# 3. Identificar automáticamente la última WeekX
# ------------------------------------------------------------

required_reduction_columns = {
    "wave",
    "flag",
    "vendor_code",
}

missing_reduction_columns = (
    required_reduction_columns - set(reduction.columns)
)

if missing_reduction_columns:
    raise KeyError(
        "Faltan columnas en reduction: "
        f"{sorted(missing_reduction_columns)}"
    )

week_source = reduction.copy()

week_source["wave"] = (
    week_source["wave"]
    .astype("string")
    .str.strip()
)

wave_lower = week_source["wave"].str.lower()

week_source = week_source.loc[
    wave_lower.str.contains("week", na=False)
    & ~wave_lower.str.contains("mini", na=False)
].copy()

week_source["vendor_code"] = normalize_vendor_code(
    week_source["vendor_code"]
)

week_source["flag"] = (
    week_source["flag"]
    .astype("string")
    .str.strip()
    .fillna("SIN_FLAG")
)

week_source = week_source.loc[
    week_source["vendor_code"].notna()
    & week_source["vendor_code"].ne("")
].copy()

if week_source.empty:
    raise ValueError(
        "No existen waves Week válidas después de excluir Mini."
    )

week_source["_week_number"] = pd.to_numeric(
    week_source["wave"].str.extract(
        r"(?i)week\D*(\d+)",
        expand=False,
    ),
    errors="coerce",
)

date_column = next(
    (
        column
        for column in ["executed_at", "start_date"]
        if column in week_source.columns
    ),
    None,
)

if date_column is not None:
    week_source["_wave_date"] = pd.to_datetime(
        week_source[date_column],
        errors="coerce",
    )
else:
    week_source["_wave_date"] = pd.NaT

wave_candidates = (
    week_source
    .groupby("wave", as_index=False)
    .agg(
        wave_date=("_wave_date", "max"),
        week_number=("_week_number", "max"),
    )
    .sort_values(
        ["wave_date", "week_number", "wave"],
        na_position="first",
    )
)

latest_week = wave_candidates.iloc[-1]["wave"]

latest_week_rows = week_source.loc[
    week_source["wave"].eq(latest_week)
].copy()

latest_week_rows["_reduced_by_awt0"] = (
    latest_week_rows["flag"]
    .str.lower()
    .str.replace(r"\s+", "", regex=True)
    .str.contains("awt0", na=False)
    & latest_week_rows["flag"]
      .str.lower()
      .str.contains("reduction", na=False)
)

# Una sola fila por vendor. Si tiene varios registros dentro de la
# misma wave, se considera AWT0 si al menos uno tiene ese criterio.
latest_week_vendors = (
    latest_week_rows
    .groupby("vendor_code", as_index=False)
    .agg(
        reducido_por_awt0=("_reduced_by_awt0", "max"),
        criterio=(
            "flag",
            lambda values: " | ".join(
                sorted(set(values.dropna().astype(str)))
            ),
        ),
    )
)

latest_week_vendors["tipo_reduccion"] = np.where(
    latest_week_vendors["reducido_por_awt0"],
    "Reducido por AWT0",
    "No reducido por AWT0",
)


# ------------------------------------------------------------
# 4. Detectar las columnas del Sheet
# ------------------------------------------------------------

columns_by_key = {
    normalize_column_key(column): column
    for column in df_awt0_source.columns
}

vendor_candidates = [
    "vendor_code",
    "vendor_id",
    "partner_id",
    "partner_code",
    "vendor",
]

sheet_vendor_column = next(
    (
        columns_by_key[candidate]
        for candidate in vendor_candidates
        if candidate in columns_by_key
    ),
    None,
)

if sheet_vendor_column is None:
    raise KeyError(
        "No pude identificar la columna vendor de Sheet1. "
        f"Columnas: {list(df_awt0_source.columns)}"
    )

awt0_candidates = [
    "awt_0",
    "awt_0min_1w_pct",
    "awt_0min_1w",
    "awt0_1w_pct",
    "awt0_1w",
    "awt_0min_pct",
    "awt0_pct",
    "awt_0_share",
    "awt0_share",
    "awt_0min",
    "awt0",
]

sheet_awt0_column = next(
    (
        columns_by_key[candidate]
        for candidate in awt0_candidates
        if candidate in columns_by_key
    ),
    None,
)

# Fallback para nombres similares no incluidos en la lista.
if sheet_awt0_column is None:
    possible_awt0_columns = []

    for column in df_awt0_source.columns:
        key = normalize_column_key(column)

        is_awt0_column = (
            "awt" in key
            and re.search(r"(^|_)0($|_|min)", key)
            and "orders" not in key
            and "count" not in key
        )

        if is_awt0_column:
            score = (
                4 * int("1w" in key)
                + 3 * int("pct" in key)
                + 3 * int("share" in key)
                + 2 * int("ratio" in key)
                + int("min" in key)
            )

            possible_awt0_columns.append(
                (score, column)
            )

    if possible_awt0_columns:
        sheet_awt0_column = sorted(
            possible_awt0_columns,
            reverse=True,
        )[0][1]

if sheet_awt0_column is None:
    awt_columns = [
        column
        for column in df_awt0_source.columns
        if "awt" in normalize_column_key(column)
    ]

    raise KeyError(
        "No pude identificar la columna share AWT0. "
        f"Columnas relacionadas con AWT: {awt_columns}"
    )

print(f"Última wave seleccionada: {latest_week}")
print(f"Columna vendor: {sheet_vendor_column}")
print(f"Columna share AWT0: {sheet_awt0_column}")


# ------------------------------------------------------------
# 5. Normalizar share a escala 0–100
# ------------------------------------------------------------

df_awt0 = df_awt0_source[
    [sheet_vendor_column, sheet_awt0_column]
].copy()

df_awt0.columns = [
    "vendor_code",
    "awt0_share_raw",
]

df_awt0["vendor_code"] = normalize_vendor_code(
    df_awt0["vendor_code"]
)

share_text = (
    df_awt0["awt0_share_raw"]
    .astype("string")
    .str.strip()
)

has_pct_symbol = share_text.str.contains(
    "%",
    regex=False,
    na=False,
)

share_number = pd.to_numeric(
    share_text
    .str.replace("%", "", regex=False)
    .str.replace(",", ".", regex=False),
    errors="coerce",
)

values_without_pct = share_number.loc[
    ~has_pct_symbol & share_number.notna()
]

values_are_fraction = (
    not values_without_pct.empty
    and values_without_pct.quantile(0.95) <= 1.000001
)

df_awt0["awt0_share_pct"] = share_number

if values_are_fraction:
    df_awt0.loc[
        ~has_pct_symbol,
        "awt0_share_pct",
    ] *= 100

invalid_share = (
    df_awt0["awt0_share_pct"].notna()
    & ~df_awt0["awt0_share_pct"].between(0, 100)
)

invalid_share_count = int(invalid_share.sum())

if invalid_share_count:
    print(
        f"Advertencia: {invalid_share_count:,} shares fuera de "
        "0–100 se tratarán como Sin dato."
    )

    df_awt0.loc[
        invalid_share,
        "awt0_share_pct",
    ] = np.nan

df_awt0 = df_awt0.loc[
    df_awt0["vendor_code"].notna()
    & df_awt0["vendor_code"].ne("")
].copy()

duplicated_vendor_rows = int(
    df_awt0["vendor_code"]
    .duplicated(keep=False)
    .sum()
)

if duplicated_vendor_rows:
    print(
        "Advertencia: "
        f"{duplicated_vendor_rows:,} filas pertenecen a vendors "
        "duplicados en Sheet1; se usará la mediana por vendor."
    )

df_awt0_vendor = (
    df_awt0
    .groupby("vendor_code", as_index=False)
    .agg(
        awt0_share_pct=("awt0_share_pct", "median")
    )
)


# ------------------------------------------------------------
# 6. Cruzar y construir batches de 10 pp
# ------------------------------------------------------------

latest_wave_awt0 = latest_week_vendors.merge(
    df_awt0_vendor,
    on="vendor_code",
    how="left",
    validate="one_to_one",
)

bucket_edges = [
    0, 10, 20, 30, 40, 50,
    60, 70, 80, 90, 100.000001,
]

bucket_labels = [
    "[0–10%)",
    "[10–20%)",
    "[20–30%)",
    "[30–40%)",
    "[40–50%)",
    "[50–60%)",
    "[60–70%)",
    "[70–80%)",
    "[80–90%)",
    "[90–100%]",
]

latest_wave_awt0["batch_awt0"] = pd.cut(
    latest_wave_awt0["awt0_share_pct"],
    bins=bucket_edges,
    labels=bucket_labels,
    right=False,
    include_lowest=True,
).astype("string")

latest_wave_awt0["batch_awt0"] = (
    latest_wave_awt0["batch_awt0"]
    .fillna("Sin dato")
)

missing_share_vendors = int(
    latest_wave_awt0["awt0_share_pct"]
    .isna()
    .sum()
)

bucket_order = bucket_labels.copy()

if missing_share_vendors:
    bucket_order.append("Sin dato")

latest_wave_awt0["batch_awt0"] = pd.Categorical(
    latest_wave_awt0["batch_awt0"],
    categories=bucket_order,
    ordered=True,
)

status_order = [
    "No reducido por AWT0",
    "Reducido por AWT0",
]

awt0_batch_summary = (
    latest_wave_awt0
    .groupby(
        ["batch_awt0", "tipo_reduccion"],
        observed=False,
    )
    .agg(vendors=("vendor_code", "nunique"))
    .reset_index()
)

awt0_batch_table = (
    awt0_batch_summary
    .pivot_table(
        index="batch_awt0",
        columns="tipo_reduccion",
        values="vendors",
        aggfunc="sum",
        fill_value=0,
        observed=False,
    )
    .reindex(index=bucket_order, fill_value=0)
    .reindex(columns=status_order, fill_value=0)
)

awt0_batch_table["Total"] = (
    awt0_batch_table.sum(axis=1)
)

total_wave_vendors = int(
    latest_wave_awt0["vendor_code"].nunique()
)

total_awt0_reduced = int(
    latest_wave_awt0.loc[
        latest_wave_awt0["reducido_por_awt0"],
        "vendor_code",
    ].nunique()
)

print(
    "\nResumen del cruce"
    f"\n• Wave: {latest_week}"
    f"\n• Vendors únicos: {total_wave_vendors:,}"
    f"\n• Reducidos por AWT0: {total_awt0_reduced:,}"
    f"\n• Otros criterios: "
    f"{total_wave_vendors - total_awt0_reduced:,}"
    f"\n• Sin share válido: {missing_share_vendors:,}"
)


# ------------------------------------------------------------
# 7. Gráfico de barras apiladas
# ------------------------------------------------------------

sns.set_theme(style="whitegrid", context="talk")

fig, ax = plt.subplots(
    figsize=(18, 9),
    dpi=120,
)

plot_columns = status_order

awt0_batch_table[plot_columns].plot(
    kind="bar",
    stacked=True,
    width=0.74,
    color=[
        "#98A2B3",
        "#2563EB",
    ],
    edgecolor="white",
    linewidth=0.9,
    ax=ax,
)

# Números dentro de cada parte de la barra.
for container in ax.containers:
    labels = [
        f"{int(bar.get_height()):,}"
        if bar.get_height() > 0
        else ""
        for bar in container
    ]

    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=13,
        fontweight="bold",
        color="white",
    )

# Total sobre cada barra.
totals = awt0_batch_table["Total"]
max_total = max(float(totals.max()), 1.0)

for position, total in enumerate(totals):
    if total > 0:
        ax.text(
            position,
            total + max_total * 0.025,
            f"{int(total):,}",
            ha="center",
            va="bottom",
            fontsize=14,
            fontweight="bold",
            color="#101828",
        )

ax.set_ylim(0, max_total * 1.18)

ax.set_title(
    f"Vendors de {latest_week} según share AWT0",
    fontsize=23,
    fontweight="bold",
    color="#101828",
    pad=34,
)

ax.text(
    0,
    1.02,
    (
        f"{total_wave_vendors:,} vendors únicos | "
        f"{total_awt0_reduced:,} reducidos por AWT0 | "
        f"{missing_share_vendors:,} sin dato"
    ),
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=15,
    color="#667085",
)

ax.set_xlabel(
    "Share de órdenes con AWT = 0",
    fontsize=17,
    fontweight="semibold",
    color="#344054",
    labelpad=15,
)

ax.set_ylabel(
    "Vendors únicos",
    fontsize=17,
    fontweight="semibold",
    color="#344054",
)

ax.tick_params(
    axis="x",
    labelrotation=35,
    labelsize=13,
    colors="#344054",
)

ax.tick_params(
    axis="y",
    labelsize=13,
    colors="#475467",
)

ax.grid(
    axis="y",
    color="#D0D5DD",
    linewidth=0.8,
    alpha=0.65,
)

ax.grid(axis="x", visible=False)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color("#D0D5DD")

legend = ax.legend(
    title="Criterio aplicado",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    fontsize=13,
)

legend.get_title().set_fontsize(14)
legend.get_title().set_fontweight("bold")

plt.tight_layout(
    rect=[0, 0, 0.84, 1]
)

plt.show()


# Tabla exacta detrás del gráfico.
display(
    awt0_batch_table
    .astype(int)
    .style
    .format("{:,.0f}")
    .set_caption(
        f"Vendors por tramo de AWT0 — {latest_week}"
    )
)


# Cobertura nacional de reductions según AWT0

La primera vista toma todos los vendors de `Sheet1` y muestra qué parte del universo CL cubre la última `WeekX`. La segunda pondera por órdenes:

- Si `Sheet1` trae órdenes por vendor, muestra la distribución nacional completa y sus tres componentes.
- Si no las trae, usa el Post de `df_pre_post` para mostrar solamente el porcentaje de órdenes CL alcanzado por reductions. El complemento nacional no se asigna a buckets porque `resto_de_chile` viene agregado.


In [ ]:
# ============================================================
# COBERTURA NACIONAL DE REDUCTIONS SEGÚN SHARE AWT0
#
# Requiere ejecutar primero la celda anterior, que crea:
#   - df_awt0_source
#   - df_awt0_vendor
#   - latest_week
#   - latest_week_rows
#   - sheet_vendor_column
#
# Produce:
#   1. Vendors CL por bucket AWT0, apilados por cobertura.
#   2. Orders CL por bucket AWT0, apiladas por cobertura.
#
# Para orders existen dos modos:
#   - Completo: si Sheet1 trae orders por vendor.
#   - Cobertura: si solo df_pre_post trae orders de treatment.
# ============================================================

from matplotlib.ticker import FuncFormatter


required_previous_objects = [
    "df_awt0_source",
    "df_awt0_vendor",
    "latest_week",
    "latest_week_rows",
    "sheet_vendor_column",
]

missing_previous_objects = [
    name
    for name in required_previous_objects
    if name not in globals()
]

if missing_previous_objects:
    raise RuntimeError(
        "Ejecuta primero la celda anterior de carga AWT0. "
        f"Faltan objetos: {missing_previous_objects}"
    )


# ============================================================
# CONFIGURACIÓN VISUAL Y CATEGORÍAS
# ============================================================

CL_BUCKET_EDGES = [
    0, 10, 20, 30, 40, 50,
    60, 70, 80, 90, 100.000001,
]

CL_BUCKET_LABELS = [
    "[0–10%)",
    "[10–20%)",
    "[20–30%)",
    "[30–40%)",
    "[40–50%)",
    "[50–60%)",
    "[60–70%)",
    "[70–80%)",
    "[80–90%)",
    "[90–100%]",
]

COVERAGE_ORDER = [
    "No incluido en reductions",
    "Otra reduction",
    "AWT0 reduction",
]

COVERAGE_COLORS = {
    "No incluido en reductions": "#D0D5DD",
    "Otra reduction": "#7F56D9",
    "AWT0 reduction": "#2563EB",
}


def parse_numeric_series(series):
    """Convierte números con %, coma decimal o separador de miles."""

    text = (
        series
        .astype("string")
        .str.strip()
        .str.replace("%", "", regex=False)
        .str.replace(" ", "", regex=False)
    )

    direct = pd.to_numeric(text, errors="coerce")

    # Segundo intento para números tipo 1,234.56.
    comma_thousands = pd.to_numeric(
        text.str.replace(",", "", regex=False),
        errors="coerce",
    )

    # Tercer intento para decimales con coma tipo 0,65.
    comma_decimal = pd.to_numeric(
        text.str.replace(",", ".", regex=False),
        errors="coerce",
    )

    return direct.fillna(comma_thousands).fillna(comma_decimal)


def plot_coverage_stacked(
    plot_table,
    series_columns,
    title,
    subtitle,
    ylabel,
    percentage=False,
):
    """Gráfico apilado con cifras internas y total por bucket."""

    if plot_table.empty or plot_table[series_columns].sum().sum() == 0:
        print(f"No hay datos para graficar: {title}")
        return

    fig, ax = plt.subplots(
        figsize=(18, 9),
        dpi=120,
    )

    plot_table[series_columns].plot(
        kind="bar",
        stacked=True,
        width=0.74,
        color=[
            COVERAGE_COLORS[column]
            for column in series_columns
        ],
        edgecolor="white",
        linewidth=0.9,
        ax=ax,
    )

    for container in ax.containers:
        labels = []

        for bar in container:
            value = bar.get_height()

            if value <= 0:
                labels.append("")
            elif percentage:
                labels.append(f"{value:.2f}%")
            else:
                labels.append(f"{int(round(value)):,}")

        ax.bar_label(
            container,
            labels=labels,
            label_type="center",
            fontsize=12,
            fontweight="bold",
            color="white",
        )

    totals = plot_table[series_columns].sum(axis=1)
    max_total = max(float(totals.max()), 1.0)

    for position, total in enumerate(totals):
        if total <= 0:
            continue

        label = (
            f"{total:.2f}%"
            if percentage
            else f"{int(round(total)):,}"
        )

        ax.text(
            position,
            total + max_total * 0.025,
            label,
            ha="center",
            va="bottom",
            fontsize=14,
            fontweight="bold",
            color="#101828",
        )

    ax.set_ylim(0, max_total * 1.20)

    ax.set_title(
        title,
        fontsize=23,
        fontweight="bold",
        color="#101828",
        pad=36,
    )

    ax.text(
        0,
        1.02,
        subtitle,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=14,
        color="#667085",
        wrap=True,
    )

    ax.set_xlabel(
        "Share de órdenes con AWT = 0",
        fontsize=17,
        fontweight="semibold",
        color="#344054",
        labelpad=15,
    )

    ax.set_ylabel(
        ylabel,
        fontsize=17,
        fontweight="semibold",
        color="#344054",
    )

    if percentage:
        ax.yaxis.set_major_formatter(
            FuncFormatter(lambda value, _: f"{value:.1f}%")
        )

    ax.tick_params(
        axis="x",
        labelrotation=35,
        labelsize=13,
        colors="#344054",
    )

    ax.tick_params(
        axis="y",
        labelsize=13,
        colors="#475467",
    )

    ax.grid(
        axis="y",
        color="#D0D5DD",
        linewidth=0.8,
        alpha=0.65,
    )

    ax.grid(axis="x", visible=False)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_color("#D0D5DD")

    legend = ax.legend(
        title="Cobertura de reductions",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
        fontsize=13,
    )

    legend.get_title().set_fontsize(14)
    legend.get_title().set_fontweight("bold")

    plt.tight_layout(rect=[0, 0, 0.82, 1])
    plt.show()


# ============================================================
# 1. UNIVERSO NACIONAL DE VENDORS + COBERTURA DE LA ÚLTIMA WEEK
# ============================================================

scope_reductions = latest_week_rows.copy()

scope_reductions["vendor_code"] = normalize_vendor_code(
    scope_reductions["vendor_code"]
)

scope_reductions["_is_awt0_reduction"] = (
    scope_reductions["flag"]
    .astype("string")
    .str.lower()
    .str.replace(r"\s+", "", regex=True)
    .str.contains("awt0", na=False)
    & scope_reductions["flag"]
      .astype("string")
      .str.lower()
      .str.contains("reduction", na=False)
)

scope_vendors = (
    scope_reductions
    .groupby("vendor_code", as_index=False)
    .agg(
        is_awt0_reduction=("_is_awt0_reduction", "max"),
    )
)

scope_vendors["is_in_reductions"] = True

# Outer conserva tanto el universo de Sheet1 como los vendors de la
# wave que eventualmente no hayan encontrado match en el Sheet.
national_awt0 = df_awt0_vendor.merge(
    scope_vendors,
    on="vendor_code",
    how="outer",
    validate="one_to_one",
)

national_awt0["is_in_reductions"] = (
    national_awt0["is_in_reductions"]
    .astype("boolean")
    .fillna(False)
    .astype(bool)
)

national_awt0["is_awt0_reduction"] = (
    national_awt0["is_awt0_reduction"]
    .astype("boolean")
    .fillna(False)
    .astype(bool)
)

national_awt0["coverage_status"] = np.select(
    [
        national_awt0["is_awt0_reduction"],
        national_awt0["is_in_reductions"],
    ],
    [
        "AWT0 reduction",
        "Otra reduction",
    ],
    default="No incluido en reductions",
)

national_awt0["awt0_bucket"] = pd.cut(
    national_awt0["awt0_share_pct"],
    bins=CL_BUCKET_EDGES,
    labels=CL_BUCKET_LABELS,
    right=False,
    include_lowest=True,
).astype("string")

national_awt0["awt0_bucket"] = (
    national_awt0["awt0_bucket"]
    .fillna("Sin dato")
)

national_bucket_order = CL_BUCKET_LABELS.copy()

if national_awt0["awt0_bucket"].eq("Sin dato").any():
    national_bucket_order.append("Sin dato")

national_awt0["awt0_bucket"] = pd.Categorical(
    national_awt0["awt0_bucket"],
    categories=national_bucket_order,
    ordered=True,
)

national_vendor_long = (
    national_awt0
    .groupby(
        ["awt0_bucket", "coverage_status"],
        observed=False,
    )
    .agg(vendors=("vendor_code", "nunique"))
    .reset_index()
)

national_vendor_table = (
    national_vendor_long
    .pivot_table(
        index="awt0_bucket",
        columns="coverage_status",
        values="vendors",
        aggfunc="sum",
        fill_value=0,
        observed=False,
    )
    .reindex(index=national_bucket_order, fill_value=0)
    .reindex(columns=COVERAGE_ORDER, fill_value=0)
)

national_vendor_table["Total vendors"] = (
    national_vendor_table.sum(axis=1)
)

total_national_vendors = int(
    national_awt0["vendor_code"].nunique()
)

total_reduction_vendors = int(
    national_awt0.loc[
        national_awt0["is_in_reductions"],
        "vendor_code",
    ].nunique()
)

vendor_coverage_pct = (
    100 * total_reduction_vendors / total_national_vendors
    if total_national_vendors
    else np.nan
)

missing_reduction_vendors_in_sheet = int(
    national_awt0.loc[
        national_awt0["is_in_reductions"]
        & national_awt0["awt0_share_pct"].isna(),
        "vendor_code",
    ].nunique()
)

plot_coverage_stacked(
    plot_table=national_vendor_table,
    series_columns=COVERAGE_ORDER,
    title="Panorama CL de vendors por share AWT0",
    subtitle=(
        f"Cobertura de {latest_week}: "
        f"{total_reduction_vendors:,} de "
        f"{total_national_vendors:,} vendors "
        f"({vendor_coverage_pct:.1f}%). "
        f"Sin share AWT0: {missing_reduction_vendors_in_sheet:,} "
        "vendors alcanzados."
    ),
    ylabel="Vendors únicos",
    percentage=False,
)

display(
    national_vendor_table
    .astype(int)
    .style
    .format("{:,.0f}")
    .set_caption(
        f"Vendors CL y cobertura de reductions — {latest_week}"
    )
)


# ============================================================
# 2. DETECTAR ORDERS POR VENDOR EN SHEET1
# ============================================================

columns_by_key_national = {
    normalize_column_key(column): column
    for column in df_awt0_source.columns
}

order_column_candidates = [
    "total_orders",
    "orders",
    "orders_1w",
    "total_orders_1w",
    "orders_last_week",
    "last_week_orders",
    "order_count",
    "orders_count",
    "orders_2w",
    "total_orders_2w",
]

sheet_orders_column = next(
    (
        columns_by_key_national[candidate]
        for candidate in order_column_candidates
        if candidate in columns_by_key_national
    ),
    None,
)


# ============================================================
# 3A. MODO COMPLETO: SHEET1 TRAE ORDERS POR VENDOR
# ============================================================

orders_chart_created = False

if sheet_orders_column is not None:
    print(
        "Orders por vendor detectadas en Sheet1: "
        f"{sheet_orders_column}"
    )

    sheet_orders = df_awt0_source[
        [sheet_vendor_column, sheet_orders_column]
    ].copy()

    sheet_orders.columns = [
        "vendor_code",
        "vendor_orders",
    ]

    sheet_orders["vendor_code"] = normalize_vendor_code(
        sheet_orders["vendor_code"]
    )

    sheet_orders["vendor_orders"] = parse_numeric_series(
        sheet_orders["vendor_orders"]
    )

    sheet_orders = sheet_orders.loc[
        sheet_orders["vendor_code"].notna()
        & sheet_orders["vendor_code"].ne("")
    ].copy()

    # Si hay duplicados, la mediana evita duplicar el volumen cuando
    # la misma métrica fue repetida en más de una fila del Sheet.
    sheet_orders_vendor = (
        sheet_orders
        .groupby("vendor_code", as_index=False)
        .agg(vendor_orders=("vendor_orders", "median"))
    )

    national_orders = national_awt0.merge(
        sheet_orders_vendor,
        on="vendor_code",
        how="left",
        validate="one_to_one",
    )

    national_orders["vendor_orders"] = (
        national_orders["vendor_orders"]
        .fillna(0)
        .clip(lower=0)
    )

    national_orders_total = float(
        national_orders["vendor_orders"].sum()
    )

    if national_orders_total > 0:
        national_orders_long = (
            national_orders
            .groupby(
                ["awt0_bucket", "coverage_status"],
                observed=False,
            )
            .agg(orders=("vendor_orders", "sum"))
            .reset_index()
        )

        national_orders_long["orders_cl_pct"] = (
            100
            * national_orders_long["orders"]
            / national_orders_total
        )

        national_orders_table = (
            national_orders_long
            .pivot_table(
                index="awt0_bucket",
                columns="coverage_status",
                values="orders_cl_pct",
                aggfunc="sum",
                fill_value=0,
                observed=False,
            )
            .reindex(index=national_bucket_order, fill_value=0)
            .reindex(columns=COVERAGE_ORDER, fill_value=0)
        )

        national_orders_table["Total % orders CL"] = (
            national_orders_table.sum(axis=1)
        )

        reductions_orders_pct = float(
            national_orders.loc[
                national_orders["is_in_reductions"],
                "vendor_orders",
            ].sum()
            / national_orders_total
            * 100
        )

        plot_coverage_stacked(
            plot_table=national_orders_table,
            series_columns=COVERAGE_ORDER,
            title="Panorama CL de órdenes por share AWT0",
            subtitle=(
                "Cada barra muestra su aporte al 100% de órdenes CL. "
                f"{latest_week} alcanza {reductions_orders_pct:.2f}% "
                "de las órdenes del universo del Sheet."
            ),
            ylabel="Porcentaje de órdenes CL",
            percentage=True,
        )

        display(
            national_orders_table
            .style
            .format("{:.2f}%")
            .set_caption(
                "Share de órdenes CL y cobertura de reductions"
            )
        )

        orders_chart_created = True


# ============================================================
# 3B. FALLBACK: DF_PRE_POST SOLO PERMITE MEDIR REDUCTIONS
#
# Como resto_de_chile viene comprimido, este modo NO inventa la
# distribución de los vendors no reducidos entre buckets. Solo muestra
# qué porcentaje de las órdenes CL fue alcanzado por reductions.
# ============================================================

if not orders_chart_created:
    required_pre_post_columns = {
        "wave",
        "analysis_group",
        "period",
        "vendor_code",
        "total_orders",
        "cl_total_orders",
    }

    can_use_pre_post = (
        "df_pre_post" in globals()
        and required_pre_post_columns.issubset(
            df_pre_post.columns
        )
    )

    if can_use_pre_post:
        reduction_orders = df_pre_post.copy()

        reduction_orders["wave"] = (
            reduction_orders["wave"]
            .astype("string")
            .str.strip()
        )

        reduction_orders["analysis_group"] = (
            reduction_orders["analysis_group"]
            .astype("string")
            .str.strip()
            .str.lower()
        )

        reduction_orders["period"] = (
            reduction_orders["period"]
            .astype("string")
            .str.strip()
            .str.lower()
        )

        reduction_orders["vendor_code"] = normalize_vendor_code(
            reduction_orders["vendor_code"]
        )

        reduction_orders["total_orders"] = pd.to_numeric(
            reduction_orders["total_orders"],
            errors="coerce",
        ).fillna(0)

        reduction_orders["cl_total_orders"] = pd.to_numeric(
            reduction_orders["cl_total_orders"],
            errors="coerce",
        )

        reduction_orders = reduction_orders.loc[
            reduction_orders["wave"].eq(str(latest_week))
            & reduction_orders["analysis_group"].eq("treatment")
            & reduction_orders["period"].eq("post")
        ].copy()

        if not reduction_orders.empty:
            cl_total_candidates = (
                reduction_orders["cl_total_orders"]
                .dropna()
            )

            cl_total_orders_post = (
                float(cl_total_candidates.max())
                if not cl_total_candidates.empty
                else np.nan
            )

            vendor_reduction_orders = (
                reduction_orders
                .groupby("vendor_code", as_index=False)
                .agg(vendor_orders=("total_orders", "sum"))
            )

            reduction_order_coverage = (
                national_awt0.loc[
                    national_awt0["is_in_reductions"]
                ]
                .merge(
                    vendor_reduction_orders,
                    on="vendor_code",
                    how="left",
                    validate="one_to_one",
                )
            )

            reduction_order_coverage["vendor_orders"] = (
                reduction_order_coverage["vendor_orders"]
                .fillna(0)
            )

            if (
                pd.notna(cl_total_orders_post)
                and cl_total_orders_post > 0
            ):
                coverage_orders_long = (
                    reduction_order_coverage
                    .groupby(
                        ["awt0_bucket", "coverage_status"],
                        observed=False,
                    )
                    .agg(orders=("vendor_orders", "sum"))
                    .reset_index()
                )

                coverage_orders_long["orders_cl_pct"] = (
                    100
                    * coverage_orders_long["orders"]
                    / cl_total_orders_post
                )

                covered_statuses = [
                    "Otra reduction",
                    "AWT0 reduction",
                ]

                coverage_orders_table = (
                    coverage_orders_long
                    .pivot_table(
                        index="awt0_bucket",
                        columns="coverage_status",
                        values="orders_cl_pct",
                        aggfunc="sum",
                        fill_value=0,
                        observed=False,
                    )
                    .reindex(
                        index=national_bucket_order,
                        fill_value=0,
                    )
                    .reindex(
                        columns=covered_statuses,
                        fill_value=0,
                    )
                )

                coverage_orders_table["Total alcanzado"] = (
                    coverage_orders_table.sum(axis=1)
                )

                total_covered_orders_pct = float(
                    coverage_orders_table[
                        covered_statuses
                    ].to_numpy().sum()
                )

                plot_coverage_stacked(
                    plot_table=coverage_orders_table,
                    series_columns=covered_statuses,
                    title=(
                        "Cobertura de orders CL por share AWT0"
                    ),
                    subtitle=(
                        f"{latest_week} alcanza "
                        f"{total_covered_orders_pct:.2f}% de las "
                        "orders CL Post. Solo se grafica la parte "
                        "cubierta: el resto nacional viene agregado y "
                        "no puede asignarse a buckets."
                    ),
                    ylabel="Porcentaje de órdenes CL alcanzado",
                    percentage=True,
                )

                display(
                    coverage_orders_table
                    .style
                    .format("{:.2f}%")
                    .set_caption(
                        f"Cobertura de orders CL — {latest_week}"
                    )
                )

                orders_chart_created = True


if not orders_chart_created:
    print(
        "\nNo fue posible construir el gráfico de orders todavía.\n"
        "Para la vista nacional completa, Sheet1 necesita una columna "
        "de orders por vendor del mismo período usado para AWT_0.\n"
        "Alternativamente, ejecuta la query Pre/Post, confirma 'sí' y "
        "vuelve a ejecutar esta celda para ver la cobertura de "
        "reductions como % de orders CL."
    )


# Objetos de auditoría disponibles:
# - national_awt0
# - national_vendor_table
# - national_orders_table       (si Sheet1 trae orders)
# - coverage_orders_table       (fallback con df_pre_post)


In [ ]:
# ============================================================
# PANORAMA CL DE awt0_share × REDUCTIONS — TODOS LOS WEEK
#
# Pegar al final del notebook, después de cargar:
#   - reduction
#   - df_pre_post (para calcular el % de orders CL por Week)
#   - gc (opcional; si no existe, se autentica Google Sheets)
#
# REGLAS:
#   - Usa EXCLUSIVAMENTE la columna awt0_share de Sheet1.
#   - Incluye todos los wave cuyo nombre contiene "Week" o "Mini".
#   - NO conserva solo la última wave por vendor.
#   - Un vendor reincidente aparece en cada Week correspondiente.
#   - Dentro de cada Week, cada vendor se cuenta una sola vez.
# ============================================================

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from matplotlib.ticker import FuncFormatter


# ------------------------------------------------------------
# CONFIGURACIÓN
# ------------------------------------------------------------

SHEET_ID_AWT0 = "1bq1G_luDEW4g-K4ePYBmMu68e021JIyPWznrqPLGewo"
SHEET_TAB_AWT0 = "shares_awt0"
AWT0_SHARE_COLUMN_KEY = "awt0_share"

BUCKET_EDGES = [
    0, 10, 20, 30, 40, 50,
    60, 70, 80, 90, 100.000001,
]

BUCKET_LABELS = [
    "[0–10%)",
    "[10–20%)",
    "[20–30%)",
    "[30–40%)",
    "[40–50%)",
    "[50–60%)",
    "[60–70%)",
    "[70–80%)",
    "[80–90%)",
    "[90–100%]",
]

COVERAGE_ORDER = [
    "No incluido en reductions",
    "Otra reduction",
    "AWT0 reduction",
]

REDUCTION_ORDER = [
    "Otra reduction",
    "AWT0 reduction",
]

COVERAGE_COLORS = {
    "No incluido en reductions": "#D0D5DD",
    "Otra reduction": "#7F56D9",
    "AWT0 reduction": "#2563EB",
}


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def normalize_column_key(value):
    value = (
        str(value)
        .strip()
        .lower()
        .replace("%", " pct ")
    )

    return re.sub(
        r"_+",
        "_",
        re.sub(r"[^a-z0-9]+", "_", value),
    ).strip("_")


def normalize_vendor_code(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def wave_sort_number(value):
    match = re.search(r"(?i)week\D*(\d+)", str(value))
    return int(match.group(1)) if match else 10**9


def plot_stacked_buckets(
    table,
    columns,
    title,
    subtitle,
    ylabel,
    percentage=False,
):
    data = table[columns].copy()

    if data.empty or data.to_numpy().sum() <= 0:
        print(f"Sin datos para: {title}")
        return

    sns.set_theme(style="whitegrid", context="talk")

    fig, ax = plt.subplots(figsize=(19, 9.5), dpi=120)

    data.plot(
        kind="bar",
        stacked=True,
        width=0.76,
        color=[COVERAGE_COLORS[column] for column in columns],
        edgecolor="white",
        linewidth=1.0,
        ax=ax,
    )

    totals = data.sum(axis=1)
    max_total = max(float(totals.max()), 1.0)

    for container in ax.containers:
        labels = []

        for bar in container:
            value = float(bar.get_height())

            if value <= 0:
                labels.append("")
            elif percentage:
                labels.append(f"{value:.2f}%")
            else:
                labels.append(f"{int(round(value)):,}")

        ax.bar_label(
            container,
            labels=labels,
            label_type="center",
            fontsize=13,
            fontweight="bold",
            color="white",
        )

    for position, total in enumerate(totals):
        total = float(total)

        if total <= 0:
            continue

        total_label = (
            f"{total:.2f}%"
            if percentage
            else f"{int(round(total)):,}"
        )

        ax.text(
            position,
            total + max_total * 0.025,
            total_label,
            ha="center",
            va="bottom",
            fontsize=14,
            fontweight="bold",
            color="#101828",
        )

    ax.set_ylim(0, max_total * 1.20)

    ax.set_title(
        title,
        fontsize=24,
        fontweight="bold",
        color="#101828",
        pad=38,
    )

    ax.text(
        0,
        1.02,
        subtitle,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=15,
        color="#667085",
        wrap=True,
    )

    ax.set_xlabel(
        "Bucket de awt0_share",
        fontsize=18,
        fontweight="semibold",
        color="#344054",
        labelpad=15,
    )

    ax.set_ylabel(
        ylabel,
        fontsize=18,
        fontweight="semibold",
        color="#344054",
    )

    if percentage:
        ax.yaxis.set_major_formatter(
            FuncFormatter(lambda value, _: f"{value:.1f}%")
        )

    ax.tick_params(
        axis="x",
        labelrotation=35,
        labelsize=14,
        colors="#344054",
    )

    ax.tick_params(
        axis="y",
        labelsize=14,
        colors="#475467",
    )

    ax.grid(
        axis="y",
        color="#D0D5DD",
        linewidth=0.8,
        alpha=0.65,
    )

    ax.grid(axis="x", visible=False)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_color("#D0D5DD")

    legend = ax.legend(
        title="Cobertura de reductions",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
        fontsize=14,
    )

    legend.get_title().set_fontsize(15)
    legend.get_title().set_fontweight("bold")

    plt.tight_layout(rect=[0, 0, 0.82, 1])
    plt.show()


# ------------------------------------------------------------
# 1. VALIDAR REDUCTIONS Y CONSERVAR TODOS LOS WEEK + MINI
# ------------------------------------------------------------

if "reduction" not in globals():
    raise RuntimeError(
        "Primero debes ejecutar la celda que carga reduction."
    )

required_reduction_columns = {
    "wave",
    "flag",
    "vendor_code",
}

missing_reduction_columns = (
    required_reduction_columns - set(reduction.columns)
)

if missing_reduction_columns:
    raise KeyError(
        "Faltan columnas en reduction: "
        f"{sorted(missing_reduction_columns)}"
    )

week_source = reduction.copy()

week_source["wave"] = (
    week_source["wave"]
    .astype("string")
    .str.strip()
)

wave_lower = week_source["wave"].str.lower()

week_source = week_source.loc[
    wave_lower.str.contains(r"week|mini", na=False, regex=True)
].copy()

week_source["vendor_code"] = normalize_vendor_code(
    week_source["vendor_code"]
)

week_source["flag"] = (
    week_source["flag"]
    .astype("string")
    .fillna("SIN_FLAG")
    .str.strip()
)

week_source = week_source.loc[
    week_source["vendor_code"].notna()
    & week_source["vendor_code"].ne("")
].copy()

if week_source.empty:
    raise ValueError(
        "No hay waves cuyo nombre contenga Week o Mini."
    )

flag_normalized = (
    week_source["flag"]
    .astype("string")
    .fillna("")
    .str.lower()
    .str.replace(r"\s+", "", regex=True)
)

week_source["_is_awt0_reduction"] = (
    flag_normalized.str.contains("awt0", na=False)
    & flag_normalized.str.contains("reduction", na=False)
)

# Una fila por wave × vendor. No se elimina el historial del vendor.
week_vendor_reductions = (
    week_source
    .groupby(["wave", "vendor_code"], as_index=False)
    .agg(
        is_awt0_reduction=("_is_awt0_reduction", "max"),
        criterios=(
            "flag",
            lambda values: " | ".join(
                sorted(set(values.dropna().astype(str)))
            ),
        ),
    )
)

week_vendor_reductions["is_in_reductions"] = True

wave_order = sorted(
    week_vendor_reductions["wave"].unique(),
    key=lambda value: (wave_sort_number(value), str(value)),
)

print(
    "Weeks y Mini incluidas: "
    + ", ".join(map(str, wave_order))
)


# ------------------------------------------------------------
# 2. CARGAR Sheet1
# ------------------------------------------------------------

if "gc" not in globals():
    import gspread
    from google.colab import auth
    import google.auth

    auth.authenticate_user()

    credentials, _ = google.auth.default(scopes=[
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive",
    ])

    gc = gspread.authorize(credentials)

worksheet_awt0 = (
    gc.open_by_key(SHEET_ID_AWT0)
    .worksheet(SHEET_TAB_AWT0)
)

values_awt0 = worksheet_awt0.get_all_values()

if len(values_awt0) < 2:
    raise ValueError(
        "Sheet1 está vacío o no contiene filas de datos."
    )

headers_awt0 = [
    str(column).strip()
    for column in values_awt0[0]
]

duplicated_headers = pd.Series(headers_awt0)[
    pd.Series(headers_awt0).duplicated(keep=False)
].unique().tolist()

if duplicated_headers:
    raise ValueError(
        "Sheet1 tiene encabezados duplicados: "
        f"{duplicated_headers}"
    )

df_awt0_source = pd.DataFrame(
    values_awt0[1:],
    columns=headers_awt0,
)

columns_by_key = {
    normalize_column_key(column): column
    for column in df_awt0_source.columns
}

vendor_candidates = [
    "vendor_code",
    "vendor_id",
    "partner_id",
    "partner_code",
    "vendor",
]

sheet_vendor_column = next(
    (
        columns_by_key[candidate]
        for candidate in vendor_candidates
        if candidate in columns_by_key
    ),
    None,
)

if sheet_vendor_column is None:
    raise KeyError(
        "No pude identificar la columna vendor de Sheet1. "
        f"Columnas disponibles: {list(df_awt0_source.columns)}"
    )

# Se exige awt0_share. AWT_0 no se acepta como reemplazo.
if AWT0_SHARE_COLUMN_KEY not in columns_by_key:
    raise KeyError(
        "Sheet1 debe contener la columna awt0_share. "
        f"Columnas disponibles: {list(df_awt0_source.columns)}"
    )

sheet_awt0_column = columns_by_key[
    AWT0_SHARE_COLUMN_KEY
]

print(
    f"Sheet cargado: {SHEET_TAB_AWT0} | "
    f"{len(df_awt0_source):,} filas"
)
print(f"Columna vendor: {sheet_vendor_column}")
print(f"Columna usada para buckets: {sheet_awt0_column}")


# ------------------------------------------------------------
# 3. NORMALIZAR awt0_share A 0–100
# ------------------------------------------------------------

df_awt0 = df_awt0_source[
    [sheet_vendor_column, sheet_awt0_column]
].copy()

df_awt0.columns = [
    "vendor_code",
    "awt0_share_raw",
]

df_awt0["vendor_code"] = normalize_vendor_code(
    df_awt0["vendor_code"]
)

share_text = (
    df_awt0["awt0_share_raw"]
    .astype("string")
    .str.strip()
)

has_pct_symbol = share_text.str.contains(
    "%",
    regex=False,
    na=False,
)

share_number = pd.to_numeric(
    share_text
    .str.replace("%", "", regex=False)
    .str.replace(",", ".", regex=False),
    errors="coerce",
)

values_without_pct = share_number.loc[
    ~has_pct_symbol & share_number.notna()
]

values_are_fraction = (
    not values_without_pct.empty
    and values_without_pct.quantile(0.95) <= 1.000001
)

df_awt0["awt0_share_pct"] = share_number

if values_are_fraction:
    df_awt0.loc[
        ~has_pct_symbol,
        "awt0_share_pct",
    ] *= 100

invalid_share = (
    df_awt0["awt0_share_pct"].notna()
    & ~df_awt0["awt0_share_pct"].between(0, 100)
)

if invalid_share.any():
    print(
        "Advertencia: "
        f"{int(invalid_share.sum()):,} valores de awt0_share "
        "fuera de 0–100 pasarán a Sin dato."
    )

    df_awt0.loc[
        invalid_share,
        "awt0_share_pct",
    ] = np.nan

df_awt0 = df_awt0.loc[
    df_awt0["vendor_code"].notna()
    & df_awt0["vendor_code"].ne("")
].copy()

df_awt0_vendor = (
    df_awt0
    .groupby("vendor_code", as_index=False)
    .agg(
        awt0_share_pct=("awt0_share_pct", "median")
    )
)


# ------------------------------------------------------------
# 4. CREAR UNIVERSO CL PARA CADA WEEK
# ------------------------------------------------------------

coverage_frames = []

for wave in wave_order:
    wave_vendors = week_vendor_reductions.loc[
        week_vendor_reductions["wave"].eq(wave),
        [
            "vendor_code",
            "is_awt0_reduction",
            "is_in_reductions",
            "criterios",
        ],
    ].copy()

    # Outer deja visibles como Sin dato los vendors de reductions
    # que no encuentren awt0_share en Sheet1.
    wave_coverage = df_awt0_vendor.merge(
        wave_vendors,
        on="vendor_code",
        how="outer",
        validate="one_to_one",
    )

    wave_coverage["wave"] = wave

    wave_coverage["is_in_reductions"] = (
        wave_coverage["is_in_reductions"]
        .astype("boolean")
        .fillna(False)
        .astype(bool)
    )

    wave_coverage["is_awt0_reduction"] = (
        wave_coverage["is_awt0_reduction"]
        .astype("boolean")
        .fillna(False)
        .astype(bool)
    )

    wave_coverage["coverage_status"] = np.select(
        [
            wave_coverage["is_awt0_reduction"],
            wave_coverage["is_in_reductions"],
        ],
        [
            "AWT0 reduction",
            "Otra reduction",
        ],
        default="No incluido en reductions",
    )

    wave_coverage["awt0_bucket"] = pd.cut(
        wave_coverage["awt0_share_pct"],
        bins=BUCKET_EDGES,
        labels=BUCKET_LABELS,
        right=False,
        include_lowest=True,
    ).astype("string")

    wave_coverage["awt0_bucket"] = (
        wave_coverage["awt0_bucket"]
        .fillna("Sin dato")
    )

    coverage_frames.append(wave_coverage)

awt0_coverage_all_weeks = pd.concat(
    coverage_frames,
    ignore_index=True,
)

bucket_order = BUCKET_LABELS.copy()

if awt0_coverage_all_weeks["awt0_bucket"].eq("Sin dato").any():
    bucket_order.append("Sin dato")

awt0_coverage_all_weeks["awt0_bucket"] = pd.Categorical(
    awt0_coverage_all_weeks["awt0_bucket"],
    categories=bucket_order,
    ordered=True,
)


# ------------------------------------------------------------
# 5. VENDORS — UN GRÁFICO APILADO POR WEEK
# ------------------------------------------------------------

vendor_tables_by_wave = {}
vendor_summary_frames = []

for wave in wave_order:
    wave_coverage = awt0_coverage_all_weeks.loc[
        awt0_coverage_all_weeks["wave"].eq(wave)
    ].copy()

    vendor_long = (
        wave_coverage
        .groupby(
            ["awt0_bucket", "coverage_status"],
            observed=False,
        )
        .agg(vendors=("vendor_code", "nunique"))
        .reset_index()
    )

    vendor_table = (
        vendor_long
        .pivot_table(
            index="awt0_bucket",
            columns="coverage_status",
            values="vendors",
            aggfunc="sum",
            fill_value=0,
            observed=False,
        )
        .reindex(index=bucket_order, fill_value=0)
        .reindex(columns=COVERAGE_ORDER, fill_value=0)
    )

    vendor_table["Total vendors"] = (
        vendor_table[COVERAGE_ORDER].sum(axis=1)
    )

    vendor_table["% vendors en reductions"] = np.where(
        vendor_table["Total vendors"].gt(0),
        100
        * vendor_table[REDUCTION_ORDER].sum(axis=1)
        / vendor_table["Total vendors"],
        np.nan,
    )

    total_cl_vendors = int(
        wave_coverage["vendor_code"].nunique()
    )

    total_reduction_vendors = int(
        wave_coverage.loc[
            wave_coverage["is_in_reductions"],
            "vendor_code",
        ].nunique()
    )

    vendor_coverage_pct = (
        100 * total_reduction_vendors / total_cl_vendors
        if total_cl_vendors
        else np.nan
    )

    missing_share = int(
        wave_coverage.loc[
            wave_coverage["is_in_reductions"]
            & wave_coverage["awt0_share_pct"].isna(),
            "vendor_code",
        ].nunique()
    )

    plot_stacked_buckets(
        table=vendor_table,
        columns=COVERAGE_ORDER,
        title=f"Vendors CL por awt0_share — {wave}",
        subtitle=(
            f"{total_reduction_vendors:,} de "
            f"{total_cl_vendors:,} vendors están en esta Week "
            f"({vendor_coverage_pct:.1f}%). "
            f"Vendors de reductions sin awt0_share: {missing_share:,}."
        ),
        ylabel="Vendors únicos",
        percentage=False,
    )

    display(
        vendor_table
        .style
        .format({
            "No incluido en reductions": "{:,.0f}",
            "Otra reduction": "{:,.0f}",
            "AWT0 reduction": "{:,.0f}",
            "Total vendors": "{:,.0f}",
            "% vendors en reductions": "{:.1f}%",
        })
        .set_caption(
            f"Vendors CL por bucket de awt0_share — {wave}"
        )
    )

    vendor_tables_by_wave[wave] = vendor_table

    vendor_export = vendor_table.reset_index()
    vendor_export.insert(0, "wave", wave)
    vendor_summary_frames.append(vendor_export)

awt0_vendor_summary_all_weeks = pd.concat(
    vendor_summary_frames,
    ignore_index=True,
)


# ------------------------------------------------------------
# 6. ORDERS — % CL ALCANZADO POR CADA WEEK
#
# Usa las orders Post de treatment en df_pre_post y las divide por
# cl_total_orders de esa misma Week. Solo grafica la parte alcanzada:
# el resto nacional está agregado y no puede distribuirse por bucket.
# ------------------------------------------------------------

orders_tables_by_wave = {}
orders_summary_frames = []

required_pre_post_columns = {
    "wave",
    "analysis_group",
    "period",
    "vendor_code",
    "total_orders",
    "cl_total_orders",
}

can_build_orders = (
    "df_pre_post" in globals()
    and required_pre_post_columns.issubset(df_pre_post.columns)
)

if not can_build_orders:
    print(
        "\nNo se construyeron los gráficos de orders. "
        "Ejecuta primero la query y las celdas que crean df_pre_post. "
        "Los gráficos de vendors sí quedaron disponibles."
    )
else:
    orders_source = df_pre_post.copy()

    orders_source["wave"] = (
        orders_source["wave"]
        .astype("string")
        .str.strip()
    )

    orders_source["analysis_group"] = (
        orders_source["analysis_group"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    orders_source["period"] = (
        orders_source["period"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    orders_source["vendor_code"] = normalize_vendor_code(
        orders_source["vendor_code"]
    )

    orders_source["total_orders"] = pd.to_numeric(
        orders_source["total_orders"],
        errors="coerce",
    ).fillna(0)

    orders_source["cl_total_orders"] = pd.to_numeric(
        orders_source["cl_total_orders"],
        errors="coerce",
    )

    orders_source = orders_source.loc[
        orders_source["analysis_group"].eq("treatment")
        & orders_source["period"].eq("post")
    ].copy()

    for wave in wave_order:
        wave_orders = orders_source.loc[
            orders_source["wave"].eq(str(wave))
        ].copy()

        if wave_orders.empty:
            print(
                f"\n{wave}: sin filas treatment/Post en df_pre_post; "
                "se omite su gráfico de orders."
            )
            continue

        cl_total_candidates = (
            wave_orders["cl_total_orders"].dropna()
        )

        cl_total_orders_post = (
            float(cl_total_candidates.max())
            if not cl_total_candidates.empty
            else np.nan
        )

        if (
            pd.isna(cl_total_orders_post)
            or cl_total_orders_post <= 0
        ):
            print(
                f"\n{wave}: cl_total_orders no es válido; "
                "se omite su gráfico de orders."
            )
            continue

        vendor_orders = (
            wave_orders
            .groupby("vendor_code", as_index=False)
            .agg(vendor_orders=("total_orders", "sum"))
        )

        wave_coverage = awt0_coverage_all_weeks.loc[
            awt0_coverage_all_weeks["wave"].eq(wave)
            & awt0_coverage_all_weeks["is_in_reductions"]
        ].copy()

        covered_orders = wave_coverage.merge(
            vendor_orders,
            on="vendor_code",
            how="left",
            validate="one_to_one",
        )

        covered_orders["vendor_orders"] = (
            covered_orders["vendor_orders"]
            .fillna(0)
            .clip(lower=0)
        )

        expected_vendors = int(
            wave_coverage["vendor_code"].nunique()
        )

        vendors_with_orders = int(
            covered_orders.loc[
                covered_orders["vendor_orders"].gt(0),
                "vendor_code",
            ].nunique()
        )

        coverage_match_pct = (
            100 * vendors_with_orders / expected_vendors
            if expected_vendors
            else np.nan
        )

        orders_long = (
            covered_orders
            .groupby(
                ["awt0_bucket", "coverage_status"],
                observed=False,
            )
            .agg(orders=("vendor_orders", "sum"))
            .reset_index()
        )

        orders_long["orders_cl_pct"] = (
            100 * orders_long["orders"] / cl_total_orders_post
        )

        orders_table = (
            orders_long
            .pivot_table(
                index="awt0_bucket",
                columns="coverage_status",
                values="orders_cl_pct",
                aggfunc="sum",
                fill_value=0,
                observed=False,
            )
            .reindex(index=bucket_order, fill_value=0)
            .reindex(columns=REDUCTION_ORDER, fill_value=0)
        )

        orders_table["Total % orders CL alcanzado"] = (
            orders_table[REDUCTION_ORDER].sum(axis=1)
        )

        total_orders_coverage_pct = float(
            orders_table[REDUCTION_ORDER].to_numpy().sum()
        )

        plot_stacked_buckets(
            table=orders_table,
            columns=REDUCTION_ORDER,
            title=f"Orders CL alcanzadas por awt0_share — {wave}",
            subtitle=(
                f"La Week alcanza {total_orders_coverage_pct:.2f}% "
                f"de las orders CL Post. Hay orders para "
                f"{vendors_with_orders:,} de {expected_vendors:,} "
                f"vendors de reductions ({coverage_match_pct:.1f}%)."
            ),
            ylabel="Porcentaje de orders CL alcanzado",
            percentage=True,
        )

        display(
            orders_table
            .style
            .format("{:.2f}%")
            .set_caption(
                f"% orders CL alcanzado por bucket — {wave}"
            )
        )

        orders_tables_by_wave[wave] = orders_table

        orders_export = orders_table.reset_index()
        orders_export.insert(0, "wave", wave)
        orders_summary_frames.append(orders_export)

    if orders_summary_frames:
        awt0_orders_summary_all_weeks = pd.concat(
            orders_summary_frames,
            ignore_index=True,
        )
    else:
        awt0_orders_summary_all_weeks = pd.DataFrame()


# ------------------------------------------------------------
# OBJETOS DE AUDITORÍA DISPONIBLES
# ------------------------------------------------------------
# week_vendor_reductions
# awt0_coverage_all_weeks
# vendor_tables_by_wave
# awt0_vendor_summary_all_weeks
# orders_tables_by_wave
# awt0_orders_summary_all_weeks


In [ ]:
# ============================================================
# TABLA AWT0: VENDORS + ORDERS POST + % ORDERS CL
# Requiere:
#   - latest_week
#   - latest_wave_awt0
#   - df_pre_post
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display


def normalize_vendor_code_local(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


STATUS_ORDER = [
    "No reducido por AWT0",
    "Reducido por AWT0",
]

# Usa el mismo orden de buckets creado en la celda anterior.
BATCH_ORDER = list(bucket_order)


# ------------------------------------------------------------
# 1. Orders Post de treatment y total de orders CL Post
# ------------------------------------------------------------

orders_post = df_pre_post.copy()

orders_post["wave"] = (
    orders_post["wave"]
    .astype("string")
    .str.strip()
)

orders_post["analysis_group"] = (
    orders_post["analysis_group"]
    .astype("string")
    .str.strip()
    .str.lower()
)

orders_post["period"] = (
    orders_post["period"]
    .astype("string")
    .str.strip()
    .str.lower()
)

orders_post["vendor_code"] = normalize_vendor_code_local(
    orders_post["vendor_code"]
)

orders_post["total_orders"] = pd.to_numeric(
    orders_post["total_orders"],
    errors="coerce",
).fillna(0)

orders_post["cl_total_orders"] = pd.to_numeric(
    orders_post["cl_total_orders"],
    errors="coerce",
)

orders_post = orders_post.loc[
    orders_post["wave"].eq(str(latest_week))
    & orders_post["analysis_group"].eq("treatment")
    & orders_post["period"].eq("post")
].copy()

if orders_post.empty:
    raise ValueError(
        f"No hay filas treatment/Post para {latest_week} en df_pre_post."
    )

cl_total_orders_post = orders_post["cl_total_orders"].dropna().max()

if pd.isna(cl_total_orders_post) or cl_total_orders_post <= 0:
    raise ValueError(
        f"cl_total_orders no es válido para {latest_week}."
    )

# Suma por vendor para no duplicar vendors que vengan en más de una ciudad.
vendor_orders_post = (
    orders_post
    .groupby("vendor_code", as_index=False)
    .agg(
        orders_post=("total_orders", "sum"),
    )
)


# ------------------------------------------------------------
# 2. Cruzar orders con buckets y criterio AWT0
# ------------------------------------------------------------

awt0_table_base = latest_wave_awt0.copy()

awt0_table_base["vendor_code"] = normalize_vendor_code_local(
    awt0_table_base["vendor_code"]
)

awt0_table_base = awt0_table_base.merge(
    vendor_orders_post,
    on="vendor_code",
    how="left",
    validate="one_to_one",
)

awt0_table_base["orders_post"] = (
    awt0_table_base["orders_post"]
    .fillna(0)
)

awt0_table_base["batch_awt0"] = pd.Categorical(
    awt0_table_base["batch_awt0"],
    categories=BATCH_ORDER,
    ordered=True,
)

awt0_table_base["tipo_reduccion"] = pd.Categorical(
    awt0_table_base["tipo_reduccion"],
    categories=STATUS_ORDER,
    ordered=True,
)


# ------------------------------------------------------------
# 3. Tablas por bucket y tipo de reducción
# ------------------------------------------------------------

vendors_table = (
    awt0_table_base
    .groupby(
        ["batch_awt0", "tipo_reduccion"],
        observed=False,
    )
    .agg(vendors=("vendor_code", "nunique"))
    .reset_index()
    .pivot_table(
        index="batch_awt0",
        columns="tipo_reduccion",
        values="vendors",
        aggfunc="sum",
        fill_value=0,
        observed=False,
    )
    .reindex(index=BATCH_ORDER, fill_value=0)
    .reindex(columns=STATUS_ORDER, fill_value=0)
)

vendors_table["Total"] = vendors_table.sum(axis=1)

orders_table = (
    awt0_table_base
    .groupby(
        ["batch_awt0", "tipo_reduccion"],
        observed=False,
    )
    .agg(orders_post=("orders_post", "sum"))
    .reset_index()
    .pivot_table(
        index="batch_awt0",
        columns="tipo_reduccion",
        values="orders_post",
        aggfunc="sum",
        fill_value=0,
        observed=False,
    )
    .reindex(index=BATCH_ORDER, fill_value=0)
    .reindex(columns=STATUS_ORDER, fill_value=0)
)

orders_table["Total"] = orders_table.sum(axis=1)

share_orders_cl_table = (
    orders_table
    .div(cl_total_orders_post)
    .mul(100)
)


# ------------------------------------------------------------
# 4. Tabla final
# ------------------------------------------------------------

awt0_batch_table_with_orders = pd.concat(
    {
        "Vendors": vendors_table,
        "Orders Post": orders_table,
        "% Orders CL Post": share_orders_cl_table,
    },
    axis=1,
)

print(
    f"Wave: {latest_week}"
    f"\nOrders CL Post: {cl_total_orders_post:,.0f}"
    f"\nOrders Post de treatment en la tabla: "
    f"{orders_table['Total'].sum():,.0f}"
    f"\nShare total treatment: "
    f"{100 * orders_table['Total'].sum() / cl_total_orders_post:.2f}%"
)

display(
    awt0_batch_table_with_orders.style
    .format(
        {
            ("Vendors", "No reducido por AWT0"): "{:,.0f}",
            ("Vendors", "Reducido por AWT0"): "{:,.0f}",
            ("Vendors", "Total"): "{:,.0f}",
            ("Orders Post", "No reducido por AWT0"): "{:,.0f}",
            ("Orders Post", "Reducido por AWT0"): "{:,.0f}",
            ("Orders Post", "Total"): "{:,.0f}",
            ("% Orders CL Post", "No reducido por AWT0"): "{:.2f}%",
            ("% Orders CL Post", "Reducido por AWT0"): "{:.2f}%",
            ("% Orders CL Post", "Total"): "{:.2f}%",
        }
    )
    .set_caption(
        f"Buckets AWT0 — vendors y cobertura de orders CL Post ({latest_week})"
    )
)

In [ ]:
# ============================================================
# COBERTURA DE TODAS LAS WAVES WEEK / MINI
# SOBRE LOS BUCKETS NACIONALES DE awt0_share
#
# Ejecutar después de cargar `shares_awt0`.
# Requiere también `reduction` y `gc`.
# ============================================================

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display


# ------------------------------------------------------------
# CONFIGURACIÓN
# ------------------------------------------------------------

# Sheet raw a nivel vendor: debe incluir vendor_code + awt0_share.
AWT0_VENDOR_SHEET_ID = "1bq1G_luDEW4g-K4ePYBmMu68e021JIyPWznrqPLGewo"
AWT0_VENDOR_SHEET_TAB = "Sheet1"

# Mantener Mini incluido.
INCLUDE_MINI = True

BUCKET_ORDER = [
    "[0–10%)",
    "[10–20%)",
    "[20–30%)",
    "[30–40%)",
    "[40–50%)",
    "[50–60%)",
    "[60–70%)",
    "[70–80%)",
    "[80–90%)",
    "[90–100%]",
]


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def normalize_column_key(value):
    value = (
        str(value)
        .strip()
        .lower()
        .replace("%", " pct ")
    )

    return re.sub(
        r"_+",
        "_",
        re.sub(r"[^a-z0-9]+", "_", value),
    ).strip("_")


def normalize_vendor_code(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def parse_number(series):
    return pd.to_numeric(
        series
        .astype("string")
        .str.strip()
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def canonical_awt0_bucket(value):
    """
    Convierte, por ejemplo:
    [0%-10%)  -> [0–10%)
    [10%-20%) -> [10–20%)
    """

    numbers = re.findall(
        r"\d+(?:\.\d+)?",
        str(value),
    )

    if len(numbers) < 2:
        return pd.NA

    lower = int(float(numbers[0]))
    upper = int(float(numbers[1]))

    if lower == 90 and upper == 100:
        return "[90–100%]"

    return f"[{lower}–{upper}%)"


# ------------------------------------------------------------
# 1. TOMAR UN SOLO PERÍODO DE shares_awt0
#    Por defecto: el período más reciente.
# ------------------------------------------------------------

required_shares_columns = {
    "start_date",
    "end_date",
    "awt0_share_bucket",
    "vendors",
    "total_vendors_cl",
    "vendor_share_cl_pct",
    "orders",
    "total_orders_cl",
    "orders_share_cl_pct",
}

missing_shares_columns = (
    required_shares_columns - set(shares_awt0.columns)
)

if missing_shares_columns:
    raise KeyError(
        "Faltan columnas en shares_awt0: "
        f"{sorted(missing_shares_columns)}"
    )

shares_period = shares_awt0.copy()

shares_period["start_date"] = pd.to_datetime(
    shares_period["start_date"],
    errors="coerce",
)

shares_period["end_date"] = pd.to_datetime(
    shares_period["end_date"],
    errors="coerce",
)

latest_period = (
    shares_period[
        ["start_date", "end_date"]
    ]
    .drop_duplicates()
    .sort_values(
        ["end_date", "start_date"]
    )
    .iloc[-1]
)

shares_period = shares_period.loc[
    shares_period["start_date"].eq(
        latest_period["start_date"]
    )
    & shares_period["end_date"].eq(
        latest_period["end_date"]
    )
].copy()

shares_period["awt0_bucket"] = (
    shares_period["awt0_share_bucket"]
    .apply(canonical_awt0_bucket)
)

numeric_columns = [
    "vendors",
    "total_vendors_cl",
    "vendor_share_cl_pct",
    "orders",
    "total_orders_cl",
    "orders_share_cl_pct",
    "awt0_orders",
    "actual_awt0_share_bucket_pct",
]

for column in numeric_columns:
    if column in shares_period.columns:
        shares_period[column] = parse_number(
            shares_period[column]
        )

national_awt0_buckets = (
    shares_period
    .groupby("awt0_bucket", as_index=False)
    .agg(
        vendors_cl=("vendors", "sum"),
        orders_cl=("orders", "sum"),
        vendor_share_cl_pct=(
            "vendor_share_cl_pct",
            "sum",
        ),
        orders_share_cl_pct=(
            "orders_share_cl_pct",
            "sum",
        ),
        awt0_orders=("awt0_orders", "sum"),
        actual_awt0_share_bucket_pct=(
            "actual_awt0_share_bucket_pct",
            "mean",
        ),
    )
    .set_index("awt0_bucket")
    .reindex(BUCKET_ORDER)
    .fillna(0)
)

total_vendors_cl = float(
    shares_period["total_vendors_cl"].dropna().max()
)

total_orders_cl = float(
    shares_period["total_orders_cl"].dropna().max()
)

print(
    "Período nacional usado desde shares_awt0:"
    f"\n• {latest_period['start_date']:%d/%m/%Y}"
    f"–{latest_period['end_date']:%d/%m/%Y}"
    f"\n• Vendors CL: {total_vendors_cl:,.0f}"
    f"\n• Orders CL: {total_orders_cl:,.0f}"
)


# ------------------------------------------------------------
# 2. CARGAR awt0_share RAW POR VENDOR
# ------------------------------------------------------------

worksheet_awt0_vendor = (
    gc.open_by_key(AWT0_VENDOR_SHEET_ID)
    .worksheet(AWT0_VENDOR_SHEET_TAB)
)

values_awt0_vendor = (
    worksheet_awt0_vendor.get_all_values()
)

if len(values_awt0_vendor) < 2:
    raise ValueError(
        "El Sheet raw de awt0_share está vacío."
    )

df_awt0_vendor_source = pd.DataFrame(
    values_awt0_vendor[1:],
    columns=[
        str(column).strip()
        for column in values_awt0_vendor[0]
    ],
)

columns_by_key = {
    normalize_column_key(column): column
    for column in df_awt0_vendor_source.columns
}

vendor_column = next(
    (
        columns_by_key[candidate]
        for candidate in [
            "vendor_code",
            "vendor_id",
            "partner_id",
            "partner_code",
            "vendor",
        ]
        if candidate in columns_by_key
    ),
    None,
)

if vendor_column is None:
    raise KeyError(
        "No encontré una columna de vendor en Sheet1."
    )

if "awt0_share" not in columns_by_key:
    raise KeyError(
        "Sheet1 debe contener exactamente la columna awt0_share. "
        f"Columnas disponibles: "
        f"{list(df_awt0_vendor_source.columns)}"
    )

awt0_share_column = columns_by_key["awt0_share"]

df_awt0_vendor = df_awt0_vendor_source[
    [vendor_column, awt0_share_column]
].copy()

df_awt0_vendor.columns = [
    "vendor_code",
    "awt0_share_raw",
]

df_awt0_vendor["vendor_code"] = normalize_vendor_code(
    df_awt0_vendor["vendor_code"]
)

share_text = (
    df_awt0_vendor["awt0_share_raw"]
    .astype("string")
    .str.strip()
)

share_has_pct = share_text.str.contains(
    "%",
    regex=False,
    na=False,
)

df_awt0_vendor["awt0_share_pct"] = parse_number(
    share_text
)

# Si está en proporción, por ejemplo 0.45, se transforma a 45%.
share_without_symbol = df_awt0_vendor.loc[
    ~share_has_pct,
    "awt0_share_pct",
].dropna()

if (
    not share_without_symbol.empty
    and share_without_symbol.quantile(0.95) <= 1.000001
):
    df_awt0_vendor.loc[
        ~share_has_pct,
        "awt0_share_pct",
    ] *= 100

df_awt0_vendor.loc[
    ~df_awt0_vendor["awt0_share_pct"].between(0, 100),
    "awt0_share_pct",
] = np.nan

df_awt0_vendor = (
    df_awt0_vendor
    .dropna(subset=["vendor_code"])
    .groupby("vendor_code", as_index=False)
    .agg(
        awt0_share_pct=(
            "awt0_share_pct",
            "median",
        )
    )
)

df_awt0_vendor["awt0_bucket"] = pd.cut(
    df_awt0_vendor["awt0_share_pct"],
    bins=[
        0, 10, 20, 30, 40, 50,
        60, 70, 80, 90, 100.000001,
    ],
    labels=BUCKET_ORDER,
    right=False,
    include_lowest=True,
).astype("string")


# ------------------------------------------------------------
# 3. UNIÓN DE VENDORS EN TODAS LAS WAVES WEEK / MINI
# ------------------------------------------------------------

required_reduction_columns = {
    "wave",
    "vendor_code",
}

missing_reduction_columns = (
    required_reduction_columns - set(reduction.columns)
)

if missing_reduction_columns:
    raise KeyError(
        "Faltan columnas en reduction: "
        f"{sorted(missing_reduction_columns)}"
    )

weeks_reduction = reduction.copy()

weeks_reduction["wave"] = (
    weeks_reduction["wave"]
    .astype("string")
    .str.strip()
)

weeks_reduction["vendor_code"] = normalize_vendor_code(
    weeks_reduction["vendor_code"]
)

wave_mask = weeks_reduction["wave"].str.contains(
    "week",
    case=False,
    na=False,
)

if INCLUDE_MINI:
    wave_mask = (
        wave_mask
        | weeks_reduction["wave"].str.contains(
            "mini",
            case=False,
            na=False,
        )
    )

weeks_reduction = weeks_reduction.loc[
    wave_mask
    & weeks_reduction["vendor_code"].notna()
    & weeks_reduction["vendor_code"].ne("")
].copy()

# Unión: si un vendor aparece en Week32, Week34 y Mini,
# se cuenta una vez en el panorama consolidado.
vendors_any_week = (
    weeks_reduction
    .groupby("vendor_code", as_index=False)
    .agg(
        waves=(
            "wave",
            lambda values: " | ".join(
                sorted(set(values.astype(str)))
            ),
        ),
        n_waves=("wave", "nunique"),
    )
)

vendors_any_week["en_alguna_week"] = True

print(
    "\nCobertura de reductions:"
    f"\n• Waves incluidas: "
    f"{weeks_reduction['wave'].nunique():,.0f}"
    f"\n• Vendors únicos en alguna Week/Mini: "
    f"{len(vendors_any_week):,.0f}"
)


# ------------------------------------------------------------
# 4. CRUZAR vendors Week CON BUCKETS AWT0
# ------------------------------------------------------------

vendors_week_by_bucket = (
    vendors_any_week
    .merge(
        df_awt0_vendor,
        on="vendor_code",
        how="left",
        validate="one_to_one",
    )
)

vendors_without_awt0_share = int(
    vendors_week_by_bucket["awt0_bucket"]
    .isna()
    .sum()
)

vendors_week_by_bucket = vendors_week_by_bucket.dropna(
    subset=["awt0_bucket"]
).copy()

week_counts_by_bucket = (
    vendors_week_by_bucket
    .groupby("awt0_bucket")
    .agg(
        vendors_en_week=("vendor_code", "nunique"),
        waves_promedio_por_vendor=("n_waves", "mean"),
    )
    .reindex(BUCKET_ORDER)
    .fillna(0)
)


# ------------------------------------------------------------
# 5. TABLA COMPLEMENTARIA:
#    UNIVERSO CL + COBERTURA DE CUALQUIER WEEK
# ------------------------------------------------------------

awt0_week_coverage_table = national_awt0_buckets.join(
    week_counts_by_bucket,
    how="left",
).fillna(0)

awt0_week_coverage_table["vendors_en_week"] = (
    awt0_week_coverage_table["vendors_en_week"]
    .astype(int)
)

awt0_week_coverage_table["no_en_week"] = (
    awt0_week_coverage_table["vendors_cl"]
    - awt0_week_coverage_table["vendors_en_week"]
)

# Evita números negativos visibles si los universos no calzan.
awt0_week_coverage_table["no_en_week"] = (
    awt0_week_coverage_table["no_en_week"]
    .clip(lower=0)
)

awt0_week_coverage_table[
    "% vendors del bucket en Week"
] = np.where(
    awt0_week_coverage_table["vendors_cl"].gt(0),
    100
    * awt0_week_coverage_table["vendors_en_week"]
    / awt0_week_coverage_table["vendors_cl"],
    np.nan,
)

awt0_week_coverage_table = awt0_week_coverage_table[
    [
        "vendors_cl",
        "no_en_week",
        "vendors_en_week",
        "% vendors del bucket en Week",
        "vendor_share_cl_pct",
        "orders_cl",
        "orders_share_cl_pct",
        "awt0_orders",
        "actual_awt0_share_bucket_pct",
        "waves_promedio_por_vendor",
    ]
]

awt0_week_coverage_table.columns = [
    "Vendors CL",
    "No en Week/Mini",
    "En alguna Week/Mini",
    "% vendors del bucket en Week/Mini",
    "% vendors CL",
    "Orders CL",
    "% orders CL",
    "Orders AWT0",
    "AWT0 real del bucket",
    "Promedio waves por vendor",
]


# ------------------------------------------------------------
# 6. GRÁFICO APILADO: VENDORS CL VS VENDORS EN WAVES
# ------------------------------------------------------------

plot_table = awt0_week_coverage_table[
    [
        "No en Week/Mini",
        "En alguna Week/Mini",
    ]
].copy()

sns.set_theme(style="whitegrid", context="talk")

fig, ax = plt.subplots(
    figsize=(19, 9.5),
    dpi=120,
)

plot_table.plot(
    kind="bar",
    stacked=True,
    width=0.76,
    color=[
        "#D0D5DD",
        "#2563EB",
    ],
    edgecolor="white",
    linewidth=1,
    ax=ax,
)

for container in ax.containers:
    labels = [
        f"{int(bar.get_height()):,}"
        if bar.get_height() > 0
        else ""
        for bar in container
    ]

    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=13,
        fontweight="bold",
        color="white",
    )

totals = plot_table.sum(axis=1)
max_total = max(float(totals.max()), 1)

for position, total in enumerate(totals):
    if total > 0:
        ax.text(
            position,
            total + max_total * 0.025,
            f"{int(total):,}",
            ha="center",
            va="bottom",
            fontsize=14,
            fontweight="bold",
            color="#101828",
        )

ax.set_ylim(0, max_total * 1.20)

ax.set_title(
    "Cobertura de reductions por bucket de AWT0 share",
    fontsize=24,
    fontweight="bold",
    color="#101828",
    pad=38,
)

ax.text(
    0,
    1.02,
    (
        "Azul: vendor que aparece en al menos una wave Week o Mini. "
        "Gris: vendor CL que no aparece en esas waves."
    ),
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=15,
    color="#667085",
)

ax.set_xlabel(
    "Bucket de awt0_share",
    fontsize=18,
    fontweight="semibold",
)

ax.set_ylabel(
    "Vendors únicos CL",
    fontsize=18,
    fontweight="semibold",
)

ax.tick_params(
    axis="x",
    labelrotation=35,
    labelsize=14,
)

ax.tick_params(
    axis="y",
    labelsize=14,
)

ax.grid(axis="x", visible=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

legend = ax.legend(
    title="Cobertura",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    fontsize=14,
)

legend.get_title().set_fontsize(15)
legend.get_title().set_fontweight("bold")

plt.tight_layout(
    rect=[0, 0, 0.84, 1],
)

plt.show()


# ------------------------------------------------------------
# 7. TABLAS
# ------------------------------------------------------------

print(
    "\nValidación:"
    f"\n• Vendors con awt0_share raw: "
    f"{len(df_awt0_vendor):,.0f}"
    f"\n• Vendors en Week/Mini sin match de awt0_share: "
    f"{vendors_without_awt0_share:,}"
    f"\n• Vendors CL según shares_awt0: "
    f"{total_vendors_cl:,.0f}"
)

display(
    national_awt0_buckets.style
    .format({
        "vendors_cl": "{:,.0f}",
        "orders_cl": "{:,.0f}",
        "vendor_share_cl_pct": "{:.2f}%",
        "orders_share_cl_pct": "{:.2f}%",
        "awt0_orders": "{:,.0f}",
        "actual_awt0_share_bucket_pct": "{:.2f}%",
    })
    .set_caption(
        "Universo nacional por bucket de awt0_share"
    )
)

display(
    awt0_week_coverage_table.style
    .format({
        "Vendors CL": "{:,.0f}",
        "No en Week/Mini": "{:,.0f}",
        "En alguna Week/Mini": "{:,.0f}",
        "% vendors del bucket en Week/Mini": "{:.2f}%",
        "% vendors CL": "{:.2f}%",
        "Orders CL": "{:,.0f}",
        "% orders CL": "{:.2f}%",
        "Orders AWT0": "{:,.0f}",
        "AWT0 real del bucket": "{:.2f}%",
        "Promedio waves por vendor": "{:.2f}",
    })
    .set_caption(
        "Cobertura de cualquier Week/Mini sobre el universo CL"
    )
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ============================================================
# CONFIG
# ============================================================

BUCKETS = [
    "[0%-10%)",
    "[10%-20%)",
    "[20%-30%)",
    "[30%-40%)",
    "[40%-50%)",
    "[50%-60%)",
    "[60%-70%)",
    "[70%-80%)",
    "[80%-90%)",
    "[90%-100%]",
]

BIN_EDGES = [
    0, 0.10, 0.20, 0.30, 0.40,
    0.50, 0.60, 0.70, 0.80, 0.90,
    1.0000001
]


# ============================================================
# 1. PREPARAR shares_awt0
#    Esta tabla define el total de orders CL por bucket.
# ============================================================

shares_plot = shares_awt0.copy()

for col in ["orders", "total_orders_cl", "orders_share_cl_pct"]:
    if col in shares_plot.columns:
        shares_plot[col] = pd.to_numeric(
            shares_plot[col],
            errors="coerce"
        )

shares_plot["awt0_share_bucket"] = (
    shares_plot["awt0_share_bucket"]
    .astype(str)
    .str.strip()
)

# Homologar el último bucket.
shares_plot["awt0_share_bucket"] = shares_plot["awt0_share_bucket"].replace({
    "[90%-100%)": "[90%-100%]"
})

# Consolidar por bucket.
shares_plot = (
    shares_plot
    .groupby("awt0_share_bucket", as_index=False)
    .agg(
        orders=("orders", "sum"),
        total_orders_cl=("total_orders_cl", "max"),
    )
)

shares_plot["orders_share_cl_pct"] = (
    100
    * shares_plot["orders"]
    / shares_plot["total_orders_cl"]
)

shares_plot["awt0_share_bucket"] = pd.Categorical(
    shares_plot["awt0_share_bucket"],
    categories=BUCKETS,
    ordered=True,
)

shares_plot = (
    shares_plot
    .sort_values("awt0_share_bucket")
    .reset_index(drop=True)
)


# ============================================================
# 2. TOMAR TODAS LAS REDUCTION WAVES TIPO "Week..."
# ============================================================

reduced = df_merged[
    df_merged["analysis_group"]
    .astype(str)
    .str.strip()
    .eq("treatment")
].copy()

# Solo waves del tipo Week32, Week33, Week34, etc.
reduced["week_number"] = pd.to_numeric(
    reduced["wave"]
    .astype(str)
    .str.extract(r"(?i)^Week\s*(\d+)$")[0],
    errors="coerce"
)

reduced = reduced[
    reduced["week_number"].notna()
].copy()

if reduced.empty:
    raise ValueError(
        "No se encontraron filas treatment con wave tipo WeekXX en df_merged."
    )

# ============================================================
# 3. EVITAR DOBLE CONTEO DE VENDORS ENTRE WEEK WAVES
#
# Si un vendor aparece en varias Week waves, usamos su Week más reciente.
# ============================================================

reduced = (
    reduced
    .sort_values(["vendor_code", "week_number"])
    .drop_duplicates(
        subset=["vendor_code"],
        keep="last"
    )
    .copy()
)

# Asegurar tipos.
reduced["total_orders"] = pd.to_numeric(
    reduced["total_orders"],
    errors="coerce"
)

reduced["awt_0min"] = pd.to_numeric(
    reduced["awt_0min"],
    errors="coerce"
)

reduced = reduced.dropna(
    subset=["total_orders", "awt_0min"]
).copy()

reduced = reduced[
    reduced["total_orders"].gt(0)
    & reduced["awt_0min"].between(
        0,
        1,
        inclusive="both"
    )
].copy()


# ============================================================
# 4. CREAR AWT0 SHARE BUCKET PARA LOS VENDORS REDUCIDOS
# ============================================================

reduced["awt0_share_bucket"] = pd.cut(
    reduced["awt_0min"],
    bins=BIN_EDGES,
    labels=BUCKETS,
    right=False,
    include_lowest=True,
)

reduced_by_bucket = (
    reduced
    .groupby(
        "awt0_share_bucket",
        observed=False,
        as_index=False
    )
    .agg(
        reduction_wave_orders=("total_orders", "sum"),
        reduction_wave_vendors=("vendor_code", "nunique"),
    )
)

reduced_by_bucket["awt0_share_bucket"] = pd.Categorical(
    reduced_by_bucket["awt0_share_bucket"],
    categories=BUCKETS,
    ordered=True,
)


# ============================================================
# 5. COMBINAR CON EL TOTAL CL DE shares_awt0
# ============================================================

awt0_bucket_summary = (
    shares_plot
    .merge(
        reduced_by_bucket,
        on="awt0_share_bucket",
        how="left",
    )
)

awt0_bucket_summary[
    ["reduction_wave_orders", "reduction_wave_vendors"]
] = (
    awt0_bucket_summary[
        ["reduction_wave_orders", "reduction_wave_vendors"]
    ]
    .fillna(0)
)

# Share de CL representado por las orders de vendors reducidos.
awt0_bucket_summary["reduction_wave_share_cl_pct"] = (
    100
    * awt0_bucket_summary["reduction_wave_orders"]
    / awt0_bucket_summary["total_orders_cl"]
)

# Qué proporción del bucket corresponde a vendors reducidos.
awt0_bucket_summary["reduction_share_within_bucket_pct"] = (
    100
    * awt0_bucket_summary["reduction_wave_orders"]
    / awt0_bucket_summary["orders"]
)

# Para apilar la barra:
awt0_bucket_summary["other_orders"] = (
    awt0_bucket_summary["orders"]
    - awt0_bucket_summary["reduction_wave_orders"]
)

# Evitar negativos visuales si los universos de ambos dataframes no son
# perfectamente comparables.
awt0_bucket_summary["other_orders"] = (
    awt0_bucket_summary["other_orders"]
    .clip(lower=0)
)


# ============================================================
# 6. RESUMEN DE WAVES INCLUIDAS
# ============================================================

week_waves_included = (
    reduced[
        ["wave", "week_number"]
    ]
    .drop_duplicates()
    .sort_values("week_number")
)

print("Week waves incluidas:")
print(
    ", ".join(
        week_waves_included["wave"].astype(str)
    )
)

print(
    f"\nVendors únicos reducidos: "
    f"{reduced['vendor_code'].nunique():,.0f}"
)


# ============================================================
# 7. TABLA FINAL
# ============================================================

display_cols = [
    "awt0_share_bucket",
    "orders",
    "orders_share_cl_pct",
    "reduction_wave_orders",
    "reduction_wave_share_cl_pct",
    "reduction_share_within_bucket_pct",
    "reduction_wave_vendors",
]

display(
    awt0_bucket_summary[display_cols]
    .round({
        "orders_share_cl_pct": 2,
        "reduction_wave_share_cl_pct": 2,
        "reduction_share_within_bucket_pct": 2,
    })
)

total_cl_orders = awt0_bucket_summary["total_orders_cl"].max()
total_reduction_wave_orders = (
    awt0_bucket_summary["reduction_wave_orders"].sum()
)

total_reduction_wave_share_cl_pct = (
    100
    * total_reduction_wave_orders
    / total_cl_orders
)

print(
    f"\nTodas las Week waves: "
    f"{total_reduction_wave_orders:,.0f} orders = "
    f"{total_reduction_wave_share_cl_pct:.2f}% de las orders CL"
)


# ============================================================
# 8. GRÁFICO APILADO
#
# Altura total de la barra:
#   orders CL que pertenecen al bucket AWT0.
#
# Segmento inferior:
#   orders asociadas a vendors reducidos en alguna Week wave.
#
# Segmento superior:
#   resto de orders del bucket.
#
# Arriba:
#   cantidad total de orders
#   % de orders CL del bucket
#
# Dentro de Reduction Waves:
#   cantidad de orders reducidas
#   % de CL
#   % del bucket
# ============================================================

plot_df = awt0_bucket_summary.copy()

x = np.arange(len(plot_df))

reduction_orders = (
    plot_df["reduction_wave_orders"]
    .to_numpy()
)

other_orders = (
    plot_df["other_orders"]
    .to_numpy()
)

total_orders = (
    plot_df["orders"]
    .to_numpy()
)

fig, ax = plt.subplots(
    figsize=(14, 7)
)

ax.bar(
    x,
    reduction_orders,
    label="Reduction Waves (todas las Week)",
)

ax.bar(
    x,
    other_orders,
    bottom=reduction_orders,
    label="Resto de orders CL",
)

offset = max(
    total_orders.max() * 0.018,
    1
)

for i, row in plot_df.iterrows():

    total = row["orders"]
    total_share = row["orders_share_cl_pct"]

    reduced_orders = row["reduction_wave_orders"]
    reduced_share_cl = row["reduction_wave_share_cl_pct"]
    reduced_within_bucket = row["reduction_share_within_bucket_pct"]

    # Total del bucket.
    ax.text(
        i,
        total + offset,
        f"{total:,.0f}\n{total_share:.2f}% CL",
        ha="center",
        va="bottom",
        fontsize=9,
    )

    # Segmento reducido.
    if reduced_orders > 0:
        ax.text(
            i,
            reduced_orders / 2,
            (
                f"{reduced_orders:,.0f}\n"
                f"{reduced_share_cl:.2f}% CL\n"
                f"{reduced_within_bucket:.1f}% bucket"
            ),
            ha="center",
            va="center",
            fontsize=8,
        )

ax.set_xticks(x)

ax.set_xticklabels(
    plot_df["awt0_share_bucket"].astype(str),
    rotation=45,
    ha="right",
)

ax.set_xlabel(
    "AWT0 share bucket"
)

ax.set_ylabel(
    "Orders"
)

ax.set_title(
    "Orders CL por AWT0 share bucket\n"
    "y participación de todas las Reduction Waves tipo Week"
)

ax.yaxis.set_major_formatter(
    FuncFormatter(
        lambda value, _:
        f"{value/1000:.0f}k"
        if value >= 1000
        else f"{value:.0f}"
    )
)

ax.legend()

ax.set_ylim(
    0,
    max(
        total_orders.max() * 1.18,
        1
    )
)

plt.tight_layout()
plt.show()
